# Vicilin trimer: GROMACS setup, runs and analysis (Colab)

Working notebook for the pea vicilin trimer simulations in this repository: system setup, 1 ns runs at 300 K, 373 K and 373 K with box shear (two replicates each), and the analysis of every run.

- **Software:** GROMACS 2026.3 (conda-forge CUDA build) on a Colab T4 GPU; MDAnalysis, SciPy, pandas and Matplotlib for analysis. See `environment.yml`.
- **Layout:** the sections follow the order the work was done in. Sections 1–6 are setup and runs, 7–8 are the per-run analysis, 9 compares runs, 10–11 cover replicate 2.
- **Paths:** file paths point to the author's Google Drive folder (`/content/drive/MyDrive/Plant Protein Extrusion MD`). Change them to your own folder before running.
- **Outputs are cleared.** Figures and tables are in `analysis/`, and the run settings are in `simulation_setup/`.

## 1. Mount Drive and prepare the structure
Mount Google Drive, convert the AlphaFold3 CIF to PDB and check the chains (3 chains, vicilin trimer).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Update this path if Legumin_MD is inside another folder
PDB_PATH = "/content/drive/MyDrive/Vicilin Trimer/Vicilin trimer.cif"

if os.path.exists(PDB_PATH):
    print("PDB found!")
    print("File size:", round(os.path.getsize(PDB_PATH) / 1024, 2), "KB")
else:
    print("PDB not found. Check the folder path.")

In [ ]:
!pip -q install biopython

from Bio.PDB import MMCIFParser, PDBIO
import os

CIF_PATH = "/content/drive/MyDrive/Vicilin Trimer/Vicilin trimer.cif"
PDB_PATH = "/content/drive/MyDrive/Vicilin Trimer/Vicilin trimer.pdb"

parser = MMCIFParser(QUIET=True)
structure = parser.get_structure("vicilin_trimer", CIF_PATH)

io = PDBIO()
io.set_structure(structure)
io.save(PDB_PATH)

print("Conversion successful!")
print("PDB saved at:", PDB_PATH)
print("File size:", round(os.path.getsize(PDB_PATH) / 1024, 2), "KB")
print("Chains:", [chain.id for chain in structure.get_chains()])

In [ ]:
from collections import defaultdict

PDB_PATH = "/content/drive/MyDrive/Vicilin Trimer/Vicilin trimer.pdb"

chains = defaultdict(set)
atom_counts = defaultdict(int)

with open(PDB_PATH, "r") as f:
    for line in f:
        if line.startswith("ATOM  "):
            chain = line[21].strip() or "_"
            residue_number = line[22:27].strip()
            residue_name = line[17:20].strip()

            chains[chain].add((residue_number, residue_name))
            atom_counts[chain] += 1

print("Number of chains:", len(chains))
print("Chain IDs:", sorted(chains.keys()))

for chain in sorted(chains):
    print(
        f"Chain {chain}: "
        f"{len(chains[chain])} residues, "
        f"{atom_counts[chain]} atoms"
    )

print("Total protein atoms:", sum(atom_counts.values()))

In [ ]:
from collections import Counter, defaultdict

PDB_PATH = "/content/drive/MyDrive/Vicilin Trimer/Vicilin trimer.pdb"

residues = defaultdict(dict)
record_counts = Counter()
atom_names = defaultdict(Counter)
other_residues = Counter()

with open(PDB_PATH) as f:
    for line in f:
        record = line[:6].strip()

        if record:
            record_counts[record] += 1

        if record == "ATOM":
            chain = line[21].strip() or "_"
            resid = (line[22:27].strip(), line[17:20].strip())
            atom_name = line[12:16].strip()

            residues[chain][resid] = True
            atom_names[chain][atom_name] += 1

        elif record == "HETATM":
            other_residues[line[17:20].strip()] += 1

print("PDB record types:", record_counts)
print("\nResidue numbering by chain:")

for chain in sorted(residues):
    numbers = [int(r[0].split()[0]) for r in residues[chain]]
    print(
        f"Chain {chain}: first={min(numbers)}, "
        f"last={max(numbers)}, unique residues={len(numbers)}"
    )

print("\nHETATM residue types:", other_residues or "None")
print("\nUnique atom names in chain A:")
print(sorted(atom_names["A"].keys()))

## 2. Install GROMACS 2026.3 (CUDA build)
Installs condacolab, then creates the `gmx_cuda` conda environment with the conda-forge CUDA build of GROMACS 2026.3.

In [ ]:
%pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
!CONDA_OVERRIDE_CUDA=12.9 conda create -y \
    -n gmx_cuda \
    -c conda-forge \
    "gromacs=2026.3=nompi_cuda*"

In [ ]:
!conda run -n gmx_cuda gmx --version

## 3. System build and energy minimisation
`pdb2gmx` → `editconf` → `solvate` → `genion` → energy minimisation. Some cells here still use the `legumin_hexamer_*` file names from the legumin build; the vicilin ion and EM steps are at the end of this section and in section 4.

In [ ]:
import os
import shutil

SOURCE = "/content/drive/MyDrive/Vicilin Trimer/Vicilin trimer.pdb"
PROJECT = "/content/drive/MyDrive/Plant Protein Extrusion MD"
STRUCTURES = os.path.join(PROJECT, "structures")
ANALYSIS = os.path.join(PROJECT, "analysis")
SIMULATIONS = os.path.join(PROJECT, "simulations")

for folder in [STRUCTURES, ANALYSIS, SIMULATIONS]:
    os.makedirs(folder, exist_ok=True)

DEST = os.path.join(STRUCTURES, "legumin_hexamer_original.pdb")

if not os.path.isfile(SOURCE):
    raise FileNotFoundError(f"Source PDB not found: {SOURCE}")

if not os.path.exists(DEST):
    shutil.copy2(SOURCE, DEST)

print("Project folder:", PROJECT)
print("Original PDB preserved:", os.path.isfile(SOURCE))
print("Working copy:", DEST)
print("Working copy size:", round(os.path.getsize(DEST) / 1024, 2), "KB")
print("GROMACS environment: /usr/local/envs/gmx_cuda")

In [ ]:
!conda run -n gmx_cuda gmx pdb2gmx -h

In [ ]:
from pathlib import Path

GMX_DATA = Path("/usr/local/envs/gmx_cuda/share/gromacs/top")

print("Topology directory exists:", GMX_DATA.is_dir())

if GMX_DATA.is_dir():
    print("\nInstalled Amber force fields:")
    for item in sorted(GMX_DATA.glob("*amber*.ff")):
        print(item.name)

    print("\nAll force-field directories:")
    for item in sorted(GMX_DATA.glob("*.ff")):
        print(item.name)
else:
    print("Directory not found; checking the environment:")
    !conda run -n gmx_cuda gmx --version

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!conda run -n gmx_cuda gmx pdb2gmx \
    -f structures/legumin_hexamer_original.pdb \
    -o structures/legumin_hexamer_processed.gro \
    -p topol.top \
    -i posre.itp \
    -ff amber99sb-ildn \
    -water spce \
    -chainsep id_or_ter \
    -merge no

In [ ]:
import os

PROJECT = "/content/drive/MyDrive/Plant Protein Extrusion MD"

files = [
    "structures/legumin_hexamer_processed.gro",
    "topol.top",
    "posre.itp",
]

for name in files:
    path = os.path.join(PROJECT, name)
    print(f"{name}: {'FOUND' if os.path.isfile(path) else 'MISSING'}")
    if os.path.isfile(path):
        print(f"  Size: {os.path.getsize(path) / 1024:.1f} KB")

print("\nTopology molecule definitions:")
with open(os.path.join(PROJECT, "topol.top")) as f:
    show = False
    for line in f:
        if line.strip() == "[ molecules ]":
            show = True
        if show:
            print(line.rstrip())

In [ ]:
import os

PROJECT = "/content/drive/MyDrive/Plant Protein Extrusion MD"

print("Checking possible position-restraint files:\n")

for root, dirs, files in os.walk(PROJECT):
    for name in files:
        if name == "posre.itp" or name.startswith("posre_"):
            path = os.path.join(root, name)
            print(path, f"({os.path.getsize(path) / 1024:.1f} KB)")

print("\nChecking topology includes:")
with open(os.path.join(PROJECT, "topol.top")) as f:
    for line in f:
        if "#include" in line or "[ molecules ]" in line:
            print(line.rstrip())

In [ ]:
import os

PROJECT = "/content/drive/MyDrive/Plant Protein Extrusion MD"

for chain in "ABC":
    filename = f"topol_Protein_chain_{chain}.itp"
    path = os.path.join(PROJECT, filename)

    print(f"\n--- Chain {chain}: {filename} ---")

    if not os.path.isfile(path):
        print("MISSING")
        continue

    with open(path) as f:
        for line in f:
            if "posre" in line.lower() or line.strip() == "[ moleculetype ]":
                print(line.rstrip())

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!conda run -n gmx_cuda gmx editconf \
    -f structures/legumin_hexamer_processed.gro \
    -o structures/legumin_hexamer_box.gro \
    -c \
    -d 1.2 \
    -bt dodecahedron

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!conda run -n gmx_cuda gmx solvate \
    -cp structures/legumin_hexamer_box.gro \
    -cs spc216.gro \
    -o structures/legumin_hexamer_solv.gro \
    -p topol.top

In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")
mdp = project / "em.mdp"

mdp.write_text("""; Energy minimization
integrator      = steep
emtol           = 1000.0
emstep          = 0.01
nsteps          = 50000

cutoff-scheme   = Verlet
nstlist         = 20
coulombtype     = PME
rcoulomb        = 1.0
rvdw            = 1.0
pbc             = xyz

constraints     = h-bonds
""")

print(f"Created: {mdp}")
print(mdp.read_text())

In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")
top = project / "topol.top"

print("Topology solvent and ion entries:")
for line in top.read_text().splitlines():
    if "spce.itp" in line or "ions.itp" in line or line.strip().startswith("SOL"):
        print(line)

ions = Path("/usr/local/envs/gmx_cuda/share/gromacs/top/amber99sb-ildn.ff/ions.itp")
print("\nIon definitions file exists:", ions.exists())

if ions.exists():
    for line in ions.read_text().splitlines():
        if line.strip().startswith("[ ") or line.strip().startswith("NA") or line.strip().startswith("CL"):
            print(line)

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!printf "SOL\n" | conda run -n gmx_cuda gmx genion \
    -s ions.tpr \
    -o structures/legumin_hexamer_ions.gro \
    -p topol.top \
    -pname NA \
    -nname CL \
    -neutral

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!conda run -n gmx_cuda gmx grompp \
    -f em.mdp \
    -c structures/legumin_hexamer_solv.gro \
    -p topol.top \
    -o ions.tpr \
    -maxwarn 1

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!conda run -n gmx_cuda gmx genion \
    -s ions.tpr \
    -o structures/legumin_hexamer_ions.gro \
    -p topol.top \
    -pname NA \
    -nname CL \
    -neutral \
    --select 'group "SOL"'

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!bash -c 'printf "13\n" | conda run -n gmx_cuda gmx genion \
    -s ions.tpr \
    -o structures/legumin_hexamer_ions.gro \
    -p topol.top \
    -pname NA \
    -nname CL \
    -neutral'

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!printf '13\n' | /usr/local/envs/gmx_cuda/bin.AVX2_256/gmx genion \
    -s ions.tpr \
    -o structures/legumin_hexamer_ions.gro \
    -p topol.top \
    -pname NA \
    -nname CL \
    -neutral

In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/Plant Protein Extrusion MD")
gro = project / "structures/legumin_hexamer_ions.gro"
top = project / "topol.top"

print("Ionized structure exists:", gro.exists())

if gro.exists():
    with gro.open() as f:
        f.readline()
        atom_count = int(f.readline().strip())
    print("Total atoms in ionized structure:", atom_count)

print("\nUpdated topology:")
for line in top.read_text().splitlines():
    if line.strip().startswith(("SOL", "NA", "CL")):
        print(line)

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!conda run -n gmx_cuda gmx grompp \
    -f em.mdp \
    -c structures/legumin_hexamer_ions.gro \
    -p topol.top \
    -o em.tpr \
    -maxwarn 0

In [ ]:
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!conda run -n gmx_cuda gmx mdrun \
    -deffnm em \
    -ntmpi 1 \
    -ntomp 4 \
    -pin off

## 4. Vicilin trimer: ions, EM and 300 K runs
Ion addition and EM for the vicilin trimer, then NVT, NPT and 1 ns production at 300 K (r1 and r2).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"
!ls

In [ ]:
import os
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin.AVX2_256:/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

In [ ]:
%%bash
SEED=111; T=300; TAG=nvt_${T}_r1

cat > ${TAG}.mdp << EOF
define        = -DPOSRES
integrator    = md
dt            = 0.002
nsteps        = 50000          ; 100 ps
continuation  = no
constraints   = h-bonds
constraint-algorithm = lincs
cutoff-scheme = Verlet
coulombtype   = PME
rcoulomb      = 1.0
rvdw          = 1.0
DispCorr      = EnerPres
tcoupl        = V-rescale
tc-grps       = Protein Non-Protein
tau-t         = 0.1 0.1
ref-t         = ${T} ${T}
pcoupl        = no
gen-vel       = yes
gen-temp      = ${T}
gen-seed      = ${SEED}
nstenergy     = 500
nstlog        = 500
nstxout-compressed = 5000
EOF

gmx grompp -f ${TAG}.mdp -c em.gro -r em.gro -p topol.top -o ${TAG}.tpr
gmx mdrun -deffnm ${TAG} -ntmpi 1 -ntomp 2 -pin off

In [ ]:
!tail -n 8 topol.top

In [ ]:
%%bash
cp topol.top topol_wrong_ions.bak
sed -i '/^NA /d' topol.top
sed -i 's/^SOL .*/SOL                50992/' topol.top
tail -n 6 topol.top

In [ ]:
%%bash
gmx grompp -f ions.mdp -c structures/vicilin_trimer_solv.gro -p topol.top -o ions.tpr
echo SOL | gmx genion -s ions.tpr -o structures/vicilin_trimer_ions.gro -p topol.top -pname NA -nname CL -neutral -conc 0.15
tail -n 6 topol.top

In [ ]:
%%bash
cat > ions.mdp << EOF
integrator = steep
nsteps     = 0
cutoff-scheme = Verlet
coulombtype = cutoff
rcoulomb   = 1.0
rvdw       = 1.0
pbc        = xyz
EOF

gmx grompp -f ions.mdp -c structures/vicilin_trimer_solv.gro -p topol.top -o ions.tpr -maxwarn 1
echo SOL | gmx genion -s ions.tpr -o structures/vicilin_trimer_ions.gro -p topol.top -pname NA -nname CL -neutral -conc 0.15
tail -n 6 topol.top

In [ ]:
%%bash
python3 - << 'EOF'
t = open('topol.top').read()
i = t.index('[ molecules ]')
t = t[:i] + """[ molecules ]
; Compound        #mols
Protein_chain_A     1
Protein_chain_B     1
Protein_chain_C     1
SOL         50655
NA          179
CL          158
"""
open('topol.top', 'w').write(t)
EOF
tail -n 8 topol.top
sed -n 2p structures/vicilin_trimer_ions.gro

In [ ]:
%%bash
gmx grompp -f em.mdp -c structures/vicilin_trimer_ions.gro -p topol.top -o em.tpr
gmx mdrun -deffnm em -ntmpi 1 -ntomp 2 -pin off

In [ ]:
%%bash
set -e
T=300; R=1; SEED=111

# NVT (100 ps)
TAG=nvt_${T}_r${R}
cat > ${TAG}.mdp << EOF
define = -DPOSRES
integrator = md
dt = 0.002
nsteps = 50000
continuation = no
constraints = h-bonds
constraint-algorithm = lincs
cutoff-scheme = Verlet
coulombtype = PME
rcoulomb = 1.0
rvdw = 1.0
DispCorr = EnerPres
tcoupl = V-rescale
tc-grps = Protein Non-Protein
tau-t = 0.1 0.1
ref-t = ${T} ${T}
pcoupl = no
gen-vel = yes
gen-temp = ${T}
gen-seed = ${SEED}
nstenergy = 500
nstlog = 500
nstxout-compressed = 5000
EOF
gmx grompp -f ${TAG}.mdp -c em.gro -r em.gro -p topol.top -o ${TAG}.tpr
gmx mdrun -deffnm ${TAG} -ntmpi 1 -ntomp 2 -pin off

# NPT (500 ps)
PREV=${TAG}; TAG=npt_${T}_r${R}
sed -e 's/^nsteps.*/nsteps = 250000/' \
    -e 's/^continuation.*/continuation = yes/' \
    -e 's/^pcoupl .*/pcoupl = C-rescale\npcoupltype = isotropic\ntau-p = 2.0\ncompressibility = 4.5e-5\nref-p = 1.0/' \
    -e 's/^gen-vel.*/gen-vel = no/' \
    -e '/^gen-temp/d' -e '/^gen-seed/d' ${PREV}.mdp > ${TAG}.mdp
gmx grompp -f ${TAG}.mdp -c ${PREV}.gro -r ${PREV}.gro -t ${PREV}.cpt -p topol.top -o ${TAG}.tpr
gmx mdrun -deffnm ${TAG} -ntmpi 1 -ntomp 2 -pin off

In [ ]:
%%bash
set -e
T=300; R=1
PREV=nvt_${T}_r${R}; TAG=npt_${T}_r${R}

grep -q refcoord ${TAG}.mdp || echo "refcoord-scaling = com" >> ${TAG}.mdp

gmx grompp -f ${TAG}.mdp -c ${PREV}.gro -r ${PREV}.gro -t ${PREV}.cpt -p topol.top -o ${TAG}.tpr
gmx mdrun -deffnm ${TAG} -ntmpi 1 -ntomp 2 -pin off

In [ ]:
%%bash
echo "Density" | gmx energy -f npt_300_r1.edr -o npt_300_r1_density.xvg | tail -n 4
echo "Temperature" | gmx energy -f npt_300_r1.edr | tail -n 4
echo "Potential" | gmx energy -f npt_300_r1.edr | tail -n 4
echo "Pressure" | gmx energy -f npt_300_r1.edr | tail -n 4
tail -n 1 npt_300_r1.gro

In [ ]:
%%bash
set -e
T=300; R=1; SEED=111
PREV=npt_${T}_r${R}; TAG=md_${T}_r${R}

cat > ${TAG}.mdp << EOF
integrator = md
dt = 0.002
nsteps = 500000
continuation = yes
constraints = h-bonds
constraint-algorithm = lincs
cutoff-scheme = Verlet
coulombtype = PME
rcoulomb = 1.0
rvdw = 1.0
DispCorr = EnerPres
tcoupl = V-rescale
tc-grps = Protein Non-Protein
tau-t = 0.1 0.1
ref-t = ${T} ${T}
pcoupl = no
gen-vel = no
ld-seed = ${SEED}
nstenergy = 500
nstlog = 5000
nstxout-compressed = 500
compressed-x-grps = Protein
EOF

gmx grompp -f ${TAG}.mdp -c ${PREV}.gro -t ${PREV}.cpt -p topol.top -o ${TAG}.tpr
gmx mdrun -deffnm ${TAG} -ntmpi 1 -ntomp 2 -pin off

In [ ]:
%%bash
set -e
T=300; R=2
SEED=$((R*111))
echo "T=$T rep=$R seed=$SEED"

COMMON="integrator = md
dt = 0.002
constraints = h-bonds
constraint-algorithm = lincs
cutoff-scheme = Verlet
coulombtype = PME
rcoulomb = 1.0
rvdw = 1.0
DispCorr = EnerPres
tcoupl = V-rescale
tc-grps = Protein Non-Protein
tau-t = 0.1 0.1
ref-t = ${T} ${T}"

# NVT 100 ps
TAG=nvt_${T}_r${R}
cat > ${TAG}.mdp << EOF
define = -DPOSRES
${COMMON}
nsteps = 50000
continuation = no
pcoupl = no
gen-vel = yes
gen-temp = ${T}
gen-seed = ${SEED}
nstenergy = 500
nstlog = 500
nstxout-compressed = 5000
EOF
gmx grompp -f ${TAG}.mdp -c em.gro -r em.gro -p topol.top -o ${TAG}.tpr
gmx mdrun -deffnm ${TAG} -cpt 5 -ntmpi 1 -ntomp 2 -pin off

# NPT 500 ps
PREV=${TAG}; TAG=npt_${T}_r${R}
cat > ${TAG}.mdp << EOF
define = -DPOSRES
${COMMON}
nsteps = 250000
continuation = yes
pcoupl = C-rescale
pcoupltype = isotropic
tau-p = 2.0
compressibility = 4.5e-5
ref-p = 1.0
refcoord-scaling = com
gen-vel = no
nstenergy = 500
nstlog = 500
nstxout-compressed = 5000
EOF
gmx grompp -f ${TAG}.mdp -c ${PREV}.gro -r ${PREV}.gro -t ${PREV}.cpt -p topol.top -o ${TAG}.tpr
gmx mdrun -deffnm ${TAG} -cpt 5 -ntmpi 1 -ntomp 2 -pin off

# Production 1 ns (NVT, no restraints)
PREV=${TAG}; TAG=md_${T}_r${R}
cat > ${TAG}.mdp << EOF
${COMMON}
nsteps = 500000
continuation = yes
pcoupl = no
gen-vel = no
ld-seed = ${SEED}
nstenergy = 500
nstlog = 5000
nstxout-compressed = 500
compressed-x-grps = Protein
EOF
gmx grompp -f ${TAG}.mdp -c ${PREV}.gro -t ${PREV}.cpt -p topol.top -o ${TAG}.tpr
gmx mdrun -deffnm ${TAG} -cpt 5 -ntmpi 1 -ntomp 2 -pin off

In [ ]:
%%bash
tail -n 6 md_300_r2.log
awk '/Step           Time/{getline; s=$1} END{printf "last step: %d / 500000\n", s}' md_300_r1.log
grep -c "LINCS WARNING" md_300_r2.log || true
echo "Temperature" | gmx energy -f md_300_r2.edr 2>/dev/null | grep -E "^Temperature"
echo "Potential" | gmx energy -f md_300_r2.edr 2>/dev/null | grep -E "^Potential"
echo "Pressure" | gmx energy -f md_300_r2.edr 2>/dev/null | grep -E "^Pressure"

## 5. Vicilin trimer 373 K, replicate 1
NVT, NPT and production at 373 K (r1).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin.AVX2_256:/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"
!gmx --version | head -n 3
!nvidia-smi | head -n 12
!ls em.gro topol.top
!ls nvt_373* npt_373* 2>/dev/null

In [ ]:
%%bash
set -e
T=373; R=1
SEED=$((R*111))
echo "T=$T rep=$R seed=$SEED"

COMMON="integrator = md
dt = 0.002
constraints = h-bonds
constraint-algorithm = lincs
cutoff-scheme = Verlet
coulombtype = PME
rcoulomb = 1.0
rvdw = 1.0
DispCorr = EnerPres
tcoupl = V-rescale
tc-grps = Protein Non-Protein
tau-t = 0.1 0.1
ref-t = ${T} ${T}"

# NVT
TAG=nvt_${T}_r${R}
cat > ${TAG}.mdp << EOF
define = -DPOSRES
${COMMON}
nsteps = 50000
continuation = no
pcoupl = no
gen-vel = yes
gen-temp = ${T}
gen-seed = ${SEED}
nstenergy = 500
nstlog = 500
nstxout-compressed = 5000
EOF
gmx grompp -f ${TAG}.mdp -c em.gro -r em.gro -p topol.top -o ${TAG}.tpr
gmx mdrun -deffnm ${TAG} -cpt 5 -ntmpi 1 -ntomp 2 -pin off

# NPT
PREV=${TAG}; TAG=npt_${T}_r${R}
cat > ${TAG}.mdp << EOF
define = -DPOSRES
${COMMON}
nsteps = 250000
continuation = yes
pcoupl = C-rescale
pcoupltype = isotropic
tau-p = 2.0
compressibility = 4.5e-5
ref-p = 1.0
refcoord-scaling = com
gen-vel = no
nstenergy = 500
nstlog = 500
nstxout-compressed = 5000
EOF
gmx grompp -f ${TAG}.mdp -c ${PREV}.gro -r ${PREV}.gro -t ${PREV}.cpt -p topol.top -o ${TAG}.tpr
gmx mdrun -deffnm ${TAG} -cpt 5 -ntmpi 1 -ntomp 2 -pin off
echo "DONE NVT + NPT"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin.AVX2_256:/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
%cd "/content/drive/MyDrive/Plant Protein Extrusion MD"
!gmx --version | head -n 3
!nvidia-smi | head -n 12

In [ ]:
!ls -l npt_373_r1.*
!awk '/Step           Time/{getline; s=$1} END{printf "%d / 250000 steps (%.1f%%)\n", s, s/2500}' npt_373_r1.log

In [ ]:
%%bash
tail -n 5 npt_373_r1.log
for term in Density Temperature Potential Pressure; do
  echo $term | gmx energy -f npt_373_r1.edr 2>&1 | grep -E "^$term|Last energy frame"
done
tail -n 1 npt_373_r1.gro

In [ ]:
%%bash
set -e
T=373; R=1; SEED=111
PREV=npt_${T}_r${R}; TAG=md_${T}_r${R}

cat > ${TAG}.mdp << EOF
integrator = md
dt = 0.002
nsteps = 500000
continuation = yes
constraints = h-bonds
constraint-algorithm = lincs
cutoff-scheme = Verlet
coulombtype = PME
rcoulomb = 1.0
rvdw = 1.0
DispCorr = EnerPres
tcoupl = V-rescale
tc-grps = Protein Non-Protein
tau-t = 0.1 0.1
ref-t = ${T} ${T}
pcoupl = no
gen-vel = no
ld-seed = ${SEED}
nstenergy = 500
nstlog = 5000
nstxout-compressed = 500
compressed-x-grps = Protein
EOF

gmx grompp -f ${TAG}.mdp -c ${PREV}.gro -t ${PREV}.cpt -p topol.top -o ${TAG}.tpr
gmx mdrun -deffnm ${TAG} -cpt 5 -ntmpi 1 -ntomp 2 -pin off

In [ ]:
%%bash
tail -n 6 md_373_r1.log
awk '/Step           Time/{getline; s=$1} END{printf "last step: %d / 500000\n", s}' md_373_r1.log
grep -c "LINCS WARNING" md_373_r1.log || true
for term in Temperature Potential Pressure; do
  echo $term | gmx energy -f md_373_r1.edr 2>/dev/null | grep -E "^$term"
done
ls -l md_373_r1.*

In [ ]:
%%bash
set -e
T=373; R=1; SEED=111
PREV=npt_${T}_r${R}; TAG=md_${T}_shear_r${R}
BY=$(tail -n 1 ${PREV}.gro | awk '{print $2}')
DEF=$(awk -v b=$BY 'BEGIN{printf "%.6f", 2.5e-4*b}')
echo "b_y = $BY nm  ->  deform b_x velocity = $DEF nm/ps"

cat > ${TAG}.mdp << EOF
integrator = md
dt = 0.002
nsteps = 500000
continuation = yes
constraints = h-bonds
constraint-algorithm = lincs
cutoff-scheme = Verlet
coulombtype = PME
rcoulomb = 1.0
rvdw = 1.0
DispCorr = EnerPres
tcoupl = V-rescale
tc-grps = System
tau-t = 0.1
ref-t = ${T}
pcoupl = no
deform = 0 0 0 ${DEF} 0 0
gen-vel = no
ld-seed = ${SEED}
nstenergy = 500
nstlog = 5000
nstxout-compressed = 500
compressed-x-grps = Protein
EOF

gmx grompp -f ${TAG}.mdp -c ${PREV}.gro -t ${PREV}.cpt -p topol.top -o ${TAG}.tpr
gmx mdrun -deffnm ${TAG} -cpt 5 -ntmpi 1 -ntomp 2 -pin off

## 6. 373 K shear r1: recovering the interrupted run
Checks on the truncated trajectory, restart from the checkpoint and joining the parts into one 1 ns trajectory.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!find "/content/drive/MyDrive" -type f -name "*.edr"

In [ ]:
!ls -lh "/content/drive/MyDrive/Plant Protein Extrusion MD/"

In [ ]:
!gmx check -f md_373_shear_r1.xtc

In [ ]:
!ls -lh --time-style=full-iso \
md_373_shear_r1.xtc \
md_373_shear_r1.log \
md_373_shear_r1.cpt \
md_373_shear_r1.edr

In [ ]:
!cd "/content/drive/MyDrive/Plant Protein Extrusion MD"

!find backups -type f \
  \( -name "md_373_shear_r1.xtc" -o -name "md_373_shear_r1.edr" \) \
  -printf '%TY-%Tm-%Td %TH:%TM  %s bytes  %p\n'

In [ ]:
import os
import shutil
from datetime import datetime
from google.colab import drive

drive.mount('/content/drive')

src = "/content/drive/MyDrive/Plant Protein Extrusion MD"
dest = os.path.join(
    src, "recovered_final_run",
    datetime.now().strftime("%Y%m%d_%H%M%S")
)
os.makedirs(dest, exist_ok=True)

files = [
    "md_373_shear_r1.xtc",
    "md_373_shear_r1.edr",
    "md_373_shear_r1.log",
    "md_373_shear_r1.cpt",
    "md_373_shear_r1.tpr",
    "md_373_shear_r1.mdp",
    "md_373_shear_r1.gro",
]

for name in files:
    source = os.path.join(src, name)
    if os.path.isfile(source):
        target = os.path.join(dest, name)
        shutil.copy2(source, target)
        print(f"Saved: {name} ({os.path.getsize(target):,} bytes)")
    else:
        print(f"Not found: {name}")

print("\nRecovery folder:", dest)

In [ ]:
import os
import glob
import subprocess

BASE = "/content/drive/MyDrive/Plant Protein Extrusion MD"
RECOVERY = os.path.join(BASE, "recovered_final_run")

print("=== RECOVERED FILES ===")
for f in glob.glob(RECOVERY + "/**/md_373_shear_r1.*", recursive=True):
    print(os.path.basename(f), f"{os.path.getsize(f):,} bytes")

xtcs = glob.glob(RECOVERY + "/**/md_373_shear_r1.xtc", recursive=True)
if xtcs:
    print("\n=== XTC FRAME CHECK ===")
    result = subprocess.run(
        ["/usr/local/envs/gmx_cuda/bin.AVX2_256/gmx",
         "check", "-f", xtcs[-1]],
        capture_output=True, text=True
    )
    print(result.stdout[-4000:])
    if result.stderr:
        print(result.stderr[-1000:])

logs = glob.glob(RECOVERY + "/**/md_373_shear_r1.log", recursive=True)
if logs:
    print("\n=== FINAL LOG LINES ===")
    with open(logs[-1], errors="replace") as f:
        print("".join(f.readlines()[-25:]))

In [ ]:
import subprocess

gmx = "/usr/local/envs/gmx_cuda/bin.AVX2_256/gmx"
cpt = "/content/drive/MyDrive/Plant Protein Extrusion MD/recovered_final_run/20261012_174925/md_373_shear_r1.cpt"

result = subprocess.run(
    [gmx, "dump", "-cp", cpt],
    capture_output=True, text=True
)

print((result.stdout + result.stderr)[-6000:])

In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    dirs[:] = [d for d in dirs if d not in {"proc", "sys", "dev"}]
    for name in files:
        if name == "md_373_shear_r1.xtc" or name == "md_373_shear_r1.edr":
            path = os.path.join(root, name)
            print(f"{os.path.getsize(path):,} bytes | {path}")

In [ ]:
import subprocess

gmx = "/usr/local/envs/gmx_cuda/bin.AVX2_256/gmx"
xtc = "/content/drive/MyDrive/Plant Protein Extrusion MD/backups/md_373_shear_r1/20261012_172918/md_373_shear_r1.xtc"

result = subprocess.run(
    [gmx, "check", "-f", xtc],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

In [ ]:
import os, shutil

base = "/content/drive/MyDrive/Plant Protein Extrusion MD"
src = os.path.join(base, "backups/md_373_shear_r1/20261012_172918")
dst = os.path.join(base, "recovered_valid_935ps")

os.makedirs(dst, exist_ok=True)

for name in ["md_373_shear_r1.xtc", "md_373_shear_r1.edr"]:
    source = os.path.join(src, name)
    if os.path.isfile(source):
        shutil.copy2(source, os.path.join(dst, name))
        print(name, "copied:", os.path.getsize(source), "bytes")
    else:
        print("Missing:", name)

In [ ]:
gmx = "/usr/local/envs/gmx_cuda/bin.AVX2_256/gmx"
edr = "/content/drive/MyDrive/Plant Protein Extrusion MD/recovered_valid_935ps/md_373_shear_r1.edr"

import subprocess
r = subprocess.run([gmx, "check", "-e", edr],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr)

In [ ]:
cd "/content/drive/MyDrive/Plant Protein Extrusion MD/recovered_valid_935ps"

gmx check -f md_373_shear_r1.xtc

In [ ]:
!gmx check -f md_373_shear_r1.xtc 2>&1 | grep -iE "last frame|reading frame"
!tail -n 25 md_373_shear_r1.log

In [ ]:
!cd "Plant Protein Extrusion MD"
!gmx check -f md_373_shear_r1.xtc 2>&1 | grep -iE "last frame|step|time"
!gmx check -f backups/md_373_shear_r1/20261012_173419/md_373_shear_r1.xtc 2>&1 | grep -iE "last frame|step|time"
!ls -lh --time-style=full-iso md_373_shear_r1.xtc backups/md_373_shear_r1/20261012_173419/

In [ ]:
!mkdir -p /content/resume && cd /content/resume
!cp "/content/drive/MyDrive/Plant Protein Extrusion MD/backups/md_373_shear_r1/20261012_172918/"* .
!cp "/content/drive/MyDrive/Plant Protein Extrusion MD/backups/md_373_shear_r1/20261012_173419/md_373_shear_r1.tpr" .
!ls -lh

In [ ]:
!mkdir -p /content/resume && cd /content/resume
!D="/content/drive/MyDrive/Plant Protein Extrusion MD"
!cp "$D"/md_373_shear_r1.{tpr,cpt,xtc,log,edr,mdp} .
!cp "$D"/md_373_shear_r1_prev.cpt .
!gmx check -f md_373_shear_r1.cpt 2>&1 | grep -iE "step|time"

In [ ]:
%%bash
set -e
D="/content/drive/MyDrive/Plant Protein Extrusion MD"
mkdir -p /content/resume && cd /content/resume
cp "$D"/md_373_shear_r1.{tpr,cpt,xtc,log,edr,mdp} .
cp "$D"/md_373_shear_r1_prev.cpt .
ls -lh --time-style=full-iso
gmx check -f md_373_shear_r1.cpt 2>&1 | grep -iE "step|time"

In [ ]:
%%bash
cd /content/resume
gmx mdrun -s md_373_shear_r1.tpr -deffnm md_373_shear_r1 -cpi md_373_shear_r1.cpt -append -nb gpu -pme gpu -update gpu 2>&1 | tail -n 25
gmx check -f md_373_shear_r1.xtc 2>&1 | tail -n 8
tail -n 5 md_373_shear_r1.log

In [ ]:
%%bash
cd /content/resume
gmx mdrun -s md_373_shear_r1.tpr -deffnm md_373_shear_r1 -cpi md_373_shear_r1.cpt -append -nb gpu -pme gpu -update cpu 2>&1 | tail -n 25
gmx check -f md_373_shear_r1.xtc 2>&1 | tail -n 8
tail -n 5 md_373_shear_r1.log

In [ ]:
%%bash
for DIR in /content/resume "/content/drive/MyDrive/Plant Protein Extrusion MD/recovered_complete_1ns"; do
  [ -d "$DIR" ] || continue
  cd "$DIR"; echo "################ $DIR"
  echo "--- XTC (want ~1001 frames, timestep 1, last 1000)"
  gmx check -f md_373_shear_r1.xtc 2>&1 | grep -E "Last frame|^Step|^Time|^Box"
  echo "--- EDR (want first 0, last 1000)"
  gmx check -e md_373_shear_r1.edr 2>&1 | grep -iE "frame|time|step" | head -n 6
  echo "--- LOG (want step 500000 and Finished)"
  grep -A1 -E "^\s+Step\s+Time" md_373_shear_r1.log | tail -n 1
  grep -E "Finished mdrun" md_373_shear_r1.log
  echo "--- GRO (want t= 1000)"
  head -n 1 md_373_shear_r1.gro
  echo "--- CPT (want 1000)"
  gmx check -f md_373_shear_r1.cpt 2>&1 | grep -iE "step|time"
  ls -lh --time-style=full-iso md_373_shear_r1.{xtc,edr,log,gro,cpt}
done

In [ ]:
%%bash
cd /content/resume
OUT="/content/drive/MyDrive/Plant Protein Extrusion MD/recovered_complete_1ns"
mkdir -p "$OUT"
cp md_373_shear_r1.{tpr,cpt,xtc,log,edr,gro,mdp} md_373_shear_r1_prev.cpt "$OUT"/
sync
md5sum md_373_shear_r1.xtc md_373_shear_r1.edr "$OUT"/md_373_shear_r1.xtc "$OUT"/md_373_shear_r1.edr

In [ ]:
%%bash
D="/content/drive/MyDrive/Plant Protein Extrusion MD"
cd "$D"
for R in md_300_r1 md_300_r2 md_373_r1; do
  echo "################ $R"
  echo "--- XTC (want 1001 frames, timestep 1, last 1000)"
  gmx check -f $R.xtc 2>&1 | grep -E "Last frame|^Step|^Time|^Box"
  echo "--- EDR (want last 1000)"
  gmx check -e $R.edr 2>&1 | grep -iE "Last energy|Found"
  echo "--- LOG (want 500000 + Finished)"
  grep -A1 -E "^\s+Step\s+Time" $R.log | tail -n 1
  grep "Finished mdrun" $R.log
done

In [ ]:
%%bash
D="/content/drive/MyDrive/Plant Protein Extrusion MD"
cd "$D"
R=md_300_r2
gmx check -f $R.xtc 2>&1 | grep -E "Last frame|^Step|^Time|^Box"
gmx check -e $R.edr 2>&1 | grep -iE "Last energy|Found"
grep -A1 -E "^\s+Step\s+Time" $R.log | tail -n 1
grep "Finished mdrun" $R.log
gmx check -f $R.cpt 2>&1 | grep -iE "time"

## 7. First analysis pass (vicilin 300 K r1 and r2)
Whole-trimer and per-chain RMSD, RMSF, Rg, SASA, H-bonds and DSSP, built step by step.

In [ ]:
%%bash
R=md_300_r1; TAG=vicilin_300_r1
D="/content/drive/MyDrive/Plant Protein Extrusion MD"
W=/content/work/$TAG; mkdir -p $W/xvg $W/frames $W/logs; cd $W
cp "$D"/$R.{tpr,xtc,edr} .

# make molecules whole, remove jumps, fit on backbone
printf "System\n" | gmx trjconv -s $R.tpr -f $R.xtc -pbc whole -o whole.xtc > logs/whole.txt 2>&1
printf "System\n" | gmx trjconv -s $R.tpr -f whole.xtc -pbc nojump -o nojump.xtc > logs/nojump.txt 2>&1
printf "Backbone\nSystem\n" | gmx trjconv -s $R.tpr -f nojump.xtc -fit rot+trans -o fit.xtc > logs/fit.txt 2>&1

# first/last frames to check the chains by eye
printf "Protein\n" | gmx trjconv -s $R.tpr -f nojump.xtc -dump 0 -o frames/first.pdb > logs/dump0.txt 2>&1
printf "Protein\n" | gmx trjconv -s $R.tpr -f nojump.xtc -dump 1000 -o frames/last.pdb > logs/dump1.txt 2>&1

# analyses (xvg in ps)
printf "Backbone\nBackbone\n" | gmx rms -s $R.tpr -f fit.xtc -o xvg/rmsd_backbone.xvg > logs/rms.txt 2>&1 || echo "rms FAILED"
printf "C-alpha\n" | gmx rmsf -s $R.tpr -f fit.xtc -o xvg/rmsf_calpha.xvg -res > logs/rmsf.txt 2>&1 || echo "rmsf FAILED"
printf "Protein\n" | gmx gyrate -s $R.tpr -f nojump.xtc -o xvg/gyrate.xvg > logs/gyrate.txt 2>&1 || echo "gyrate FAILED"
printf "Protein\nProtein\n" | gmx sasa -s $R.tpr -f nojump.xtc -o xvg/sasa_total.xvg > logs/sasa.txt 2>&1 || echo "sasa FAILED"
printf "Protein\nProtein\n" | gmx hbond -s $R.tpr -f nojump.xtc -num xvg/hbonds_protein.xvg > logs/hbond.txt 2>&1 || echo "hbond FAILED"
gmx dssp -s $R.tpr -f nojump.xtc -sel 'group "Protein"' -o xvg/ss.dat -num xvg/ss_count.xvg > logs/dssp.txt 2>&1 || echo "dssp FAILED"
printf "Temperature\nPotential\nPressure\nDensity\n0\n" | gmx energy -f $R.edr -o xvg/energy.xvg > logs/energy.txt 2>&1 || echo "energy FAILED"

ls -lh xvg frames

In [ ]:
%%bash
R=md_300_r1; TAG=vicilin_300_r1
D="/content/drive/MyDrive/Plant Protein Extrusion MD"
cd "$D"

echo "===== CHECK $R ====="
OUT=$(gmx check -f $R.xtc 2>&1 | tr '\r' '\n')
NF=$(echo "$OUT" | awk '/^Step/{print $2}')
LT=$(echo "$OUT" | grep "Last frame" | awk '{print $NF}')
echo "XTC frames: $NF | last time: $LT"
gmx check -e $R.edr 2>&1 | grep -iE "Last energy|Found"
grep -A1 -E "^\s+Step\s+Time" $R.log | tail -n 1
grep "Finished mdrun" $R.log || echo "NO 'Finished mdrun' in log"
if [ "$NF" != "1001" ] || [ "${LT%.*}" != "1000" ]; then
  echo ">>> TRAJECTORY NOT COMPLETE - stopping, paste this output to Claude"; exit 1
fi
echo ">>> COMPLETE, running analysis"

W=/content/work/$TAG; rm -rf $W; mkdir -p $W/xvg $W/frames $W/logs; cd $W
cp "$D"/$R.{tpr,xtc,edr} .
echo "atoms in XTC: $(gmx check -f $R.xtc 2>&1 | grep -i 'Atoms' | head -n 1)"
echo "atoms in TPR: $(gmx dump -s $R.tpr 2>&1 | grep -m1 -i 'natoms')"

printf "Protein\n" | gmx convert-tpr -s $R.tpr -o protein.tpr > logs/convert.txt 2>&1 || { echo "convert-tpr FAILED"; tail -n 10 logs/convert.txt; }
T=protein.tpr
fail(){ echo "$1 FAILED"; tail -n 8 logs/$2.txt; }

printf "Protein\n" | gmx trjconv -s $T -f $R.xtc -pbc whole -o whole.xtc > logs/whole.txt 2>&1 || fail whole whole
printf "Protein\n" | gmx trjconv -s $T -f whole.xtc -pbc nojump -o nojump.xtc > logs/nojump.txt 2>&1 || fail nojump nojump
printf "Backbone\nProtein\n" | gmx trjconv -s $T -f nojump.xtc -fit rot+trans -o fit.xtc > logs/fit.txt 2>&1 || fail fit fit
printf "Protein\n" | gmx trjconv -s $T -f nojump.xtc -dump 0 -o frames/first.pdb > logs/dump0.txt 2>&1 || fail dump0 dump0
printf "Protein\n" | gmx trjconv -s $T -f nojump.xtc -dump 1000 -o frames/last.pdb > logs/dump1.txt 2>&1 || fail dump1 dump1

printf "Backbone\nBackbone\n" | gmx rms -s $T -f fit.xtc -o xvg/rmsd_backbone.xvg > logs/rms.txt 2>&1 || fail rms rms
printf "C-alpha\n" | gmx rmsf -s $T -f fit.xtc -o xvg/rmsf_calpha.xvg -res > logs/rmsf.txt 2>&1 || fail rmsf rmsf
printf "Protein\n" | gmx gyrate -s $T -f nojump.xtc -o xvg/gyrate.xvg > logs/gyrate.txt 2>&1 || fail gyrate gyrate
printf "Protein\nProtein\n" | gmx sasa -s $T -f nojump.xtc -o xvg/sasa_total.xvg > logs/sasa.txt 2>&1 || fail sasa sasa
printf "Protein\nProtein\n" | gmx hbond -s $T -f nojump.xtc -num xvg/hbonds_protein.xvg > logs/hbond.txt 2>&1 || fail hbond hbond
gmx dssp -s $T -f nojump.xtc -sel 'group "Protein"' -o xvg/ss.dat -num xvg/ss_count.xvg > logs/dssp.txt 2>&1 || fail dssp dssp
printf "Temperature\nPotential\nPressure\nDensity\n0\n" | gmx energy -f $R.edr -o xvg/energy.xvg > logs/energy.txt 2>&1 || fail energy energy

ls -lh xvg frames

In [ ]:
%%bash
TAG=vicilin_300_r1
OUT="/content/drive/MyDrive/Plant Protein Extrusion MD/analysis/$TAG"
mkdir -p "$OUT"
cp -r /content/work/$TAG/{xvg,frames,logs} "$OUT"/
sync
echo "--- local";  find /content/work/$TAG/xvg /content/work/$TAG/frames -type f | wc -l
echo "--- Drive";  find "$OUT/xvg" "$OUT/frames" -type f | wc -l
ls -lh "$OUT/xvg" "$OUT/frames"

In [ ]:
import os, re, shutil, numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

TAG = "vicilin_300_r1"
W = f"/content/work/{TAG}"
os.makedirs(f"{W}/graphs", exist_ok=True)

def read_xvg(p):
    title = xl = yl = ""; leg = {}; rows = []
    for l in open(p):
        if l.startswith("#"): continue
        if l.startswith("@"):
            for key, pat in (("t", r'@\s+title\s+"(.*)"'), ("x", r'@\s+xaxis\s+label\s+"(.*)"'), ("y", r'@\s+yaxis\s+label\s+"(.*)"')):
                m = re.match(pat, l)
                if m:
                    if key == "t": title = m.group(1)
                    elif key == "x": xl = m.group(1)
                    else: yl = m.group(1)
            m = re.match(r'@\s+s(\d+)\s+legend\s+"(.*)"', l)
            if m: leg[int(m.group(1))] = m.group(2)
            continue
        try: rows.append([float(v) for v in l.split()])
        except ValueError: pass
    return title, xl, yl, leg, np.array(rows)

for f in sorted(os.listdir(f"{W}/xvg")):
    if not f.endswith(".xvg"): continue
    t, xl, yl, leg, d = read_xvg(f"{W}/xvg/{f}")
    if d.size == 0: continue
    n = d.shape[1] - 1
    if f.startswith("energy") and n > 1:
        fig, ax = plt.subplots(n, 1, figsize=(7, 2.2*n), sharex=True)
        for i in range(n):
            ax[i].plot(d[:, 0], d[:, i+1], lw=1); ax[i].set_ylabel(leg.get(i, f"col{i+1}"), fontsize=8)
        ax[-1].set_xlabel(xl)
    else:
        fig, a = plt.subplots(figsize=(7, 4))
        for i in range(n):
            a.plot(d[:, 0], d[:, i+1], lw=1, label=leg.get(i))
        a.set_xlabel(xl); a.set_ylabel(yl); a.set_title(f"{TAG}: {t}")
        if leg and n > 1: a.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(f"{W}/graphs/{f[:-4]}.png", dpi=300); plt.close(fig)

OUT = f"/content/drive/MyDrive/Plant Protein Extrusion MD/analysis/{TAG}"
for sub in ("xvg", "graphs", "frames", "logs"):
    shutil.copytree(f"{W}/{sub}", f"{OUT}/{sub}", dirs_exist_ok=True)
print(sorted(os.listdir(OUT + "/graphs")))

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

fn = "rmsf_calpha.xvg"
if not os.path.exists(fn):
    files.upload()  # pick rmsf_calpha.xvg

data = np.loadtxt(fn, comments=("#", "@"))
x, y = data[:, 0], data[:, 1]

# new chain wherever residue number drops; fallback: 3 equal parts
breaks = np.where(np.diff(x) < 0)[0] + 1
if len(breaks) == 0:
    breaks = [len(x) // 3, 2 * len(x) // 3]
ys = np.split(y, breaks)
print("Chains found:", len(ys), "| residues per chain:", [len(c) for c in ys])

colors = ["#1f77b4", "#d62728", "#2ca02c", "#ff7f0e", "#9467bd", "#8c564b"]
fig, ax = plt.subplots(figsize=(8, 4.5), dpi=300)
for i, yi in enumerate(ys):
    ax.plot(np.arange(1, len(yi) + 1), yi,
            color=colors[i % len(colors)], lw=1.3,
            label=f"Chain {chr(65 + i)}")

ax.set_xlabel("Residue")
ax.set_ylabel("RMSF (nm)")
ax.set_title("vicilin_300_r1: C-alpha RMSF per chain")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig("rmsf_chains.png")
plt.show()
files.download("rmsf_chains.png")

In [ ]:
import os, re, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

base = "/content/drive/MyDrive/Plant Protein Extrusion MD/analysis/vicilin_300_r1"
graphs = os.path.join(base, "graphs")
run = os.path.basename(base)
LAST_PS = 500   # window for mean ± SD
SMOOTH = 20     # running-average frames

def clean(s):
    for a, b in [(r"\xb\f{}", "β"), (r"\xa\f{}", "α"), (r"\xp\f{}", "π"),
                 (r"\s10\N", "₁₀"), (r"\S2\N", "²")]:
        s = s.replace(a, b)
    s = re.sub(r"\\s(.*?)\\N", r"_\1", s)
    s = re.sub(r"\\S(.*?)\\N", r"^\1", s)
    return re.sub(r"\\[A-Za-z]\{\}|\\N", "", s)

def read_xvg(path):
    meta = dict(title="", xlabel="", ylabel="", legends={})
    rows = []
    for line in open(path):
        if line.startswith("#"):
            continue
        if line.startswith("@"):
            if (m := re.match(r'@\s+title\s+"(.*)"', line)):
                meta["title"] = clean(m.group(1))
            elif (m := re.match(r'@\s+xaxis\s+label\s+"(.*)"', line)):
                meta["xlabel"] = clean(m.group(1))
            elif (m := re.match(r'@\s+yaxis\s+label\s+"(.*)"', line)):
                meta["ylabel"] = clean(m.group(1))
            elif (m := re.match(r'@\s+s(\d+)\s+legend\s+"(.*)"', line)):
                meta["legends"][int(m.group(1))] = clean(m.group(2))
        elif line.strip() and not line.startswith("&"):
            rows.append([float(v) for v in line.split()])
    return meta, np.array(rows)

def find(name):
    hits = glob.glob(os.path.join(base, "**", name), recursive=True)
    if not hits:
        print("NOT FOUND:", name)
    return hits[0] if hits else None

def smooth(y, n=SMOOTH):
    return pd.Series(y).rolling(n, center=True, min_periods=1).mean().values

stats = []
def add_stats(file, label, t, y):
    m = t >= t.max() - LAST_PS
    tt, yy = t[m], y[m]
    blocks = np.array_split(yy, 5)
    stats.append(dict(file=file, series=label,
                      mean=yy.mean(), sd=yy.std(),
                      block_sd=np.std([b.mean() for b in blocks]),
                      drift_per_ns=np.polyfit(tt, yy, 1)[0] * 1000))

def plot_ts(fname, title, ylabel=None, split=False, do_smooth=True):
    p = find(fname)
    if not p:
        return
    meta, d = read_xvg(p)
    t = d[:, 0]
    cols = [i for i in range(1, d.shape[1]) if np.abs(d[:, i]).max() > 1e-9]
    cmap = plt.cm.tab10.colors
    n = len(cols) if split else 1
    fig, axes = plt.subplots(n, 1, figsize=(8, 2.6 * n + 1.5),
                             sharex=True, dpi=300, squeeze=False)
    for k, i in enumerate(cols):
        ax = axes[k, 0] if split else axes[0, 0]
        lab = meta["legends"].get(i - 1, f"col{i}")
        c = cmap[k % 10]
        if do_smooth:
            ax.plot(t, d[:, i], color=c, lw=0.5, alpha=0.3)
            ax.plot(t, smooth(d[:, i]), color=c, lw=1.4, label=lab)
        else:
            ax.plot(t, d[:, i], color=c, lw=0.9, label=lab)
        if split:
            ax.set_ylabel(lab)
        add_stats(fname, lab, t, d[:, i])
    ax0 = axes[0, 0]
    axes[-1, 0].set_xlabel("Time (ps)")
    if not split:
        ax0.set_ylabel(ylabel or meta["ylabel"])
        if len(cols) > 1:
            ax0.legend(frameon=False, fontsize=7, ncol=2)
    ax0.set_title(f"{run}: {title}")
    fig.tight_layout()
    out = os.path.join(graphs, fname.replace(".xvg", "_clean.png"))
    fig.savefig(out); plt.show(); print("Saved:", out)

plot_ts("rmsd_backbone.xvg", "Backbone RMSD", "RMSD (nm)")
plot_ts("gyrate.xvg", "Radius of gyration", "Rg (nm)")
plot_ts("sasa_total.xvg", "Total SASA", "SASA (nm²)")
plot_ts("hbonds_protein.xvg", "Intramolecular H-bonds", "Number of H-bonds")
plot_ts("ss_count.xvg", "Secondary structure content",
        "Number of residues", do_smooth=False)
plot_ts("energy.xvg", "Thermodynamics", split=True)

# ---- RMSF per chain ----
p = find("rmsf_calpha.xvg")
if p:
    _, d = read_xvg(p)
    x, y = d[:, 0], d[:, 1]
    br = np.where(np.diff(x) < 0)[0] + 1
    if len(br) == 0:
        br = [len(x) // 3, 2 * len(x) // 3]
    ys = np.split(y, br)
    print("Chains:", len(ys), [len(c) for c in ys])
    L = min(len(c) for c in ys)
    colors = ["#1f77b4", "#d62728", "#2ca02c", "#ff7f0e", "#9467bd", "#8c564b"]
    fig, ax = plt.subplots(figsize=(8, 4.5), dpi=300)
    for i, yi in enumerate(ys):
        ax.plot(np.arange(1, len(yi) + 1), yi, color=colors[i % 6],
                lw=1.2, label=f"Chain {chr(65 + i)}")
    ax.plot(np.arange(1, L + 1), np.mean([c[:L] for c in ys], axis=0),
            "k--", lw=1.5, label="Chain average")
    ax.set_xlabel("Residue"); ax.set_ylabel("RMSF (nm)")
    ax.set_title(f"{run}: C-alpha RMSF per chain")
    ax.legend(frameon=False)
    fig.tight_layout()
    out = os.path.join(graphs, "rmsf_chains.png")
    fig.savefig(out); plt.show(); print("Saved:", out)

# ---- summary table ----
df = pd.DataFrame(stats).round(4)
df.to_csv(os.path.join(graphs, "summary_stats.csv"), index=False)
print(df.to_string(index=False))

In [ ]:
%pip install MDAnalysis

In [ ]:
import sys
print(sys.executable)
print(sys.version)

In [ ]:
!/usr/bin/python3.real -m pip install --break-system-packages MDAnalysis

In [ ]:
import MDAnalysis as mda
print(mda.__version__)

In [ ]:
# ============================================================
# FAST, CHECKPOINTED VICILIN 300 K PER-CHAIN RMSD ANALYSIS
# ============================================================

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import MDAnalysis as mda
from MDAnalysis.analysis import align, rms
from MDAnalysis.lib.distances import calc_bonds
from tqdm.auto import tqdm

# ---------------- PATHS ----------------
proj = "/content/drive/MyDrive/Plant Protein Extrusion MD"
gro = f"{proj}/md_300_r1.gro"
xtc = f"{proj}/md_300_r1.xtc"

graphs = f"{proj}/analysis/vicilin_300_r1/graphs"
os.makedirs(graphs, exist_ok=True)

checkpoint = f"{graphs}/rmsd_checkpoint.csv"
final_csv = f"{graphs}/rmsd_per_chain.csv"
stats_csv = f"{graphs}/rmsd_per_chain_stats.csv"
plot_png = f"{graphs}/rmsd_per_chain.png"

CHECKPOINT_EVERY = 25
LAST_PS = 500

assert os.path.isfile(gro), f"Missing GRO: {gro}"
assert os.path.isfile(xtc), f"Missing XTC: {xtc}"

# ---------------- LOAD TRAJECTORY ----------------
print("Loading topology...")

top = mda.Universe(gro)
protein_top = top.select_atoms("protein")

reader = mda.coordinates.XTC.XTCReader(xtc)
n_xtc = reader.n_atoms
reader.close()

print("Total GRO atoms:", top.atoms.n_atoms)
print("Protein GRO atoms:", protein_top.n_atoms)
print("XTC atoms:", n_xtc)

if n_xtc == top.atoms.n_atoms:
    u = mda.Universe(gro, xtc)
elif n_xtc == protein_top.n_atoms:
    u = mda.Merge(protein_top)
    u.load_new(xtc)
else:
    raise ValueError(
        "XTC atom count matches neither the full GRO nor the protein selection."
    )

prot = u.select_atoms("protein")
bb = prot.select_atoms("backbone")

if len(bb) == 0:
    raise ValueError("No protein backbone atoms found.")

# ---------------- DETECT CHAINS ----------------
# No guess_bonds() or unwrap(): these are unnecessary for
# the RMSD calculation if the coordinates are already suitable.

print("Detecting chains from backbone geometry...")

Cs = prot.select_atoms("name C")
Ns = prot.select_atoms("name N")

if not (len(Cs) == len(Ns) == len(prot.residues)):
    raise ValueError("Unexpected backbone atom counts; inspect topology.")

u.trajectory[0]

dist = calc_bonds(
    Cs.positions[:-1],
    Ns.positions[1:],
    box=u.dimensions
)

resids = prot.residues.resids
breaks = np.where(
    (dist > 2.0) | (np.diff(resids) < 0)
)[0] + 1

chains = [
    prot.residues[idx].atoms
    for idx in np.split(np.arange(len(prot.residues)), breaks)
]

print("Total frames:", len(u.trajectory))
print("Detected chains:", len(chains))
print("Residues per chain:", [len(c.residues) for c in chains])

if len(chains) != 3:
    raise ValueError(
        f"Expected 3 chains, detected {len(chains)}. "
        "Check chain segmentation before proceeding."
    )

chain_bb = [c.select_atoms("backbone") for c in chains]

# Map each chain's backbone atoms into the complex backbone.
masks = [
    np.isin(bb.indices, c.indices)
    for c in chain_bb
]

if not all(mask.any() for mask in masks):
    raise ValueError("At least one chain has no mapped backbone atoms.")

# ---------------- REFERENCE STRUCTURE ----------------
u.trajectory[0]

ref = bb.positions.copy()
ref -= ref.mean(axis=0)

chain_ref = [c.positions.copy() for c in chain_bb]

# ---------------- RESUME CHECKPOINT ----------------
columns = ["frame_index", "time_ps"]

for i in range(len(chains)):
    letter = chr(65 + i)
    columns += [
        f"chain{letter}_fit_complex",
        f"chain{letter}_fit_own"
    ]

if os.path.exists(checkpoint):
    saved = pd.read_csv(checkpoint)

    if not set(columns).issubset(saved.columns):
        raise ValueError(
            "Checkpoint columns do not match this analysis. "
            "Rename or inspect the checkpoint before restarting."
        )

    saved = saved[columns].drop_duplicates(
        subset="frame_index", keep="last"
    ).sort_values("frame_index")

    start_frame = (
        int(saved["frame_index"].iloc[-1]) + 1
        if len(saved) else 0
    )

    print(f"Resuming at frame {start_frame}/{len(u.trajectory)}")
else:
    saved = pd.DataFrame(columns=columns)
    start_frame = 0
    print("No checkpoint found. Starting from frame 0.")

if start_frame >= len(u.trajectory):
    print("All trajectory frames are already checkpointed.")
else:
    # ---------------- RMSD LOOP ----------------
    pending = []
    started = time.time()

    pbar = tqdm(
        total=len(u.trajectory),
        initial=start_frame,
        desc="Calculating RMSD",
        unit="frame"
    )

    try:
        for frame_index, ts in enumerate(u.trajectory):
            if frame_index < start_frame:
                continue

            # Fit entire complex to frame-0 reference.
            P = bb.positions.copy()
            P -= P.mean(axis=0)

            R, _ = align.rotation_matrix(P, ref)
            P_fit = P @ R.T

            row = {
                "frame_index": frame_index,
                "time_ps": float(ts.time)
            }

            # Per-chain RMSD after whole-complex fitting.
            for i, mask in enumerate(masks):
                letter = chr(65 + i)

                row[f"chain{letter}_fit_complex"] = (
                    np.sqrt(
                        np.mean(
                            np.sum(
                                (P_fit[mask] - ref[mask]) ** 2,
                                axis=1
                            )
                        )
                    ) / 10.0
                )

            # Per-chain RMSD after fitting each chain separately.
            for i, (chain, reference) in enumerate(
                zip(chain_bb, chain_ref)
            ):
                letter = chr(65 + i)

                row[f"chain{letter}_fit_own"] = (
                    rms.rmsd(
                        chain.positions,
                        reference,
                        center=True,
                        superposition=True
                    ) / 10.0
                )

            pending.append(row)
            pbar.update(1)

            # Save checkpoint frequently to limit lost work.
            if len(pending) >= CHECKPOINT_EVERY:
                updated = pd.concat(
                    [saved, pd.DataFrame(pending)],
                    ignore_index=True
                )

                temp = checkpoint + ".tmp"
                updated.to_csv(temp, index=False)
                os.replace(temp, checkpoint)

                saved = updated
                pending = []

                elapsed = time.time() - started
                pbar.set_postfix(
                    saved=len(saved),
                    elapsed_min=round(elapsed / 60, 1)
                )

        # Save the final partial batch.
        if pending:
            updated = pd.concat(
                [saved, pd.DataFrame(pending)],
                ignore_index=True
            )

            temp = checkpoint + ".tmp"
            updated.to_csv(temp, index=False)
            os.replace(temp, checkpoint)
            saved = updated

    finally:
        pbar.close()

# ---------------- FINALIZE RESULTS ----------------
# Reload the durable checkpoint rather than relying on memory.
saved = pd.read_csv(checkpoint)
saved = saved.drop_duplicates(
    subset="frame_index", keep="last"
).sort_values("frame_index")

expected = len(u.trajectory)

print(f"Frames saved: {len(saved)}/{expected}")

if len(saved) != expected:
    print(
        "Analysis is incomplete. The checkpoint is preserved; "
        "rerun this cell to resume."
    )
else:
    # Final CSV: same columns as the original analysis.
    output = saved.drop(columns=["frame_index"])
    output.to_csv(final_csv, index=False)

    # ---------------- SUMMARY STATISTICS ----------------
    t = output["time_ps"].to_numpy()
    value_cols = [
        c for c in output.columns if c != "time_ps"
    ]

    recent = t >= t.max() - LAST_PS

    if recent.sum() < 2:
        raise ValueError(
            "Insufficient frames in the final 500 ps for statistics."
        )

    stats = pd.DataFrame({
        name: [
            output.loc[recent, name].mean(),
            output.loc[recent, name].std(),
            np.polyfit(
                t[recent],
                output.loc[recent, name],
                1
            )[0] * 1000
        ]
        for name in value_cols
    }, index=["mean", "sd", "drift_per_ns"]).T.round(4)

    stats.to_csv(stats_csv)

    # ---------------- PLOT ----------------
    colors = [
        "#1f77b4", "#d62728", "#2ca02c",
        "#ff7f0e", "#9467bd", "#8c564b"
    ]

    complex_cols = [
        f"chain{chr(65+i)}_fit_complex"
        for i in range(len(chains))
    ]
    own_cols = [
        f"chain{chr(65+i)}_fit_own"
        for i in range(len(chains))
    ]

    fig, axes = plt.subplots(
        2, 1, figsize=(8, 7),
        sharex=True, dpi=300
    )

    for ax, names, title in [
        (axes[0], complex_cols, "Fit on whole complex"),
        (axes[1], own_cols, "Fit on own chain")
    ]:
        for i, name in enumerate(names):
            ax.plot(
                t, output[name],
                color=colors[i],
                lw=1.0,
                label=f"Chain {chr(65+i)}"
            )

        ax.set_ylabel("Backbone RMSD (nm)")
        ax.set_title(title, fontsize=10)
        ax.legend(frameon=False, fontsize=8)

    axes[1].set_xlabel("Time (ps)")
    fig.suptitle("vicilin_300_r1: per-chain backbone RMSD")
    fig.tight_layout()
    fig.savefig(plot_png, dpi=300)
    plt.show()
    plt.close(fig)

    print("\nAnalysis completed.")
    print("RMSD CSV:", final_csv)
    print("Statistics:", stats_csv)
    print("Graph:", plot_png)
    print("\nStatistics over the final 500 ps:")
    print(stats)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

proj = "/content/drive/MyDrive/Plant Protein Extrusion MD"
folder = f"{proj}/analysis/vicilin_300_r1/graphs"

# Reuse existing RMSD data — no recalculation
df = pd.read_csv(f"{folder}/rmsd_per_chain.csv")
t = df["time_ps"].to_numpy()

fig, axes = plt.subplots(
    2, 1,
    figsize=(16, 8),
    sharex=True,
    dpi=300
)

# Complex-aligned RMSD
for chain in ["A", "B", "C"]:
    axes[0].plot(
        t, df[f"chain{chain}_fit_complex"],
        linewidth=0.65,
        label=f"Chain {chain}"
    )

axes[0].set_ylabel("RMSD (nm)")
axes[0].set_title("RMSD aligned to the protein complex")
axes[0].legend(frameon=False, ncol=3)

# Individually aligned RMSD
for chain in ["A", "B", "C"]:
    axes[1].plot(
        t, df[f"chain{chain}_fit_own"],
        linewidth=0.65,
        label=f"Chain {chain}"
    )

axes[1].set_xlabel("Time (ps)")
axes[1].set_ylabel("RMSD (nm)")
axes[1].set_title("RMSD aligned independently per chain")
axes[1].legend(frameon=False, ncol=3)

for ax in axes:
    ax.grid(True, linewidth=0.35, alpha=0.2)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="out", width=0.7)

fig.tight_layout()

out = f"{folder}/rmsd_per_chain_landscape.png"
fig.savefig(out, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

print("Saved:", out)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

proj = "/content/drive/MyDrive/Plant Protein Extrusion MD"
folder = f"{proj}/analysis/vicilin_300_r1/graphs"

df = pd.read_csv(f"{folder}/rmsd_per_chain.csv")
t = df["time_ps"].to_numpy()

# Graph 1: Complex-aligned RMSD
fig, ax = plt.subplots(figsize=(16, 4.8), dpi=300)

for chain in ["A", "B", "C"]:
    ax.plot(
        t, df[f"chain{chain}_fit_complex"],
        linewidth=0.65,
        label=f"Chain {chain}"
    )

ax.set_xlabel("Time (ps)")
ax.set_ylabel("RMSD (nm)")
ax.set_title("Complex-aligned RMSD", loc="left", pad=12)
ax.legend(frameon=False, ncol=3)
ax.grid(True, linewidth=0.35, alpha=0.2)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(direction="out", width=0.7)
fig.tight_layout()

fig.savefig(
    f"{folder}/rmsd_complex_landscape.png",
    dpi=300, bbox_inches="tight", facecolor="white"
)
plt.show()
plt.close(fig)


# Graph 2: Independently aligned RMSD
fig, ax = plt.subplots(figsize=(16, 4.8), dpi=300)

for chain in ["A", "B", "C"]:
    ax.plot(
        t, df[f"chain{chain}_fit_own"],
        linewidth=0.65,
        label=f"Chain {chain}"
    )

ax.set_xlabel("Time (ps)")
ax.set_ylabel("RMSD (nm)")
ax.set_title("Independently aligned RMSD", loc="left", pad=12)
ax.legend(frameon=False, ncol=3)
ax.grid(True, linewidth=0.35, alpha=0.2)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(direction="out", width=0.7)
fig.tight_layout()

fig.savefig(
    f"{folder}/rmsd_individual_landscape.png",
    dpi=300, bbox_inches="tight", facecolor="white"
)
plt.show()
plt.close(fig)

print("Both separate landscape graphs saved successfully.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# Project paths
proj = "/content/drive/MyDrive/Plant Protein Extrusion MD"
folder = f"{proj}/analysis/vicilin_300_r1/graphs"
os.makedirs(folder, exist_ok=True)

# Load existing RMSD results
df = pd.read_csv(f"{folder}/rmsd_per_chain.csv")
t = df["time_ps"].to_numpy()

# Consistent clean scientific styling
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 10,
    "lines.linewidth": 0.65,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white"
})

# ==========================================
# GRAPH 1: COMPLEX-ALIGNED RMSD
# ==========================================
fig, ax = plt.subplots(figsize=(16, 6.5), dpi=300)

for chain in ["A", "B", "C"]:
    ax.plot(
        t,
        df[f"chain{chain}_fit_complex"],
        linewidth=0.65,
        label=f"Chain {chain}"
    )

ax.set_xlabel("Time (ps)", labelpad=8)
ax.set_ylabel("RMSD (nm)", labelpad=8)
ax.set_title("Complex-Aligned RMSD", loc="left", pad=14)

ax.legend(
    loc="best",
    frameon=False,
    ncol=3,
    handlelength=2.5
)

ax.grid(True, linewidth=0.35, alpha=0.20)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", direction="out", width=0.7, length=4)

fig.tight_layout()

fig.savefig(
    f"{folder}/rmsd_complex_landscape.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)


# ==========================================
# GRAPH 2: INDEPENDENTLY ALIGNED RMSD
# ==========================================
fig, ax = plt.subplots(figsize=(16, 6.5), dpi=300)

for chain in ["A", "B", "C"]:
    ax.plot(
        t,
        df[f"chain{chain}_fit_own"],
        linewidth=0.65,
        label=f"Chain {chain}"
    )

ax.set_xlabel("Time (ps)", labelpad=8)
ax.set_ylabel("RMSD (nm)", labelpad=8)
ax.set_title("Independently Aligned RMSD", loc="left", pad=14)

ax.legend(
    loc="best",
    frameon=False,
    ncol=3,
    handlelength=2.5
)

ax.grid(True, linewidth=0.35, alpha=0.20)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", direction="out", width=0.7, length=4)

fig.tight_layout()

fig.savefig(
    f"{folder}/rmsd_individual_landscape.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)

print("\nBoth RMSD graphs saved successfully:")
print(f"{folder}/rmsd_complex_landscape.png")
print(f"{folder}/rmsd_individual_landscape.png")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

proj = "/content/drive/MyDrive/Plant Protein Extrusion MD"
folder = f"{proj}/analysis/vicilin_300_r1/graphs"
os.makedirs(folder, exist_ok=True)

# Reuse existing RMSD data
df = pd.read_csv(f"{folder}/rmsd_per_chain.csv")
t = df["time_ps"].to_numpy()

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 10,
    "lines.linewidth": 0.65,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white"
})

def style_graph(ax, title):
    ax.set_xlabel("Time (ps)", labelpad=8)
    ax.set_ylabel("RMSD (nm)", labelpad=8)
    ax.set_title(title, loc="left", pad=14)
    ax.legend(frameon=False, ncol=3, handlelength=2.5)
    ax.grid(True, linewidth=0.35, alpha=0.20)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", direction="out", width=0.7, length=4)

# GRAPH 1: Complex-aligned RMSD
fig, ax = plt.subplots(figsize=(12, 6.5), dpi=300)

for chain in ["A", "B", "C"]:
    ax.plot(t, df[f"chain{chain}_fit_complex"],
            linewidth=0.65, label=f"Chain {chain}")

style_graph(ax, "Complex-Aligned RMSD")
fig.tight_layout()
fig.savefig(f"{folder}/rmsd_complex_landscape.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
plt.close(fig)

# GRAPH 2: Independently aligned RMSD
fig, ax = plt.subplots(figsize=(12, 6.5), dpi=300)

for chain in ["A", "B", "C"]:
    ax.plot(t, df[f"chain{chain}_fit_own"],
            linewidth=0.65, label=f"Chain {chain}")

style_graph(ax, "Independently Aligned RMSD")
fig.tight_layout()
fig.savefig(f"{folder}/rmsd_individual_landscape.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
plt.close(fig)

print("Both separate landscape graphs saved successfully.")
print("Complex-aligned:", f"{folder}/rmsd_complex_landscape.png")
print("Independently aligned:", f"{folder}/rmsd_individual_landscape.png")

In [ ]:
import os
os.environ["P"]  = "/content/drive/MyDrive/Plant Protein Extrusion MD"
os.environ["S"]  = "md_300_r2"   # file prefix
os.environ["OUT"] = os.environ["P"] + "/analysism/vicilin_300_r2"
for d in ["graphs","logs","frames","xvgs"]:
    os.makedirs(f'{os.environ["OUT"]}/{d}', exist_ok=True)

In [ ]:
%%bash
cd "$P"
ls -lh $S.tpr $S.xtc $S.edr $S.gro
gmx check -f $S.xtc 2>&1 | grep -E "Last frame|Reading frame|Step|Time" | tail -5
gmx check -e $S.edr 2>&1 | grep -E "Last frame|Reading frame" | tail -3

In [ ]:
import os
os.environ["PATH"] = "/content/gmx/bin:" + os.environ["PATH"]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ["P"]   = "/content/drive/MyDrive/Plant Protein Extrusion MD"
os.environ["S"]   = "md_300_r2"
os.environ["OUT"] = os.environ["P"] + "/analysism/vicilin_300_r2"
for d in ["graphs","logs","frames","xvgs"]:
    os.makedirs(f'{os.environ["OUT"]}/{d}', exist_ok=True)

In [ ]:
%%bash
ls /content/gmx/bin/gmx 2>&1
ls /content/bin/micromamba 2>&1
which gmx || echo "gmx not on PATH"

In [ ]:
%%bash
which gmx
gmx --version | head -3

In [ ]:
%%bash
ls -d /usr/local/envs/gmx_cuda/bin/gmx /opt/conda/envs/gmx_cuda/bin/gmx 2>/dev/null
G=$(ls /usr/local/envs/gmx_cuda/bin/gmx /opt/conda/envs/gmx_cuda/bin/gmx 2>/dev/null | head -1)
echo "found: $G"
ln -sf "$G" /usr/local/bin/gmx
gmx --version | head -5

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ["P"]   = "/content/drive/MyDrive/Plant Protein Extrusion MD"
os.environ["S"]   = "md_300_r2"
os.environ["OUT"] = os.environ["P"] + "/analysism/vicilin_300_r2"
for d in ["graphs","logs","frames","xvgs"]:
    os.makedirs(f'{os.environ["OUT"]}/{d}', exist_ok=True)
print(os.listdir(os.environ["OUT"]))

In [ ]:
%%bash
cd "$P"
ls -lh $S.*
gmx check -f $S.xtc 2>&1 | tail -25
echo ---- TPR ----
gmx dump -s $S.tpr 2>/dev/null | grep -E "^\s+(nsteps|dt|nstxout-compressed)\s"

In [ ]:
import os
os.system('ln -sfn "/content/drive/MyDrive/Plant Protein Extrusion MD" /content/pm')
os.environ["P"]   = "/content/pm"
os.environ["S"]   = "md_300_r2"
os.environ["OUT"] = "/content/pm/analysism/vicilin_300_r2"
for d in ["graphs","logs","frames","xvgs"]:
    os.makedirs(f'{os.environ["OUT"]}/{d}', exist_ok=True)
print(os.listdir("/content/pm/analysism/vicilin_300_r2"))

In [ ]:
%%bash
cd "$P"
F="$OUT/frames"

printf "Protein\n" | gmx convert-tpr -s $S.tpr -o "$F/protein.tpr" 2>&1 | tail -4
gmx dump -s "$F/protein.tpr" 2>/dev/null | grep -m1 natoms

printf "Protein\n" | gmx trjconv -s "$F/protein.tpr" -f $S.xtc -o "$F/whole.xtc" -pbc whole 2>&1 | tail -2
printf "Protein\n" | gmx trjconv -s "$F/protein.tpr" -f "$F/whole.xtc" -o "$F/nojump.xtc" -pbc nojump 2>&1 | tail -2
printf "Backbone\nProtein\n" | gmx trjconv -s "$F/protein.tpr" -f "$F/nojump.xtc" -o "$F/fit_skip5.xtc" -fit rot+trans -skip 5 2>&1 | tail -2

rm -f "$F/whole.xtc" "$F/nojump.xtc"
gmx check -f "$F/fit_skip5.xtc" 2>&1 | grep -E "# Atoms|Last frame|Step" || true

In [ ]:
%%bash
cd "$P"
F="$OUT/frames"
printf "Backbone\nBackbone\n" | gmx rms -s "$F/protein.tpr" -f "$F/fit_skip5.xtc" \
  -o "$OUT/xvgs/rmsd_backbone.xvg" -tu ns 2>&1 | tee "$OUT/logs/rmsd.log" | tail -2
grep -E "^@ (title|xaxis|yaxis)" "$OUT/xvgs/rmsd_backbone.xvg"
tail -2 "$OUT/xvgs/rmsd_backbone.xvg"

In [ ]:
%%bash
cd "$OUT/frames"
printf "splitch 1\nq\n" | gmx make_ndx -f protein.tpr -o chains.ndx 2>&1 | grep -E "^ *[0-9]+ " | tail -15

In [ ]:
%%bash
cd "$P"
# gro line = atom number + 2; print the last atom of a chain and the first of the next
for a in 1 6647 6648 13294 13295 19941; do sed -n "$((a+2))p" $S.gro | cut -c1-20; done

In [ ]:
import os
P, S, OUT = os.environ["P"], os.environ["S"], os.environ["OUT"]
L = open(f"{P}/{S}.gro").read().split("\n")
atoms = L[2:2+19941]
name = [a[10:15].strip() for a in atoms]

chains = {"chA": range(1,6648), "chB": range(6648,13295), "chC": range(13295,19942)}
groups = {}
for c, r in chains.items():
    groups[c]          = list(r)
    groups[c+"_bb"]    = [i for i in r if name[i-1] in ("N","CA","C")]
    groups[c+"_CA"]    = [i for i in r if name[i-1] == "CA"]

with open(f"{OUT}/frames/chains.ndx", "w") as f:
    for g, idx in groups.items():
        f.write(f"[ {g} ]\n")
        for k in range(0, len(idx), 15):
            f.write(" ".join(f"{i:4d}" for i in idx[k:k+15]) + "\n")
for g, idx in groups.items(): print(g, len(idx))

In [ ]:
%%bash
cd "$P"
F="$OUT/frames"
for c in chA chB chC; do
  printf "${c}_bb\n${c}_bb\n" | gmx rms -s "$F/protein.tpr" -f "$F/fit_skip5.xtc" -n "$F/chains.ndx" \
    -o "$OUT/xvgs/rmsd_${c}.xvg" -tu ns > "$OUT/logs/rmsd_${c}.log" 2>&1
  echo $c; tail -1 "$OUT/xvgs/rmsd_${c}.xvg"
done

In [ ]:
%%bash
cd "$P"
F="$OUT/frames"
for c in chA chB chC; do
  printf "${c}_CA\n${c}_CA\n" | gmx rmsf -s "$F/protein.tpr" -f "$F/fit_skip5.xtc" -n "$F/chains.ndx" \
    -o "$OUT/xvgs/rmsf_${c}.xvg" -res > "$OUT/logs/rmsf_${c}.log" 2>&1
  echo $c; grep -vc "^[@#]" "$OUT/xvgs/rmsf_${c}.xvg"
done

In [ ]:
import numpy as np, matplotlib.pyplot as plt, os
OUT = os.environ["OUT"]
def xvg(f): return np.loadtxt(f, comments=('#','@'))
sty = {"chA": dict(color="#d62728", ls="-",  label="Chain A"),
       "chB": dict(color="#7fbfff", ls=":",  label="Chain B"),
       "chC": dict(color="#d62728", ls=":",  label="Chain C")}

def plot(prefix, xlabel, ylabel, title, out, tcol=0):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        d = xvg(f"{OUT}/xvgs/{prefix}_{c}.xvg")
        ax.plot(d[:,0], d[:,1], lw=1.8, **s)
    ax.set(xlabel=xlabel, ylabel=ylabel, title=title)
    ax.legend(); fig.tight_layout()
    fig.savefig(f"{OUT}/graphs/{out}.png"); plt.show()

plot("rmsd", "Time (ns)", "RMSD (nm)", "Backbone RMSD at 300 K, 1 ns (r2)", "rmsd_per_chain")
plot("rmsf", "Residue", "RMSF (nm)", "C-alpha RMSF per residue at 300 K (r2)", "rmsf_per_chain")

In [ ]:
%%bash
cd "$P"
F="$OUT/frames"
printf "Backbone\nBackbone\n" | gmx rms -s "$F/protein.tpr" -f "$F/fit_skip5.xtc" \
  -o "$OUT/xvgs/rmsd_protein.xvg" -tu ns > "$OUT/logs/rmsd_protein.log" 2>&1
tail -1 "$OUT/xvgs/rmsd_protein.xvg"

In [ ]:
import numpy as np, matplotlib.pyplot as plt, os
OUT = os.environ["OUT"]
def xvg(f): return np.loadtxt(f, comments=('#','@'))
p = xvg(f"{OUT}/xvgs/rmsd_protein.xvg")

# combined only
fig, ax = plt.subplots(figsize=(12,5), dpi=150)
ax.plot(p[:,0], p[:,1], color="black", lw=2, label="Whole protein (backbone)")
ax.set(xlabel="Time (ns)", ylabel="RMSD (nm)", title="Backbone RMSD, whole trimer, 300 K (r2)")
ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/rmsd_protein.png"); plt.show()

# combined + chains
sty = {"chA": dict(color="#d62728", ls="-", label="Chain A"),
       "chB": dict(color="#7fbfff", ls=":", label="Chain B"),
       "chC": dict(color="#d62728", ls=":", label="Chain C")}
fig, ax = plt.subplots(figsize=(12,5), dpi=150)
ax.plot(p[:,0], p[:,1], color="black", lw=2.2, label="Whole protein")
for c, s in sty.items():
    d = xvg(f"{OUT}/xvgs/rmsd_{c}.xvg"); ax.plot(d[:,0], d[:,1], lw=1.6, **s)
ax.set(xlabel="Time (ns)", ylabel="RMSD (nm)", title="Backbone RMSD: whole trimer and each chain, 300 K (r2)")
ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/rmsd_protein_and_chains.png"); plt.show()

In [ ]:
%%bash
cd "$P"
F="$OUT/frames"
printf "Protein\n" | gmx trjconv -s "$F/protein.tpr" -f "$F/fit_skip5.xtc" -o "$F/first.gro" -dump 0 2>&1 | tail -1
pip install -q MDAnalysis

In [ ]:
import os, numpy as np, MDAnalysis as mda, matplotlib.pyplot as plt
from MDAnalysis.analysis.align import rotation_matrix
OUT = os.environ["OUT"]
u = mda.Universe(f"{OUT}/frames/first.gro", f"{OUT}/frames/fit_skip5.xtc")

chains = {"chA": slice(0,6647), "chB": slice(6647,13294), "chC": slice(13294,19941)}
res = {}
for name, sl in chains.items():
    ch = u.atoms[sl]
    bb = ch.select_atoms("name N CA C"); ca = ch.select_atoms("name CA")
    u.trajectory[0]
    c0 = bb.positions.mean(0); bb0 = bb.positions - c0; ca0 = ca.positions - c0
    d2 = np.zeros(len(ca)); n = 0
    for ts in u.trajectory:                      # fit each chain on its own backbone
        c = bb.positions.mean(0)
        R, _ = rotation_matrix(bb.positions - c, bb0)
        d2 += (((ca.positions - c) @ R.T - ca0)**2).sum(1); n += 1
    rms = np.sqrt(d2/n) / 10.0                   # Angstrom -> nm
    res[name] = (ca.resids, rms)
    with open(f"{OUT}/xvgs/rmsd_perres_{name}.xvg","w") as f:
        f.write('@ title "RMSD per residue"\n@ xaxis label "Residue"\n@ yaxis label "RMSD (nm)"\n')
        for r, v in zip(ca.resids, rms): f.write(f"{r} {v:.6f}\n")
    print(name, len(rms), round(rms.mean(),3))

sty = {"chA": dict(color="#d62728", ls="-", label="Chain A"),
       "chB": dict(color="#7fbfff", ls=":", label="Chain B"),
       "chC": dict(color="#d62728", ls=":", label="Chain C")}
fig, ax = plt.subplots(figsize=(12,5), dpi=150)
for c, s in sty.items(): ax.plot(*res[c], lw=1.8, **s)
mean = np.mean([res[c][1] for c in res], axis=0)
ax.plot(res["chA"][0], mean, color="black", lw=2.2, label="Mean of chains")
ax.set(xlabel="Residue", ylabel="RMSD (nm)", title="C-alpha RMSD per residue vs first frame, 300 K (r2)")
ax.legend(); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/rmsd_per_residue.png"); plt.show()

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
OUT = os.environ["OUT"]
def xvg(f): return np.loadtxt(f, comments=('#','@'))
d = {c: xvg(f"{OUT}/xvgs/rmsd_perres_{c}.xvg") for c in ["chA","chB","chC"]}

sty = {"chA": dict(color="#d62728", ls="-", label="Chain A"),
       "chB": dict(color="#7fbfff", ls=":", label="Chain B"),
       "chC": dict(color="#d62728", ls=":", label="Chain C")}

# 1) per chain, no mean
fig, ax = plt.subplots(figsize=(12,5), dpi=150)
for c, s in sty.items():
    ax.plot(d[c][:,0], d[c][:,1], lw=1.8, **s)
ax.set(xlabel="Residue", ylabel="RMSD (nm)", title="C-alpha RMSD per residue vs first frame, 300 K (r2)")
ax.legend(); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/rmsd_per_residue.png"); plt.show()

# 2) mean of chains only, grey
res = d["chA"][:,0]
mean = np.mean([d[c][:,1] for c in d], axis=0)
with open(f"{OUT}/xvgs/rmsd_perres_mean.xvg","w") as f:
    f.write('@ title "Mean RMSD per residue"\n@ xaxis label "Residue"\n@ yaxis label "RMSD (nm)"\n')
    for r, v in zip(res, mean): f.write(f"{int(r)} {v:.6f}\n")

fig, ax = plt.subplots(figsize=(12,5), dpi=150)
ax.plot(res, mean, color="grey", lw=2, label="Mean of chains")
ax.set(xlabel="Residue", ylabel="RMSD (nm)", title="Mean C-alpha RMSD per residue (3 chains), 300 K (r2)")
ax.legend(); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/rmsd_per_residue_mean.png"); plt.show()

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
OUT = os.environ["OUT"]
def xvg(f): return np.loadtxt(f, comments=('#','@'))

sty = {"chA": dict(color="#d62728", ls="-", label="Chain A"),
       "chB": dict(color="#7fbfff", ls=":", label="Chain B"),
       "chC": dict(color="#08306b", ls=":", label="Chain C")}

def plot(prefix, xlabel, ylabel, title, out, grey=None):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        d = xvg(f"{OUT}/xvgs/{prefix}_{c}.xvg")
        ax.plot(d[:,0], d[:,1], lw=1.8, **s)
    ax.set(xlabel=xlabel, ylabel=ylabel, title=title)
    ax.legend(); fig.tight_layout()
    fig.savefig(f"{OUT}/graphs/{out}.png"); plt.show()

plot("rmsd",        "Time (ns)", "RMSD (nm)", "Backbone RMSD at 300 K, 1 ns (r2)",              "rmsd_per_chain")
plot("rmsf",        "Residue",   "RMSF (nm)", "C-alpha RMSF per residue at 300 K (r2)",           "rmsf_per_chain")
plot("rmsd_perres", "Residue",   "RMSD (nm)", "C-alpha RMSD per residue vs first frame, 300 K (r2)", "rmsd_per_residue")

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
OUT = os.environ["OUT"]
def xvg(f): return np.loadtxt(f, comments=('#','@'))

sty = {"chA": dict(color="#d62728", ls="-", label="Chain A"),
       "chB": dict(color="#7fbfff", ls=":", label="Chain B"),
       "chC": dict(color="#08306b", ls="-", label="Chain C")}

def plot(prefix, xlabel, ylabel, title, out):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        d = xvg(f"{OUT}/xvgs/{prefix}_{c}.xvg")
        ax.plot(d[:,0], d[:,1], lw=1.8, **s)
    ax.set(xlabel=xlabel, ylabel=ylabel, title=title)
    ax.legend(); fig.tight_layout()
    fig.savefig(f"{OUT}/graphs/{out}.png"); plt.show()

plot("rmsd",        "Time (ns)", "RMSD (nm)", "Backbone RMSD at 300 K, 1 ns (r2)",                  "rmsd_per_chain")
plot("rmsf",        "Residue",   "RMSF (nm)", "C-alpha RMSF per residue at 300 K (r2)",              "rmsf_per_chain")
plot("rmsd_perres", "Residue",   "RMSD (nm)", "C-alpha RMSD per residue vs first frame, 300 K (r2)", "rmsd_per_residue")

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
OUT = os.environ["OUT"]
def xvg(f): return np.loadtxt(f, comments=('#','@'))

sty = {"chA": dict(color="#d62728", ls="-", label="Chain A"),
       "chB": dict(color="#7fbfff", ls=":", label="Chain B"),
       "chC": dict(color="#1f77b4", ls="-", label="Chain C")}

def plot(prefix, xlabel, ylabel, title, out):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        d = xvg(f"{OUT}/xvgs/{prefix}_{c}.xvg")
        ax.plot(d[:,0], d[:,1], lw=1.8, **s)
    ax.set(xlabel=xlabel, ylabel=ylabel, title=title)
    ax.legend(); fig.tight_layout()
    fig.savefig(f"{OUT}/graphs/{out}.png"); plt.show()

plot("rmsd",        "Time (ns)", "RMSD (nm)", "Backbone RMSD at 300 K, 1 ns (r2)",                  "rmsd_per_chain")
plot("rmsf",        "Residue",   "RMSF (nm)", "C-alpha RMSF per residue at 300 K (r2)",              "rmsf_per_chain")
plot("rmsd_perres", "Residue",   "RMSD (nm)", "C-alpha RMSD per residue vs first frame, 300 K (r2)", "rmsd_per_residue")

In [ ]:
%%bash
cd "$OUT/frames"
grep -q "\[ Protein \]" chains.ndx || { echo "[ Protein ]"; seq 1 19941 | paste -sd' ' | fold -w 80 -s; } >> chains.ndx
for c in chA chB chC Protein; do
  printf "$c\n" | gmx gyrate -s protein.tpr -f fit_skip5.xtc -n chains.ndx \
    -o "$OUT/xvgs/rg_${c}.xvg" > "$OUT/logs/rg_${c}.log" 2>&1
  echo $c; tail -1 "$OUT/xvgs/rg_${c}.xvg"
done

In [ ]:
%%bash
cd "$OUT/frames"
for c in chA chB chC Protein; do
  printf "Protein\n$c\n" | gmx sasa -s protein.tpr -f fit_skip5.xtc -n chains.ndx \
    -o "$OUT/xvgs/sasa_${c}.xvg" -or "$OUT/xvgs/sasa_perres_${c}.xvg" -tu ns \
    > "$OUT/logs/sasa_${c}.log" 2>&1
  echo $c; tail -1 "$OUT/xvgs/sasa_${c}.xvg"; grep -vc "^[@#]" "$OUT/xvgs/sasa_perres_${c}.xvg"
done

In [ ]:
%%bash
cd "$OUT/frames"
for c in chA chB chC Protein; do
  gmx hbond -s protein.tpr -f fit_skip5.xtc -n chains.ndx -r "group $c" -t "group $c" \
    -num "$OUT/xvgs/hbnum_${c}.xvg" -tu ns > "$OUT/logs/hbond_${c}.log" 2>&1
  echo $c; tail -1 "$OUT/xvgs/hbnum_${c}.xvg"
done

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
OUT = os.environ["OUT"]
def xvg(f): return np.loadtxt(f, comments=('#','@'))
sty = {"chA": dict(color="#d62728", ls="-", label="Chain A"),
       "chB": dict(color="#7fbfff", ls=":", label="Chain B"),
       "chC": dict(color="#1f77b4", ls="-", label="Chain C")}

def plot(prefix, xlabel, ylabel, title, out, ps2ns=False):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        d = xvg(f"{OUT}/xvgs/{prefix}_{c}.xvg")
        x = d[:,0]/1000 if ps2ns else d[:,0]
        ax.plot(x, d[:,1], lw=1.8, **s)
    ax.set(xlabel=xlabel, ylabel=ylabel, title=title)
    ax.legend(); fig.tight_layout()
    fig.savefig(f"{OUT}/graphs/{out}.png"); plt.show()

plot("rg",          "Time (ns)", "Rg (nm)",          "Radius of gyration per chain, 300 K (r2)", "rg_per_chain", ps2ns=True)
plot("sasa",        "Time (ns)", "SASA (nm²)",       "SASA per chain, 300 K (r2)",               "sasa_per_chain")
plot("sasa_perres", "Residue",   "SASA (nm²)",       "SASA per residue (time-averaged), 300 K (r2)", "sasa_per_residue")
plot("hbnum",       "Time (ns)", "Number of H-bonds","Intra-chain H-bonds per chain, 300 K (r2)", "hbonds_per_chain")

In [ ]:
%%bash
cd "$OUT/xvgs"
for c in chA chB chC Protein; do echo $c $(grep -vc "^[@#]" sasa_perres_$c.xvg); done
md5sum sasa_perres_ch?.xvg

In [ ]:
import numpy as np, os
OUT = os.environ["OUT"]
def xvg(f): return np.loadtxt(f, comments=('#','@'))
tot  = xvg(f"{OUT}/xvgs/sasa_Protein.xvg")[:,1].mean()
ch   = {c: xvg(f"{OUT}/xvgs/sasa_{c}.xvg")[:,1].mean() for c in ["chA","chB","chC"]}
pr   = xvg(f"{OUT}/xvgs/sasa_perres_Protein.xvg")[:,1]
print("whole trimer total      :", round(tot,2))
print("sum of 3 chain totals   :", round(sum(ch.values()),2), ch)
print("sum of per-residue file :", round(pr.sum(),2))
for i,c in enumerate(["chA","chB","chC"]):
    print(c, "per-residue block sum:", round(pr[i*415:(i+1)*415].sum(),2))

In [ ]:
%%bash
cd "$OUT/frames"
for c in chA chB chC; do
  gmx sasa -s protein.tpr -f fit_skip5.xtc -n chains.ndx \
    -surface "group Protein" -output "group $c" \
    -o "$OUT/xvgs/sasa_${c}.xvg" -or "$OUT/xvgs/tmp_perres_${c}.xvg" -tu ns \
    > "$OUT/logs/sasa_${c}.log" 2>&1
  echo $c $(grep -vc "^[@#]" "$OUT/xvgs/tmp_perres_${c}.xvg") rows; tail -1 "$OUT/xvgs/sasa_${c}.xvg"
done

In [ ]:
%%bash
cd "$OUT/xvgs"
ls rg_*.xvg sasa_*.xvg sasa_perres_Protein.xvg hbnum_*.xvg 2>&1
ls "$OUT/frames/first.gro" 2>&1

In [ ]:
import os, glob, numpy as np, matplotlib.pyplot as plt
OUT = os.environ["OUT"]
for f in glob.glob(f"{OUT}/xvgs/tmp_*.xvg"): os.remove(f)
def xvg(f): return np.loadtxt(f, comments=('#','@'))
sty = {"chA": dict(color="#d62728", ls="-", label="Chain A"),
       "chB": dict(color="#7fbfff", ls=":", label="Chain B"),
       "chC": dict(color="#1f77b4", ls="-", label="Chain C")}

# per residue: split the 1245-row trimer file into 3 chains
d = xvg(f"{OUT}/xvgs/sasa_perres_Protein.xvg"); assert len(d) == 1245
for i, c in enumerate(sty):
    with open(f"{OUT}/xvgs/sasa_perres_{c}.xvg","w") as f:
        f.write('@ xaxis label "Residue"\n@ yaxis label "SASA (nm^2)"\n')
        for r in range(415): f.write(f"{r+1} {d[i*415+r,1]:.6f}\n")

fig, ax = plt.subplots(figsize=(12,5), dpi=150)
for c, s in sty.items():
    a = xvg(f"{OUT}/xvgs/sasa_perres_{c}.xvg"); ax.plot(a[:,0], a[:,1], lw=1.6, **s)
ax.set(xlabel="Residue", ylabel="SASA (nm²)", title="SASA per residue (time-averaged), 300 K (r2)")
ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/sasa_per_residue.png"); plt.show()

# per chain vs time: column 2 is the chain's own area
fig, ax = plt.subplots(figsize=(12,5), dpi=150)
for c, s in sty.items():
    a = xvg(f"{OUT}/xvgs/sasa_{c}.xvg"); ax.plot(a[:,0], a[:,2], lw=1.8, **s)
    print(c, "mean SASA:", round(a[:,2].mean(),1))
ax.set(xlabel="Time (ns)", ylabel="SASA (nm²)", title="SASA per chain, 300 K (r2)")
ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/sasa_per_chain.png"); plt.show()

In [ ]:
def whole(prefix, yl, title, out, ps2ns=False):
    a = xvg(f"{OUT}/xvgs/{prefix}_Protein.xvg"); x = a[:,0]/1000 if ps2ns else a[:,0]
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    ax.plot(x, a[:,1], color="grey", lw=2, label="Whole protein")
    ax.set(xlabel="Time (ns)", ylabel=yl, title=title); ax.legend(); fig.tight_layout()
    fig.savefig(f"{OUT}/graphs/{out}.png"); plt.show()

whole("rg",    "Rg (nm)",    "Radius of gyration, whole trimer, 300 K (r2)", "rg_protein", ps2ns=True)
whole("sasa",  "SASA (nm²)", "SASA, whole trimer, 300 K (r2)",               "sasa_protein")
whole("hbnum", "H-bonds",    "H-bonds, whole trimer, 300 K (r2)",            "hbonds_protein")

fig, ax = plt.subplots(figsize=(12,5), dpi=150)
for c, s in sty.items():
    a = xvg(f"{OUT}/xvgs/rg_{c}.xvg"); ax.plot(a[:,0]/1000, a[:,1], lw=1.8, **s)
ax.set(xlabel="Time (ns)", ylabel="Rg (nm)", title="Radius of gyration per chain, 300 K (r2)")
ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/rg_per_chain.png"); plt.show()

In [ ]:
%%bash
cd "$OUT/frames"
printf "Protein\n" | gmx trjconv -s protein.tpr -f fit_skip5.xtc -o first.gro -dump 0 2>&1 | tail -1
pip install -q MDAnalysis scipy

In [ ]:
import MDAnalysis as mda
from scipy.spatial import cKDTree
u = mda.Universe(f"{OUT}/frames/first.gro", f"{OUT}/frames/fit_skip5.xtc")
nm = np.array([n.strip() for n in u.atoms.names]); rn = u.atoms.resnames
resid = u.atoms.resids; chain = np.arange(len(nm))//6647
isH = np.array([n.startswith("H") for n in nm])
P0 = u.atoms.positions/10; heavy = np.where(~isH)[0]; Hs = np.where(isH)[0]
don = heavy[cKDTree(P0[heavy]).query(P0[Hs])[1]]
k = np.array([nm[i][0] in "NO" for i in don]); Hs, don = Hs[k], don[k]
acc = np.array([i for i in range(len(nm)) if nm[i][0]=="O" or (nm[i] in ("ND1","NE2") and rn[i].startswith("HI"))])
tot = np.zeros((3,416)); inter = np.zeros((3,416)); nb = []
for ts in u.trajectory:
    P = u.atoms.positions/10; n = 0
    for hi, lst in enumerate(cKDTree(P[Hs]).query_ball_tree(cKDTree(P[acc]), 0.25)):
        D = don[hi]
        for aj in lst:
            A = acc[aj]
            if A == D or (chain[A]==chain[D] and resid[A]==resid[D]): continue
            v1, v2 = P[Hs[hi]]-P[D], P[A]-P[D]; dist = np.linalg.norm(v2)
            if dist > 0.35 or v1@v2/(np.linalg.norm(v1)*dist) < np.cos(np.radians(30)): continue
            n += 1
            for X in (D, A):
                tot[chain[X], resid[X]] += 1
                if chain[D] != chain[A]: inter[chain[X], resid[X]] += 1
    nb.append(n)
nf = len(nb); print("mean H-bonds/frame:", np.mean(nb))
for name, arr, title in (("hbperres",tot,"All H-bonds per residue"), ("hbperres_inter",inter,"Inter-chain H-bonds per residue")):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for i, (c, s) in enumerate(sty.items()):
        with open(f"{OUT}/xvgs/{name}_{c}.xvg","w") as f:
            f.write('@ xaxis label "Residue"\n@ yaxis label "H-bonds per frame"\n')
            for r in range(1,416): f.write(f"{r} {arr[i,r]/nf:.5f}\n")
        ax.plot(range(1,416), arr[i,1:]/nf, lw=1.6, **s)
    ax.set(xlabel="Residue", ylabel="H-bonds per frame", title=f"{title}, 300 K (r2)")
    ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/{name}.png"); plt.show()

In [ ]:
%%bash
cd "$OUT/frames"
for p in "chA chB" "chB chC" "chA chC"; do set -- $p
  printf "$1\n$2\n" | gmx mindist -s protein.tpr -f fit_skip5.xtc -n chains.ndx \
    -od mind_$1_$2.xvg -tu ns > "$OUT/logs/mind_$1_$2.log" 2>&1
  gmx hbond -s protein.tpr -f fit_skip5.xtc -n chains.ndx -r "group $1" -t "group $2" \
    -num hb_$1_$2.xvg -tu ns > "$OUT/logs/hb_$1_$2.log" 2>&1
  echo "$1-$2  min dist (nm): $(tail -1 mind_$1_$2.xvg)   H-bonds: $(tail -1 hb_$1_$2.xvg)"
done

In [ ]:
import MDAnalysis as mda, numpy as np
from scipy.spatial import cKDTree
u = mda.Universe(f"{OUT}/frames/first.gro", f"{OUT}/frames/fit_skip5.xtc")
P = u.atoms.positions/10
ch = {"A":P[:6647], "B":P[6647:13294], "C":P[13294:]}
for a,b in (("A","B"),("B","C"),("A","C")):
    print(a+b, "raw min dist (nm):", round(cKDTree(ch[a]).query(ch[b])[0].min(),2))
print("box (nm):", u.dimensions[:3]/10)

In [ ]:
%%bash
cd "$OUT/frames"
printf "Protein\n" | gmx trjconv -s protein.tpr -f "$P/$S.xtc" -o whole2.xtc -pbc whole 2>&1 | tail -1
printf "Protein\nProtein\n" | gmx trjconv -s protein.tpr -f whole2.xtc -o cluster.xtc -pbc cluster 2>&1 | tail -1
printf "Backbone\nProtein\n" | gmx trjconv -s protein.tpr -f cluster.xtc -o fit_cluster_skip5.xtc -fit rot+trans -skip 5 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f fit_cluster_skip5.xtc -o first_cl.gro -dump 0 2>&1 | tail -1
rm -f whole2.xtc cluster.xtc
gmx check -f fit_cluster_skip5.xtc 2>&1 | grep -E "Last frame|Step"

In [ ]:
import os, numpy as np, MDAnalysis as mda
from scipy.spatial import cKDTree
OUT = os.environ["OUT"]
u = mda.Universe(f"{OUT}/frames/first_cl.gro", f"{OUT}/frames/fit_cluster_skip5.xtc")
worst = {"AB":0,"BC":0,"AC":0}
for ts in u.trajectory:
    P = u.atoms.positions/10
    ch = {"A":P[:6647], "B":P[6647:13294], "C":P[13294:]}
    for k in worst:
        worst[k] = max(worst[k], cKDTree(ch[k[0]]).query(ch[k[1]])[0].min())
print({k: round(v,2) for k,v in worst.items()})

In [ ]:
%%bash
cd "$OUT/frames"
printf "Backbone\nBackbone\n" | gmx rms -s protein.tpr -f fit_cluster_skip5.xtc \
  -o "$OUT/xvgs/rmsd_protein.xvg" -tu ns > "$OUT/logs/rmsd_protein.log" 2>&1
tail -1 "$OUT/xvgs/rmsd_protein.xvg"

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
OUT = os.environ["OUT"]
def xvg(f): return np.loadtxt(f, comments=('#','@'))
p = xvg(f"{OUT}/xvgs/rmsd_protein.xvg")

fig, ax = plt.subplots(figsize=(12,5), dpi=150)
ax.plot(p[:,0], p[:,1], color="grey", lw=2, label="Whole trimer (backbone)")
ax.set(xlabel="Time (ns)", ylabel="RMSD (nm)", title="Backbone RMSD, whole trimer, 300 K (r2)")
ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/rmsd_protein.png"); plt.show()

sty = {"chA": dict(color="#d62728", ls="-", label="Chain A"),
       "chB": dict(color="#7fbfff", ls=":", label="Chain B"),
       "chC": dict(color="#1f77b4", ls="-", label="Chain C")}
fig, ax = plt.subplots(figsize=(12,5), dpi=150)
ax.plot(p[:,0], p[:,1], color="grey", lw=2.2, label="Whole trimer")
for c, s in sty.items():
    d = xvg(f"{OUT}/xvgs/rmsd_{c}.xvg"); ax.plot(d[:,0], d[:,1], lw=1.6, **s)
ax.set(xlabel="Time (ns)", ylabel="RMSD (nm)", title="Backbone RMSD: whole trimer and each chain, 300 K (r2)")
ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/rmsd_protein_and_chains.png"); plt.show()

## 8. Per-run analysis cells
The same pipeline run for 300 K r2, 373 K r1, 373 K shear r1 and 300 K r1. Set `S` (file prefix) and `OUT` (output folder) at the top of each block.

In [ ]:
import os
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
os.system('ln -sfn "/content/drive/MyDrive/Plant Protein Extrusion MD" /content/pm')
os.environ.update(P="/content/pm", S="md_300_r2", OUT="/content/pm/analysism/vicilin_300_r2")
for d in ["graphs","logs","frames","xvgs"]: os.makedirs(f'{os.environ["OUT"]}/{d}', exist_ok=True)
os.system('pip install -q MDAnalysis scipy; gmx --version | head -1; ls $P/$S.*')

In [ ]:
%%bash
cd "$OUT/frames"
printf "Protein\n" | gmx convert-tpr -s "$P/$S.tpr" -o protein.tpr 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f "$P/$S.xtc" -o whole2.xtc -pbc whole 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f whole2.xtc -o first_w.gro -dump 0 2>&1 | tail -1

In [ ]:
import numpy as np, MDAnalysis as mda
from MDAnalysis.lib.distances import minimize_vectors
OUT = os.environ["OUT"]
u = mda.Universe(f"{OUT}/frames/first_w.gro", f"{OUT}/frames/whole2.xtc")
sl = {"A":slice(0,6647), "B":slice(6647,13294), "C":slice(13294,19941)}
with mda.Writer(f"{OUT}/frames/together.xtc", u.atoms.n_atoms) as W:
    for ts in u.trajectory[::5]:
        pos = u.atoms.positions.copy(); box = ts.dimensions.astype(np.float32)
        a0 = pos[sl["A"]].mean(0)
        for k in "BC":
            d = pos[sl[k]].mean(0) - a0
            dm = minimize_vectors(d[None,:].astype(np.float32), box)[0]
            pos[sl[k]] -= (d - dm)
        u.atoms.positions = pos; W.write(u.atoms)

In [ ]:
%%bash
cd "$OUT/frames"
printf "Backbone\nProtein\n" | gmx trjconv -s protein.tpr -f together.xtc -o fit_cluster_skip5.xtc -fit rot+trans 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f fit_cluster_skip5.xtc -o first_cl.gro -dump 0 2>&1 | tail -1
rm -f whole2.xtc together.xtc
gmx check -f fit_cluster_skip5.xtc 2>&1 | grep -E "Last frame|Step"

In [ ]:
from scipy.spatial import cKDTree
u = mda.Universe(f"{OUT}/frames/first_cl.gro", f"{OUT}/frames/fit_cluster_skip5.xtc")
worst = {"AB":0,"BC":0,"AC":0}
for ts in u.trajectory:
    Pp = u.atoms.positions/10; chp = {k:Pp[s] for k,s in sl.items()}
    for p in worst: worst[p] = max(worst[p], cKDTree(chp[p[0]]).query(chp[p[1]])[0].min())
print({k: round(float(v),2) for k,v in worst.items()})
assert max(worst.values()) < 0.5, "chains NOT together - stop and send me this line"

nm = [a[10:15].strip() for a in open(f"{OUT}/frames/first_cl.gro").read().split("\n")[2:2+19941]]
G = {}
for c, r in {"chA":range(1,6648),"chB":range(6648,13295),"chC":range(13295,19942),"Protein":range(1,19942)}.items():
    G[c] = list(r); G[c+"_bb"] = [i for i in r if nm[i-1] in ("N","CA","C")]; G[c+"_CA"] = [i for i in r if nm[i-1]=="CA"]
with open(f"{OUT}/frames/chains.ndx","w") as f:
    for g, idx in G.items():
        f.write(f"[ {g} ]\n")
        for k in range(0, len(idx), 15): f.write(" ".join(f"{i:4d}" for i in idx[k:k+15]) + "\n")
print({g: len(v) for g, v in G.items()})

In [ ]:
%%bash
cd "$OUT/frames"; X="$OUT/xvgs"; L="$OUT/logs"; T=fit_cluster_skip5.xtc
for c in Protein chA chB chC; do
  printf "${c}_bb\n${c}_bb\n" | gmx rms  -s first_cl.gro -f $T -n chains.ndx -o $X/rmsd_$c.xvg -tu ns > $L/rmsd_$c.log 2>&1
  printf "${c}_CA\n${c}_CA\n" | gmx rmsf -s first_cl.gro -f $T -n chains.ndx -o $X/rmsf_$c.xvg -res   > $L/rmsf_$c.log 2>&1
  printf "$c\n"               | gmx gyrate -s protein.tpr -f $T -n chains.ndx -o $X/rg_$c.xvg          > $L/rg_$c.log 2>&1
  gmx sasa -s protein.tpr -f $T -n chains.ndx -surface "group Protein" -output "group $c" -o $X/sasa_$c.xvg -or $X/tmp_$c.xvg -tu ns > $L/sasa_$c.log 2>&1
  gmx hbond -s protein.tpr -f $T -n chains.ndx -r "group $c" -t "group $c" -num $X/hbnum_$c.xvg -tu ns > $L/hbond_$c.log 2>&1
done
for p in "chA chB" "chB chC" "chA chC"; do set -- $p
  gmx hbond -s protein.tpr -f $T -n chains.ndx -r "group $1" -t "group $2" -num $X/hbinter_$1_$2.xvg -tu ns > $L/hbinter_$1_$2.log 2>&1
done
mv $X/tmp_Protein.xvg $X/sasa_perres_Protein.xvg; rm -f $X/tmp_ch*.xvg
for f in rmsd_Protein rmsd_chA rmsf_chA rg_Protein sasa_chA hbnum_Protein hbinter_chA_chB; do echo $f $(tail -1 $X/$f.xvg); done

In [ ]:
import matplotlib.pyplot as plt
X = f"{OUT}/xvgs"
def xvg(f): return np.loadtxt(f, comments=('#','@'))
sty = {"chA": dict(color="#d62728", ls="-", label="Chain A"),
       "chB": dict(color="#7fbfff", ls=":", label="Chain B"),
       "chC": dict(color="#1f77b4", ls="-", label="Chain C")}
def draw(out, title, xl, yl, items):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for x, y, s in items: ax.plot(x, y, lw=1.8, **s)
    ax.set(xlabel=xl, ylabel=yl, title=title); ax.legend(); fig.tight_layout()
    fig.savefig(f"{OUT}/graphs/{out}.png"); plt.show()
def chains(prefix, col=1, sc=1):
    return [(xvg(f"{X}/{prefix}_{c}.xvg")[:,0]/sc, xvg(f"{X}/{prefix}_{c}.xvg")[:,col], s) for c, s in sty.items()]
def whole(prefix, col=1, sc=1):
    a = xvg(f"{X}/{prefix}_Protein.xvg"); return [(a[:,0]/sc, a[:,col], dict(color="grey", label="Whole trimer"))]

for pre, yl, ttl, sc, col in (("rmsd","RMSD (nm)","Backbone RMSD",1,1), ("rg","Rg (nm)","Radius of gyration",1000,1),
                              ("sasa","SASA (nm²)","SASA",1,None), ("hbnum","H-bonds","Intra-chain H-bonds",1,1)):
    cc = 2 if pre == "sasa" else 1
    draw(f"{pre}_per_chain", f"{ttl} per chain, 300 K (r2)", "Time (ns)", yl, chains(pre, cc, sc))
    draw(f"{pre}_protein", f"{ttl}, whole trimer, 300 K (r2)", "Time (ns)", yl, whole(pre, 1, sc))

# residue-based: RMSF per chain; SASA per residue (split the 1245-row trimer file)
draw("rmsf_per_chain", "C-alpha RMSF per residue, 300 K (r2)", "Residue", "RMSF (nm)", chains("rmsf"))
d = xvg(f"{X}/sasa_perres_Protein.xvg"); assert len(d) == 1245
draw("sasa_per_residue", "SASA per residue (time-averaged), 300 K (r2)", "Residue", "SASA (nm²)",
     [(np.arange(1,416), d[i*415:(i+1)*415,1], s) for i, s in enumerate(sty.values())])

# inter-chain H-bonds vs time
pairs = [("chA","chB","A–B"),("chB","chC","B–C"),("chA","chC","A–C")]
draw("hbonds_inter_chain", "Inter-chain H-bonds, 300 K (r2)", "Time (ns)", "H-bonds",
     [(xvg(f"{X}/hbinter_{a}_{b}.xvg")[:,0], xvg(f"{X}/hbinter_{a}_{b}.xvg")[:,1], dict(color=c, ls=l, label=n))
      for (a,b,n), c, l in zip(pairs, ("#d62728","#7fbfff","#1f77b4"), ("-",":","-"))])

In [ ]:
from MDAnalysis.analysis.align import rotation_matrix
u = mda.Universe(f"{OUT}/frames/first_cl.gro", f"{OUT}/frames/fit_cluster_skip5.xtc")
cs = {"chA":slice(0,6647), "chB":slice(6647,13294), "chC":slice(13294,19941)}
resR = {}
for name, s in cs.items():
    chn = u.atoms[s]; bb = chn.select_atoms("name N CA C"); ca = chn.select_atoms("name CA")
    u.trajectory[0]; c0 = bb.positions.mean(0); bb0 = bb.positions - c0; ca0 = ca.positions - c0
    d2 = np.zeros(len(ca)); n = 0
    for ts in u.trajectory:
        c = bb.positions.mean(0); R, _ = rotation_matrix(bb.positions - c, bb0)
        d2 += (((ca.positions - c) @ R.T - ca0)**2).sum(1); n += 1
    resR[name] = np.sqrt(d2/n)/10
    np.savetxt(f"{X}/rmsd_perres_{name}.xvg", np.c_[np.arange(1,416), resR[name]], fmt="%d %.6f")
draw("rmsd_per_residue", "C-alpha RMSD per residue vs first frame, 300 K (r2)", "Residue", "RMSD (nm)",
     [(np.arange(1,416), resR[c], s) for c, s in sty.items()])
m = np.mean(list(resR.values()), axis=0)
draw("rmsd_per_residue_mean", "Mean C-alpha RMSD per residue (3 chains), 300 K (r2)", "Residue", "RMSD (nm)",
     [(np.arange(1,416), m, dict(color="grey", label="Mean of chains"))])

# per-residue H-bonds (box-corrected trajectory, D-A <= 0.35 nm, angle <= 30 deg)
nmA = np.array([n.strip() for n in u.atoms.names]); rn = u.atoms.resnames
resid = u.atoms.resids; chain = np.arange(len(nmA))//6647
isH = np.array([n.startswith("H") for n in nmA])
P0 = u.atoms.positions/10; heavy = np.where(~isH)[0]; Hs = np.where(isH)[0]
don = heavy[cKDTree(P0[heavy]).query(P0[Hs])[1]]
k = np.array([nmA[i][0] in "NO" for i in don]); Hs, don = Hs[k], don[k]
acc = np.array([i for i in range(len(nmA)) if nmA[i][0]=="O" or (nmA[i] in ("ND1","NE2") and rn[i].startswith("HI"))])
tot = np.zeros((3,416)); inter = np.zeros((3,416)); nb = []
for ts in u.trajectory:
    Pf = u.atoms.positions/10; n = 0
    for hi, lst in enumerate(cKDTree(Pf[Hs]).query_ball_tree(cKDTree(Pf[acc]), 0.25)):
        D = don[hi]
        for aj in lst:
            A = acc[aj]
            if A == D or (chain[A]==chain[D] and resid[A]==resid[D]): continue
            v1, v2 = Pf[Hs[hi]]-Pf[D], Pf[A]-Pf[D]; dist = np.linalg.norm(v2)
            if dist > 0.35 or v1@v2/(np.linalg.norm(v1)*dist) < np.cos(np.radians(30)): continue
            n += 1
            for Xa in (D, A):
                tot[chain[Xa], resid[Xa]] += 1
                if chain[D] != chain[A]: inter[chain[Xa], resid[Xa]] += 1
    nb.append(n)
nf = len(nb); print("mean H-bonds/frame:", round(float(np.mean(nb)),1))
for nmx, arr, ttl in (("hbperres",tot,"All H-bonds per residue"), ("hbperres_inter",inter,"Inter-chain H-bonds per residue")):
    for i, c in enumerate(sty): np.savetxt(f"{X}/{nmx}_{c}.xvg", np.c_[np.arange(1,416), arr[i,1:]/nf], fmt="%d %.5f")
    draw(nmx, f"{ttl}, 300 K (r2)", "Residue", "H-bonds per frame",
         [(np.arange(1,416), arr[i,1:]/nf, s) for i, s in enumerate(sty.values())])

In [ ]:
%%bash
cd "$OUT/frames"
for c in chA chB chC; do
  printf "$c\n" | gmx dssp -s protein.tpr -f fit_cluster_skip5.xtc -n chains.ndx -o "$OUT/xvgs/dssp_$c.dat" > "$OUT/logs/dssp_$c.log" 2>&1
  echo $c; head -c 80 "$OUT/xvgs/dssp_$c.dat"; echo; tail -3 "$OUT/logs/dssp_$c.log"
done

In [ ]:
import numpy as np, matplotlib.pyplot as plt, os
OUT = os.environ["OUT"]
a = np.array([list(l.rstrip("\n")) for l in open(f"{OUT}/xvgs/dssp_chA.dat") if l.strip()])
brk = np.where((a == "=").all(0))[0]
print("shape:", a.shape, "| chain breaks at columns:", brk)
a = a[:, ~(a == "=").all(0)]
assert a.shape[1] == 1245, a.shape
D = {c: a[:, i*415:(i+1)*415] for i, c in enumerate(("chA","chB","chC"))}

sty = {"chA": dict(color="#d62728", ls="-", label="Chain A"),
       "chB": dict(color="#7fbfff", ls=":", label="Chain B"),
       "chC": dict(color="#1f77b4", ls="-", label="Chain C")}
t = np.linspace(0, 1, len(a))
for name, ss, yl in (("helix","HGI","Helix residues (%)"), ("strand","EB","Strand residues (%)")):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        ax.plot(t, 100*np.isin(D[c], list(ss)).mean(1), lw=1.8, **s)
        print(name, c, "mean %:", round(100*np.isin(D[c], list(ss)).mean(), 1))
    ax.set(xlabel="Time (ns)", ylabel=yl, title=f"{name.capitalize()} content vs time, 300 K (r2)")
    ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_{name}_time.png"); plt.show()

    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        f = np.isin(D[c], list(ss)).mean(0); ax.plot(np.arange(1,416), f, lw=1.6, **s)
        np.savetxt(f"{OUT}/xvgs/dssp_{name}_perres_{c}.xvg", np.c_[np.arange(1,416), f], fmt="%d %.4f")
    ax.set(xlabel="Residue", ylabel=f"Fraction of frames in {name}", title=f"{name.capitalize()} per residue, 300 K (r2)")
    ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_{name}_perres.png"); plt.show()

In [ ]:
from matplotlib.colors import ListedColormap
fig, ax = plt.subplots(figsize=(12,5), dpi=150)
for c, s in sty.items():
    ax.plot(t, 100*(1-np.isin(D[c], list("HGIEB")).mean(1)), lw=1.8, **s)
ax.set(xlabel="Time (ns)", ylabel="Coil/turn residues (%)", title="Coil/turn content vs time, 300 K (r2)")
ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_coil_time.png"); plt.show()

cmap = ListedColormap(["#eeeeee", "#d62728", "#1f77b4"])   # coil, helix, strand
code = lambda m: np.where(np.isin(m, list("HGI")), 1, np.where(np.isin(m, list("EB")), 2, 0))
fig, axs = plt.subplots(3, 1, figsize=(12,8), dpi=150, sharex=True)
for ax, (c, s) in zip(axs, sty.items()):
    ax.imshow(code(D[c]), aspect="auto", cmap=cmap, vmin=0, vmax=2, origin="lower", extent=[1,415,0,1])
    ax.set_ylabel("Time (ns)"); ax.set_title(s["label"], loc="left", fontsize=10)
axs[-1].set_xlabel("Residue"); fig.suptitle("DSSP map: red = helix, blue = strand, grey = coil/turn, 300 K (r2)")
fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_map.png"); plt.show()

In [ ]:
import pandas as pd
def last(f, col=1): a = np.loadtxt(f, comments=('#','@')); return a[len(a)//2:, col].mean()
rows = []
for c in ("chA","chB","chC"):
    rows.append(dict(chain=c,
      RMSD_nm = last(f"{OUT}/xvgs/rmsd_{c}.xvg"), Rg_nm = last(f"{OUT}/xvgs/rg_{c}.xvg"),
      SASA_nm2 = last(f"{OUT}/xvgs/sasa_{c}.xvg", 2), intra_Hbonds = last(f"{OUT}/xvgs/hbnum_{c}.xvg"),
      helix_pct = 100*np.isin(D[c][len(t)//2:], list("HGI")).mean(), strand_pct = 100*np.isin(D[c][len(t)//2:], list("EB")).mean()))
rows.append(dict(chain="trimer", RMSD_nm=last(f"{OUT}/xvgs/rmsd_Protein.xvg"), Rg_nm=last(f"{OUT}/xvgs/rg_Protein.xvg"),
                 SASA_nm2=last(f"{OUT}/xvgs/sasa_Protein.xvg"), intra_Hbonds=last(f"{OUT}/xvgs/hbnum_Protein.xvg")))
df = pd.DataFrame(rows).round(3); df.to_csv(f"{OUT}/xvgs/summary_last0.5ns.csv", index=False); print(df)

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
OUT = os.environ["OUT"]

def load_dssp(path):
    a = np.array([list(l.rstrip("\n")) for l in open(path) if l.strip()])
    return a[:, ~(a == "=").all(0)]            # frames x residues, chain breaks removed

STATES = [("Alpha helix","H"),("3-10 helix","G"),("Pi helix","I"),("Beta strand","E"),("Beta bridge","B"),
          ("Turn","T"),("Bend","S"),("Polyproline II helix","P")]
def percent(a):
    P = {n: 100*(a == c).mean(1) for n, c in STATES}
    P["Coil / Unstructured"] = 100 - sum(P.values())      # everything else
    return P

a = load_dssp(f"{OUT}/xvgs/dssp_chA.dat"); print(a.shape)   # expect (201, 1245)
P = percent(a); t = np.arange(len(a))*5.0                   # 5 ps between kept frames

fig, ax = plt.subplots(figsize=(12,6.2), dpi=150)
for i, (n, y) in enumerate(P.items()): ax.plot(t, y, color=plt.cm.tab10(i), lw=1.6, label=n)
ax.set(xlabel="Time (ps)", ylabel="Percentage (%)", title="Vicilin All Secondary Structure States — 300 K (r2)")
ax.grid(alpha=.3); ax.legend(title="Secondary structure", ncol=2, loc="upper right"); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/dssp_all_states_300K.png"); plt.show()

G = {"Alpha helix": P["Alpha helix"], "Beta structure": P["Beta strand"]+P["Beta bridge"], "Turns": P["Turn"]}
G["Loops / Other"] = 100 - sum(G.values())
fig, ax = plt.subplots(figsize=(10,6), dpi=150)
for i, (n, y) in enumerate(G.items()): ax.plot(t, y, color=plt.cm.tab10(i), lw=1.8, label=n)
ax.set(xlabel="Time (ps)", ylabel="Percentage (%)", title="Vicilin Secondary Structure — 300 K (r2)")
ax.grid(alpha=.3); ax.legend(); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/dssp_groups_300K.png"); plt.show()

In [ ]:
import os
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
os.system('ln -sfn "/content/drive/MyDrive/Plant Protein Extrusion MD" /content/pm')
os.environ.update(P="/content/pm", S="md_373_r1", OUT="/content/pm/analysism/vicilin_373_r1")
for d in ["graphs","logs","frames","xvgs"]: os.makedirs(f'{os.environ["OUT"]}/{d}', exist_ok=True)
os.system('pip install -q MDAnalysis scipy; gmx --version | head -1; ls $P/$S.*')

In [ ]:
%%bash
ls /content/pm | head -50
echo ---
find /content/drive/MyDrive -name "md_373_r1*" 2>/dev/null

In [ ]:
%%bash
ls -lh /content/pm/md_373_r1.*
echo "S=$S  P=$P  OUT=$OUT"

In [ ]:
%%bash
ls -lh "$OUT/frames"

In [ ]:
import matplotlib.pyplot as plt
X = f"{OUT}/xvgs"
def xvg(f): return np.loadtxt(f, comments=('#','@'))
sty = {"chA": dict(color="#d62728", ls="-", label="Chain A"),
       "chB": dict(color="#7fbfff", ls=":", label="Chain B"),
       "chC": dict(color="#1f77b4", ls="-", label="Chain C")}
def draw(out, title, xl, yl, items):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for x, y, s in items: ax.plot(x, y, lw=1.8, **s)
    ax.set(xlabel=xl, ylabel=yl, title=title); ax.legend(); fig.tight_layout()
    fig.savefig(f"{OUT}/graphs/{out}.png"); plt.show()
def chains(prefix, col=1, sc=1):
    return [(xvg(f"{X}/{prefix}_{c}.xvg")[:,0]/sc, xvg(f"{X}/{prefix}_{c}.xvg")[:,col], s) for c, s in sty.items()]
def whole(prefix, col=1, sc=1):
    a = xvg(f"{X}/{prefix}_Protein.xvg"); return [(a[:,0]/sc, a[:,col], dict(color="grey", label="Whole trimer"))]

for pre, yl, ttl, sc, col in (("rmsd","RMSD (nm)","Backbone RMSD",1,1), ("rg","Rg (nm)","Radius of gyration",1000,1),
                              ("sasa","SASA (nm²)","SASA",1,None), ("hbnum","H-bonds","Intra-chain H-bonds",1,1)):
    cc = 2 if pre == "sasa" else 1
    draw(f"{pre}_per_chain", f"{ttl} per chain, 373 K (r1)", "Time (ns)", yl, chains(pre, cc, sc))
    draw(f"{pre}_protein", f"{ttl}, whole trimer, 373 K (r1)", "Time (ns)", yl, whole(pre, 1, sc))

# residue-based: RMSF per chain; SASA per residue (split the 1245-row trimer file)
draw("rmsf_per_chain", "C-alpha RMSF per residue, 373 K (r1)", "Residue", "RMSF (nm)", chains("rmsf"))
d = xvg(f"{X}/sasa_perres_Protein.xvg"); assert len(d) == 1245
draw("sasa_per_residue", "SASA per residue (time-averaged), 373 K (r1)", "Residue", "SASA (nm²)",
     [(np.arange(1,416), d[i*415:(i+1)*415,1], s) for i, s in enumerate(sty.values())])

# inter-chain H-bonds vs time
pairs = [("chA","chB","A–B"),("chB","chC","B–C"),("chA","chC","A–C")]
draw("hbonds_inter_chain", "Inter-chain H-bonds, 373 K (r1)", "Time (ns)", "H-bonds",
     [(xvg(f"{X}/hbinter_{a}_{b}.xvg")[:,0], xvg(f"{X}/hbinter_{a}_{b}.xvg")[:,1], dict(color=c, ls=l, label=n))
      for (a,b,n), c, l in zip(pairs, ("#d62728","#7fbfff","#1f77b4"), ("-",":","-"))])

In [ ]:
from MDAnalysis.analysis.align import rotation_matrix
u = mda.Universe(f"{OUT}/frames/first_cl.gro", f"{OUT}/frames/fit_cluster_skip5.xtc")
cs = {"chA":slice(0,6647), "chB":slice(6647,13294), "chC":slice(13294,19941)}
resR = {}
for name, s in cs.items():
    chn = u.atoms[s]; bb = chn.select_atoms("name N CA C"); ca = chn.select_atoms("name CA")
    u.trajectory[0]; c0 = bb.positions.mean(0); bb0 = bb.positions - c0; ca0 = ca.positions - c0
    d2 = np.zeros(len(ca)); n = 0
    for ts in u.trajectory:
        c = bb.positions.mean(0); R, _ = rotation_matrix(bb.positions - c, bb0)
        d2 += (((ca.positions - c) @ R.T - ca0)**2).sum(1); n += 1
    resR[name] = np.sqrt(d2/n)/10
    np.savetxt(f"{X}/rmsd_perres_{name}.xvg", np.c_[np.arange(1,416), resR[name]], fmt="%d %.6f")
draw("rmsd_per_residue", "C-alpha RMSD per residue vs first frame, 373 K (r1)", "Residue", "RMSD (nm)",
     [(np.arange(1,416), resR[c], s) for c, s in sty.items()])
m = np.mean(list(resR.values()), axis=0)
draw("rmsd_per_residue_mean", "Mean C-alpha RMSD per residue (3 chains), 373 K (r1)", "Residue", "RMSD (nm)",
     [(np.arange(1,416), m, dict(color="grey", label="Mean of chains"))])

# per-residue H-bonds (box-corrected trajectory, D-A <= 0.35 nm, angle <= 30 deg)
nmA = np.array([n.strip() for n in u.atoms.names]); rn = u.atoms.resnames
resid = u.atoms.resids; chain = np.arange(len(nmA))//6647
isH = np.array([n.startswith("H") for n in nmA])
P0 = u.atoms.positions/10; heavy = np.where(~isH)[0]; Hs = np.where(isH)[0]
don = heavy[cKDTree(P0[heavy]).query(P0[Hs])[1]]
k = np.array([nmA[i][0] in "NO" for i in don]); Hs, don = Hs[k], don[k]
acc = np.array([i for i in range(len(nmA)) if nmA[i][0]=="O" or (nmA[i] in ("ND1","NE2") and rn[i].startswith("HI"))])
tot = np.zeros((3,416)); inter = np.zeros((3,416)); nb = []
for ts in u.trajectory:
    Pf = u.atoms.positions/10; n = 0
    for hi, lst in enumerate(cKDTree(Pf[Hs]).query_ball_tree(cKDTree(Pf[acc]), 0.25)):
        D = don[hi]
        for aj in lst:
            A = acc[aj]
            if A == D or (chain[A]==chain[D] and resid[A]==resid[D]): continue
            v1, v2 = Pf[Hs[hi]]-Pf[D], Pf[A]-Pf[D]; dist = np.linalg.norm(v2)
            if dist > 0.35 or v1@v2/(np.linalg.norm(v1)*dist) < np.cos(np.radians(30)): continue
            n += 1
            for Xa in (D, A):
                tot[chain[Xa], resid[Xa]] += 1
                if chain[D] != chain[A]: inter[chain[Xa], resid[Xa]] += 1
    nb.append(n)
nf = len(nb); print("mean H-bonds/frame:", round(float(np.mean(nb)),1))
for nmx, arr, ttl in (("hbperres",tot,"All H-bonds per residue"), ("hbperres_inter",inter,"Inter-chain H-bonds per residue")):
    for i, c in enumerate(sty): np.savetxt(f"{X}/{nmx}_{c}.xvg", np.c_[np.arange(1,416), arr[i,1:]/nf], fmt="%d %.5f")
    draw(nmx, f"{ttl}, 373 K (r1)", "Residue", "H-bonds per frame",
         [(np.arange(1,416), arr[i,1:]/nf, s) for i, s in enumerate(sty.values())])

In [ ]:
import numpy as np, matplotlib.pyplot as plt, os
OUT = os.environ["OUT"]
a = np.array([list(l.rstrip("\n")) for l in open(f"{OUT}/xvgs/dssp_chA.dat") if l.strip()])
brk = np.where((a == "=").all(0))[0]
print("shape:", a.shape, "| chain breaks at columns:", brk)
a = a[:, ~(a == "=").all(0)]
assert a.shape[1] == 1245, a.shape
D = {c: a[:, i*415:(i+1)*415] for i, c in enumerate(("chA","chB","chC"))}

sty = {"chA": dict(color="#d62728", ls="-", label="Chain A"),
       "chB": dict(color="#7fbfff", ls=":", label="Chain B"),
       "chC": dict(color="#1f77b4", ls="-", label="Chain C")}
t = np.linspace(0, 1, len(a))
for name, ss, yl in (("helix","HGI","Helix residues (%)"), ("strand","EB","Strand residues (%)")):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        ax.plot(t, 100*np.isin(D[c], list(ss)).mean(1), lw=1.8, **s)
        print(name, c, "mean %:", round(100*np.isin(D[c], list(ss)).mean(), 1))
    ax.set(xlabel="Time (ns)", ylabel=yl, title=f"{name.capitalize()} content vs time, 373 K (r1)")
    ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_{name}_time.png"); plt.show()

    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        f = np.isin(D[c], list(ss)).mean(0); ax.plot(np.arange(1,416), f, lw=1.6, **s)
        np.savetxt(f"{OUT}/xvgs/dssp_{name}_perres_{c}.xvg", np.c_[np.arange(1,416), f], fmt="%d %.4f")
    ax.set(xlabel="Residue", ylabel=f"Fraction of frames in {name}", title=f"{name.capitalize()} per residue, 373 K (r1)")
    ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_{name}_perres.png"); plt.show()

In [ ]:
from matplotlib.colors import ListedColormap
fig, ax = plt.subplots(figsize=(12,5), dpi=150)
for c, s in sty.items():
    ax.plot(t, 100*(1-np.isin(D[c], list("HGIEB")).mean(1)), lw=1.8, **s)
ax.set(xlabel="Time (ns)", ylabel="Coil/turn residues (%)", title="Coil/turn content vs time, 373 K (r1)")
ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_coil_time.png"); plt.show()

cmap = ListedColormap(["#eeeeee", "#d62728", "#1f77b4"])   # coil, helix, strand
code = lambda m: np.where(np.isin(m, list("HGI")), 1, np.where(np.isin(m, list("EB")), 2, 0))
fig, axs = plt.subplots(3, 1, figsize=(12,8), dpi=150, sharex=True)
for ax, (c, s) in zip(axs, sty.items()):
    ax.imshow(code(D[c]), aspect="auto", cmap=cmap, vmin=0, vmax=2, origin="lower", extent=[1,415,0,1])
    ax.set_ylabel("Time (ns)"); ax.set_title(s["label"], loc="left", fontsize=10)
axs[-1].set_xlabel("Residue"); fig.suptitle("DSSP map: red = helix, blue = strand, grey = coil/turn, 373 K (r1)")
fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_map.png"); plt.show()

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
OUT = os.environ["OUT"]

def load_dssp(path):
    a = np.array([list(l.rstrip("\n")) for l in open(path) if l.strip()])
    return a[:, ~(a == "=").all(0)]            # frames x residues, chain breaks removed

STATES = [("Alpha helix","H"),("3-10 helix","G"),("Pi helix","I"),("Beta strand","E"),("Beta bridge","B"),
          ("Turn","T"),("Bend","S"),("Polyproline II helix","P")]
def percent(a):
    P = {n: 100*(a == c).mean(1) for n, c in STATES}
    P["Coil / Unstructured"] = 100 - sum(P.values())      # everything else
    return P

a = load_dssp(f"{OUT}/xvgs/dssp_chA.dat"); print(a.shape)   # expect (201, 1245)
P = percent(a); t = np.arange(len(a))*5.0                   # 5 ps between kept frames

fig, ax = plt.subplots(figsize=(12,6.2), dpi=150)
for i, (n, y) in enumerate(P.items()): ax.plot(t, y, color=plt.cm.tab10(i), lw=1.6, label=n)
ax.set(xlabel="Time (ps)", ylabel="Percentage (%)", title="Vicilin All Secondary Structure States — 373 K (r1)")
ax.grid(alpha=.3); ax.legend(title="Secondary structure", ncol=2, loc="upper right"); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/dssp_all_states_373K.png"); plt.show()

G = {"Alpha helix": P["Alpha helix"], "Beta structure": P["Beta strand"]+P["Beta bridge"], "Turns": P["Turn"]}
G["Loops / Other"] = 100 - sum(G.values())
fig, ax = plt.subplots(figsize=(10,6), dpi=150)
for i, (n, y) in enumerate(G.items()): ax.plot(t, y, color=plt.cm.tab10(i), lw=1.8, label=n)
ax.set(xlabel="Time (ps)", ylabel="Percentage (%)", title="Vicilin Secondary Structure — 373 K (r1)")
ax.grid(alpha=.3); ax.legend(); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/dssp_groups_373K.png"); plt.show()

In [ ]:
import os
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
os.system('ln -sfn "/content/drive/MyDrive/Plant Protein Extrusion MD" /content/pm')
os.environ.update(P="/content/pm", S="md_373_shear_r1", OUT="/content/pm/analysism/vicilin_373_shear_r1")
for d in ["graphs","logs","frames","xvgs"]: os.makedirs(f'{os.environ["OUT"]}/{d}', exist_ok=True)
os.system('pip install -q MDAnalysis scipy; gmx --version | head -1; ls $P/$S.*')

In [ ]:
%%bash
ls -lh /content/pm/md_373_shear_r1.*
echo "S=$S  OUT=$OUT"

In [ ]:
%%bash
gmx check -f "$P/$S.xtc" 2>&1 | grep -iE "Natoms|atoms|Last frame|Step"

In [ ]:
%%bash
find /content/drive/MyDrive -type d -name "Shear Run" 2>/dev/null
find /content/drive/MyDrive -name "md_373_shear_r1.xtc" -exec ls -lh {} \; 2>/dev/null
find /content/drive/MyDrive -name "md_373_shear_r1.tpr" -exec ls -lh {} \; 2>/dev/null

In [ ]:
import os, glob
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
os.system('ln -sfn "/content/drive/MyDrive/Plant Protein Extrusion MD" /content/pm')
sr = [d for d in glob.glob("/content/pm/Shear Run*") if os.path.isdir(d)][0]
os.system(f'ln -sfn "{sr}" /content/shear')
os.environ.update(P="/content/shear", S="md_373_shear_r1", OUT="/content/pm/analysism/vicilin_373_shear_r1")
for d in ["graphs","logs","frames","xvgs"]: os.makedirs(f'{os.environ["OUT"]}/{d}', exist_ok=True)
os.system('pip install -q MDAnalysis scipy; gmx --version | head -1; ls $P/$S.*')

In [ ]:
%%bash
ls -lh $P/$S.*
echo "S=$S  P=$P  OUT=$OUT"
gmx check -f "$P/$S.xtc" 2>&1 | grep -iE "Natoms|Last frame|Step|Time"

In [ ]:
%%bash
gmx check -f "$P/$S.xtc" 2>&1 | grep -iE "atoms"

In [ ]:
import matplotlib.pyplot as plt
X = f"{OUT}/xvgs"
def xvg(f): return np.loadtxt(f, comments=('#','@'))
sty = {"chA": dict(color="#d62728", ls="-", label="Chain A"),
       "chB": dict(color="#7fbfff", ls=":", label="Chain B"),
       "chC": dict(color="#1f77b4", ls="-", label="Chain C")}
def draw(out, title, xl, yl, items):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for x, y, s in items: ax.plot(x, y, lw=1.8, **s)
    ax.set(xlabel=xl, ylabel=yl, title=title); ax.legend(); fig.tight_layout()
    fig.savefig(f"{OUT}/graphs/{out}.png"); plt.show()
def chains(prefix, col=1, sc=1):
    return [(xvg(f"{X}/{prefix}_{c}.xvg")[:,0]/sc, xvg(f"{X}/{prefix}_{c}.xvg")[:,col], s) for c, s in sty.items()]
def whole(prefix, col=1, sc=1):
    a = xvg(f"{X}/{prefix}_Protein.xvg"); return [(a[:,0]/sc, a[:,col], dict(color="grey", label="Whole trimer"))]

for pre, yl, ttl, sc, col in (("rmsd","RMSD (nm)","Backbone RMSD",1,1), ("rg","Rg (nm)","Radius of gyration",1000,1),
                              ("sasa","SASA (nm²)","SASA",1,None), ("hbnum","H-bonds","Intra-chain H-bonds",1,1)):
    cc = 2 if pre == "sasa" else 1
    draw(f"{pre}_per_chain", f"{ttl} per chain, 373 K shear (r1)", "Time (ns)", yl, chains(pre, cc, sc))
    draw(f"{pre}_protein", f"{ttl}, whole trimer, 373 K shear (r1)", "Time (ns)", yl, whole(pre, 1, sc))

# residue-based: RMSF per chain; SASA per residue (split the 1245-row trimer file)
draw("rmsf_per_chain", "C-alpha RMSF per residue, 373 K shear (r1)", "Residue", "RMSF (nm)", chains("rmsf"))
d = xvg(f"{X}/sasa_perres_Protein.xvg"); assert len(d) == 1245
draw("sasa_per_residue", "SASA per residue (time-averaged), 373 K shear (r1)", "Residue", "SASA (nm²)",
     [(np.arange(1,416), d[i*415:(i+1)*415,1], s) for i, s in enumerate(sty.values())])

# inter-chain H-bonds vs time
pairs = [("chA","chB","A–B"),("chB","chC","B–C"),("chA","chC","A–C")]
draw("hbonds_inter_chain", "Inter-chain H-bonds, 373 K shear (r1)", "Time (ns)", "H-bonds",
     [(xvg(f"{X}/hbinter_{a}_{b}.xvg")[:,0], xvg(f"{X}/hbinter_{a}_{b}.xvg")[:,1], dict(color=c, ls=l, label=n))
      for (a,b,n), c, l in zip(pairs, ("#d62728","#7fbfff","#1f77b4"), ("-",":","-"))])

In [ ]:
from MDAnalysis.analysis.align import rotation_matrix
u = mda.Universe(f"{OUT}/frames/first_cl.gro", f"{OUT}/frames/fit_cluster_skip5.xtc")
cs = {"chA":slice(0,6647), "chB":slice(6647,13294), "chC":slice(13294,19941)}
resR = {}
for name, s in cs.items():
    chn = u.atoms[s]; bb = chn.select_atoms("name N CA C"); ca = chn.select_atoms("name CA")
    u.trajectory[0]; c0 = bb.positions.mean(0); bb0 = bb.positions - c0; ca0 = ca.positions - c0
    d2 = np.zeros(len(ca)); n = 0
    for ts in u.trajectory:
        c = bb.positions.mean(0); R, _ = rotation_matrix(bb.positions - c, bb0)
        d2 += (((ca.positions - c) @ R.T - ca0)**2).sum(1); n += 1
    resR[name] = np.sqrt(d2/n)/10
    np.savetxt(f"{X}/rmsd_perres_{name}.xvg", np.c_[np.arange(1,416), resR[name]], fmt="%d %.6f")
draw("rmsd_per_residue", "C-alpha RMSD per residue vs first frame, 373 K shear (r1)", "Residue", "RMSD (nm)",
     [(np.arange(1,416), resR[c], s) for c, s in sty.items()])
m = np.mean(list(resR.values()), axis=0)
draw("rmsd_per_residue_mean", "Mean C-alpha RMSD per residue (3 chains), 373 K shear (r1)", "Residue", "RMSD (nm)",
     [(np.arange(1,416), m, dict(color="grey", label="Mean of chains"))])

# per-residue H-bonds (box-corrected trajectory, D-A <= 0.35 nm, angle <= 30 deg)
nmA = np.array([n.strip() for n in u.atoms.names]); rn = u.atoms.resnames
resid = u.atoms.resids; chain = np.arange(len(nmA))//6647
isH = np.array([n.startswith("H") for n in nmA])
P0 = u.atoms.positions/10; heavy = np.where(~isH)[0]; Hs = np.where(isH)[0]
don = heavy[cKDTree(P0[heavy]).query(P0[Hs])[1]]
k = np.array([nmA[i][0] in "NO" for i in don]); Hs, don = Hs[k], don[k]
acc = np.array([i for i in range(len(nmA)) if nmA[i][0]=="O" or (nmA[i] in ("ND1","NE2") and rn[i].startswith("HI"))])
tot = np.zeros((3,416)); inter = np.zeros((3,416)); nb = []
for ts in u.trajectory:
    Pf = u.atoms.positions/10; n = 0
    for hi, lst in enumerate(cKDTree(Pf[Hs]).query_ball_tree(cKDTree(Pf[acc]), 0.25)):
        D = don[hi]
        for aj in lst:
            A = acc[aj]
            if A == D or (chain[A]==chain[D] and resid[A]==resid[D]): continue
            v1, v2 = Pf[Hs[hi]]-Pf[D], Pf[A]-Pf[D]; dist = np.linalg.norm(v2)
            if dist > 0.35 or v1@v2/(np.linalg.norm(v1)*dist) < np.cos(np.radians(30)): continue
            n += 1
            for Xa in (D, A):
                tot[chain[Xa], resid[Xa]] += 1
                if chain[D] != chain[A]: inter[chain[Xa], resid[Xa]] += 1
    nb.append(n)
nf = len(nb); print("mean H-bonds/frame:", round(float(np.mean(nb)),1))
for nmx, arr, ttl in (("hbperres",tot,"All H-bonds per residue"), ("hbperres_inter",inter,"Inter-chain H-bonds per residue")):
    for i, c in enumerate(sty): np.savetxt(f"{X}/{nmx}_{c}.xvg", np.c_[np.arange(1,416), arr[i,1:]/nf], fmt="%d %.5f")
    draw(nmx, f"{ttl}, 373 K shear (r1)", "Residue", "H-bonds per frame",
         [(np.arange(1,416), arr[i,1:]/nf, s) for i, s in enumerate(sty.values())])

In [ ]:
import numpy as np, matplotlib.pyplot as plt, os
OUT = os.environ["OUT"]
a = np.array([list(l.rstrip("\n")) for l in open(f"{OUT}/xvgs/dssp_chA.dat") if l.strip()])
brk = np.where((a == "=").all(0))[0]
print("shape:", a.shape, "| chain breaks at columns:", brk)
a = a[:, ~(a == "=").all(0)]
assert a.shape[1] == 1245, a.shape
D = {c: a[:, i*415:(i+1)*415] for i, c in enumerate(("chA","chB","chC"))}

sty = {"chA": dict(color="#d62728", ls="-", label="Chain A"),
       "chB": dict(color="#7fbfff", ls=":", label="Chain B"),
       "chC": dict(color="#1f77b4", ls="-", label="Chain C")}
t = np.linspace(0, 1, len(a))
for name, ss, yl in (("helix","HGI","Helix residues (%)"), ("strand","EB","Strand residues (%)")):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        ax.plot(t, 100*np.isin(D[c], list(ss)).mean(1), lw=1.8, **s)
        print(name, c, "mean %:", round(100*np.isin(D[c], list(ss)).mean(), 1))
    ax.set(xlabel="Time (ns)", ylabel=yl, title=f"{name.capitalize()} content vs time, 373 K shear (r1)")
    ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_{name}_time.png"); plt.show()

    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        f = np.isin(D[c], list(ss)).mean(0); ax.plot(np.arange(1,416), f, lw=1.6, **s)
        np.savetxt(f"{OUT}/xvgs/dssp_{name}_perres_{c}.xvg", np.c_[np.arange(1,416), f], fmt="%d %.4f")
    ax.set(xlabel="Residue", ylabel=f"Fraction of frames in {name}", title=f"{name.capitalize()} per residue, 373 K shear (r1)")
    ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_{name}_perres.png"); plt.show()

In [ ]:
from matplotlib.colors import ListedColormap
fig, ax = plt.subplots(figsize=(12,5), dpi=150)
for c, s in sty.items():
    ax.plot(t, 100*(1-np.isin(D[c], list("HGIEB")).mean(1)), lw=1.8, **s)
ax.set(xlabel="Time (ns)", ylabel="Coil/turn residues (%)", title="Coil/turn content vs time, 373 K shear (r1)")
ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_coil_time.png"); plt.show()

cmap = ListedColormap(["#eeeeee", "#d62728", "#1f77b4"])   # coil, helix, strand
code = lambda m: np.where(np.isin(m, list("HGI")), 1, np.where(np.isin(m, list("EB")), 2, 0))
fig, axs = plt.subplots(3, 1, figsize=(12,8), dpi=150, sharex=True)
for ax, (c, s) in zip(axs, sty.items()):
    ax.imshow(code(D[c]), aspect="auto", cmap=cmap, vmin=0, vmax=2, origin="lower", extent=[1,415,0,1])
    ax.set_ylabel("Time (ns)"); ax.set_title(s["label"], loc="left", fontsize=10)
axs[-1].set_xlabel("Residue"); fig.suptitle("DSSP map: red = helix, blue = strand, grey = coil/turn, 373 K shear (r1)")
fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_map.png"); plt.show()

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
OUT = os.environ["OUT"]

def load_dssp(path):
    a = np.array([list(l.rstrip("\n")) for l in open(path) if l.strip()])
    return a[:, ~(a == "=").all(0)]

STATES = [("Alpha helix","H"),("3-10 helix","G"),("Pi helix","I"),("Beta strand","E"),("Beta bridge","B"),
          ("Turn","T"),("Bend","S"),("Polyproline II helix","P")]
def percent(a):
    P = {n: 100*(a == c).mean(1) for n, c in STATES}
    P["Coil / Unstructured"] = 100 - sum(P.values())
    return P

a = load_dssp(f"{OUT}/xvgs/dssp_chA.dat"); print(a.shape)   # expect (201, 1245)
P = percent(a); t = np.arange(len(a))*5.0                   # 5 ps between kept frames

fig, ax = plt.subplots(figsize=(12,6.2), dpi=150)
for i, (n, y) in enumerate(P.items()): ax.plot(t, y, color=plt.cm.tab10(i), lw=1.6, label=n)
ax.set(xlabel="Time (ps)", ylabel="Percentage (%)", title="Vicilin All Secondary Structure States — 373 K shear (r1)")
ax.grid(alpha=.3); ax.legend(title="Secondary structure", ncol=2, loc="upper right"); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/dssp_all_states_373K_shear.png"); plt.show()

G = {"Alpha helix": P["Alpha helix"], "Beta structure": P["Beta strand"]+P["Beta bridge"], "Turns": P["Turn"]}
G["Loops / Other"] = 100 - sum(G.values())
fig, ax = plt.subplots(figsize=(10,6), dpi=150)
for i, (n, y) in enumerate(G.items()): ax.plot(t, y, color=plt.cm.tab10(i), lw=1.8, label=n)
ax.set(xlabel="Time (ps)", ylabel="Percentage (%)", title="Vicilin Secondary Structure — 373 K shear (r1)")
ax.grid(alpha=.3); ax.legend(); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/dssp_groups_373K_shear.png"); plt.show()

In [ ]:
import os
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
os.system('ln -sfn "/content/drive/MyDrive/Plant Protein Extrusion MD" /content/pm')
os.environ.update(P="/content/pm", S="md_300_r1", OUT="/content/pm/analysism/vicilin_300_r1")
for d in ["graphs","logs","frames","xvgs"]: os.makedirs(f'{os.environ["OUT"]}/{d}', exist_ok=True)
os.system('pip install -q MDAnalysis scipy; gmx --version | head -1; ls $P/$S.*')

In [ ]:
%%bash
ls -lh $P/$S.*
echo "S=$S  OUT=$OUT"
gmx check -f "$P/$S.xtc" 2>&1 | grep -iE "Natoms|Last frame|Step|Time"
find /content/drive/MyDrive -name "md_300_r1.xtc" -exec ls -lh {} \; 2>/dev/null

In [ ]:
from MDAnalysis.analysis.align import rotation_matrix
u = mda.Universe(f"{OUT}/frames/first_cl.gro", f"{OUT}/frames/fit_cluster_skip5.xtc")
cs = {"chA":slice(0,6647), "chB":slice(6647,13294), "chC":slice(13294,19941)}
resR = {}
for name, s in cs.items():
    chn = u.atoms[s]; bb = chn.select_atoms("name N CA C"); ca = chn.select_atoms("name CA")
    u.trajectory[0]; c0 = bb.positions.mean(0); bb0 = bb.positions - c0; ca0 = ca.positions - c0
    d2 = np.zeros(len(ca)); n = 0
    for ts in u.trajectory:
        c = bb.positions.mean(0); R, _ = rotation_matrix(bb.positions - c, bb0)
        d2 += (((ca.positions - c) @ R.T - ca0)**2).sum(1); n += 1
    resR[name] = np.sqrt(d2/n)/10
    np.savetxt(f"{X}/rmsd_perres_{name}.xvg", np.c_[np.arange(1,416), resR[name]], fmt="%d %.6f")
draw("rmsd_per_residue", "C-alpha RMSD per residue vs first frame, 300 K (r1)", "Residue", "RMSD (nm)",
     [(np.arange(1,416), resR[c], s) for c, s in sty.items()])
m = np.mean(list(resR.values()), axis=0)
draw("rmsd_per_residue_mean", "Mean C-alpha RMSD per residue (3 chains), 300 K (r1)", "Residue", "RMSD (nm)",
     [(np.arange(1,416), m, dict(color="grey", label="Mean of chains"))])

nmA = np.array([n.strip() for n in u.atoms.names]); rn = u.atoms.resnames
resid = u.atoms.resids; chain = np.arange(len(nmA))//6647
isH = np.array([n.startswith("H") for n in nmA])
P0 = u.atoms.positions/10; heavy = np.where(~isH)[0]; Hs = np.where(isH)[0]
don = heavy[cKDTree(P0[heavy]).query(P0[Hs])[1]]
k = np.array([nmA[i][0] in "NO" for i in don]); Hs, don = Hs[k], don[k]
acc = np.array([i for i in range(len(nmA)) if nmA[i][0]=="O" or (nmA[i] in ("ND1","NE2") and rn[i].startswith("HI"))])
tot = np.zeros((3,416)); inter = np.zeros((3,416)); nb = []
for ts in u.trajectory:
    Pf = u.atoms.positions/10; n = 0
    for hi, lst in enumerate(cKDTree(Pf[Hs]).query_ball_tree(cKDTree(Pf[acc]), 0.25)):
        D = don[hi]
        for aj in lst:
            A = acc[aj]
            if A == D or (chain[A]==chain[D] and resid[A]==resid[D]): continue
            v1, v2 = Pf[Hs[hi]]-Pf[D], Pf[A]-Pf[D]; dist = np.linalg.norm(v2)
            if dist > 0.35 or v1@v2/(np.linalg.norm(v1)*dist) < np.cos(np.radians(30)): continue
            n += 1
            for Xa in (D, A):
                tot[chain[Xa], resid[Xa]] += 1
                if chain[D] != chain[A]: inter[chain[Xa], resid[Xa]] += 1
    nb.append(n)
nf = len(nb); print("mean H-bonds/frame:", round(float(np.mean(nb)),1))
for nmx, arr, ttl in (("hbperres",tot,"All H-bonds per residue"), ("hbperres_inter",inter,"Inter-chain H-bonds per residue")):
    for i, c in enumerate(sty): np.savetxt(f"{X}/{nmx}_{c}.xvg", np.c_[np.arange(1,416), arr[i,1:]/nf], fmt="%d %.5f")
    draw(nmx, f"{ttl}, 300 K (r1)", "Residue", "H-bonds per frame",
         [(np.arange(1,416), arr[i,1:]/nf, s) for i, s in enumerate(sty.values())])

In [ ]:
import numpy as np, matplotlib.pyplot as plt, os
OUT = os.environ["OUT"]
a = np.array([list(l.rstrip("\n")) for l in open(f"{OUT}/xvgs/dssp_chA.dat") if l.strip()])
brk = np.where((a == "=").all(0))[0]
print("shape:", a.shape, "| chain breaks at columns:", brk)
a = a[:, ~(a == "=").all(0)]
assert a.shape[1] == 1245, a.shape
D = {c: a[:, i*415:(i+1)*415] for i, c in enumerate(("chA","chB","chC"))}

sty = {"chA": dict(color="#d62728", ls="-", label="Chain A"),
       "chB": dict(color="#7fbfff", ls=":", label="Chain B"),
       "chC": dict(color="#1f77b4", ls="-", label="Chain C")}
t = np.linspace(0, 1, len(a))
for name, ss, yl in (("helix","HGI","Helix residues (%)"), ("strand","EB","Strand residues (%)")):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        ax.plot(t, 100*np.isin(D[c], list(ss)).mean(1), lw=1.8, **s)
        print(name, c, "mean %:", round(100*np.isin(D[c], list(ss)).mean(), 1))
    ax.set(xlabel="Time (ns)", ylabel=yl, title=f"{name.capitalize()} content vs time, 300 K (r1)")
    ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_{name}_time.png"); plt.show()

    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        f = np.isin(D[c], list(ss)).mean(0); ax.plot(np.arange(1,416), f, lw=1.6, **s)
        np.savetxt(f"{OUT}/xvgs/dssp_{name}_perres_{c}.xvg", np.c_[np.arange(1,416), f], fmt="%d %.4f")
    ax.set(xlabel="Residue", ylabel=f"Fraction of frames in {name}", title=f"{name.capitalize()} per residue, 300 K (r1)")
    ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_{name}_perres.png"); plt.show()

In [ ]:
from matplotlib.colors import ListedColormap
fig, ax = plt.subplots(figsize=(12,5), dpi=150)
for c, s in sty.items():
    ax.plot(t, 100*(1-np.isin(D[c], list("HGIEB")).mean(1)), lw=1.8, **s)
ax.set(xlabel="Time (ns)", ylabel="Coil/turn residues (%)", title="Coil/turn content vs time, 300 K (r1)")
ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_coil_time.png"); plt.show()

cmap = ListedColormap(["#eeeeee", "#d62728", "#1f77b4"])
code = lambda m: np.where(np.isin(m, list("HGI")), 1, np.where(np.isin(m, list("EB")), 2, 0))
fig, axs = plt.subplots(3, 1, figsize=(12,8), dpi=150, sharex=True)
for ax, (c, s) in zip(axs, sty.items()):
    ax.imshow(code(D[c]), aspect="auto", cmap=cmap, vmin=0, vmax=2, origin="lower", extent=[1,415,0,1])
    ax.set_ylabel("Time (ns)"); ax.set_title(s["label"], loc="left", fontsize=10)
axs[-1].set_xlabel("Residue"); fig.suptitle("DSSP map: red = helix, blue = strand, grey = coil/turn, 300 K (r1)")
fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_map.png"); plt.show()

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
OUT = os.environ["OUT"]

def load_dssp(path):
    a = np.array([list(l.rstrip("\n")) for l in open(path) if l.strip()])
    return a[:, ~(a == "=").all(0)]

STATES = [("Alpha helix","H"),("3-10 helix","G"),("Pi helix","I"),("Beta strand","E"),("Beta bridge","B"),
          ("Turn","T"),("Bend","S"),("Polyproline II helix","P")]
def percent(a):
    P = {n: 100*(a == c).mean(1) for n, c in STATES}
    P["Coil / Unstructured"] = 100 - sum(P.values())
    return P

a = load_dssp(f"{OUT}/xvgs/dssp_chA.dat"); print(a.shape)   # expect (201, 1245)
P = percent(a); t = np.arange(len(a))*5.0

fig, ax = plt.subplots(figsize=(12,6.2), dpi=150)
for i, (n, y) in enumerate(P.items()): ax.plot(t, y, color=plt.cm.tab10(i), lw=1.6, label=n)
ax.set(xlabel="Time (ps)", ylabel="Percentage (%)", title="Vicilin All Secondary Structure States — 300 K (r1)")
ax.grid(alpha=.3); ax.legend(title="Secondary structure", ncol=2, loc="upper right"); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/dssp_all_states_300K_r1.png"); plt.show()

G = {"Alpha helix": P["Alpha helix"], "Beta structure": P["Beta strand"]+P["Beta bridge"], "Turns": P["Turn"]}
G["Loops / Other"] = 100 - sum(G.values())
fig, ax = plt.subplots(figsize=(10,6), dpi=150)
for i, (n, y) in enumerate(G.items()): ax.plot(t, y, color=plt.cm.tab10(i), lw=1.8, label=n)
ax.set(xlabel="Time (ps)", ylabel="Percentage (%)", title="Vicilin Secondary Structure — 300 K (r1)")
ax.grid(alpha=.3); ax.legend(); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/dssp_groups_300K_r1.png"); plt.show()

## 9. All-runs comparison and box checks
Comparison across the first four runs and a check of the box dimensions in the shear trajectory.

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/Plant Protein Extrusion MD/analysism"
RUNS = {"300 K r1":"vicilin_300_r1","300 K r2":"vicilin_300_r2",
        "373 K r1":"vicilin_373_r1","373 K shear":"vicilin_373_shear_r1"}
COL = {"300 K r1":"#1f77b4","300 K r2":"#7fbfff","373 K r1":"#d62728","373 K shear":"#2ca02c"}
LS  = {"300 K r1":"-","300 K r2":"--","373 K r1":"-","373 K shear":"--"}
CMP = f"{BASE}/comparison_all_runs"
for d in ("graphs","csv"): os.makedirs(f"{CMP}/{d}", exist_ok=True)
CH = ["chA","chB","chC"]; PAIRS = [("chA","chB"),("chB","chC"),("chA","chC")]
PN = [f"{a[-1]}-{b[-1]}" for a,b in PAIRS]
STAT = [("Alpha helix","H"),("3-10 helix","G"),("Pi helix","I"),("Beta strand","E"),
        ("Beta bridge","B"),("Turn","T"),("Bend","S"),("PPII helix","P")]
xvg = lambda f: np.loadtxt(f, comments=('#','@'))
ok = os.path.exists
def load_dssp(p):
    a = np.array([list(l.rstrip("\n")) for l in open(p) if l.strip()])
    return a[:, ~(a == "=").all(0)]

# ---------------- load everything ----------------
rows, resrows, comp = [], [], {}
def add(run, m, e, t, y): rows.extend((run, m, e, float(a), float(b)) for a, b in zip(t, y))
spec = {"RMSD (nm)":("rmsd",1,1.0), "Rg (nm)":("rg",1,1000.0),
        "SASA (nm2)":("sasa",2,1.0), "H-bonds (trimer)":("hbnum",1,1.0)}
RESPAT = {"RMSF (nm)":"rmsf_{c}", "RMSD per residue (nm)":"rmsd_perres_{c}",
          "H-bonds per residue":"hbperres_{c}", "Inter-chain H-bonds per residue":"hbperres_inter_{c}",
          "Helix fraction":"dssp_helix_perres_{c}", "Strand fraction":"dssp_strand_perres_{c}"}
for run, fd in RUNS.items():
    X = f"{BASE}/{fd}/xvgs"
    if not os.path.isdir(X): print("MISSING run folder:", run); continue
    for m, (pre, col, sc) in spec.items():
        for e in CH + ["Protein"]:
            f = f"{X}/{pre}_{e}.xvg"
            if ok(f): a = xvg(f); add(run, m, e, a[:,0]/sc, a[:,col])
            else: print("missing", run, f"{pre}_{e}.xvg")
    for (a_, b_), n in zip(PAIRS, PN):
        f = f"{X}/hbinter_{a_}_{b_}.xvg"
        if ok(f): a = xvg(f); add(run, "Inter H-bonds", n, a[:,0], a[:,1])
    f = f"{X}/dssp_chA.dat"
    if ok(f):
        a = load_dssp(f); n = len(a); t = np.linspace(0, 1, n)   # assumes frames span 1 ns
        ents = {"Protein": a, **{c: a[:, i*415:(i+1)*415] for i, c in enumerate(CH)}}
        for e, mm in ents.items():
            hel = 100*np.isin(mm, list("HGI")).mean(1); stn = 100*np.isin(mm, list("EB")).mean(1)
            add(run, "Helix (%)", e, t, hel); add(run, "Strand (%)", e, t, stn)
            add(run, "Coil/turn (%)", e, t, 100-hel-stn)
        comp[run] = {nm: 100*(a[n//2:] == ch).mean() for nm, ch in STAT}
        comp[run]["Coil/other"] = 100 - sum(comp[run].values())
    for c in CH:
        for m, pat in RESPAT.items():
            f = f"{X}/{pat.format(c=c)}.xvg"
            if ok(f):
                v = xvg(f)[:,1]
                if len(v) == 415: resrows.extend((run, m, c, i+1, float(x)) for i, x in enumerate(v))
    f = f"{X}/sasa_perres_Protein.xvg"
    if ok(f):
        d = xvg(f)
        if len(d) == 1245:
            for i, c in enumerate(CH):
                resrows.extend((run, "SASA per residue (nm2)", c, j+1, float(x)) for j, x in enumerate(d[i*415:(i+1)*415, 1]))

ts  = pd.DataFrame(rows, columns=["run","metric","entity","time_ns","value"])
res = pd.DataFrame(resrows, columns=["run","metric","chain","residue","value"])
ts.to_csv(f"{CMP}/csv/timeseries_long.csv", index=False)
res.to_csv(f"{CMP}/csv/residue_metrics_long.csv", index=False)
pd.DataFrame(comp).T.round(2).to_csv(f"{CMP}/csv/dssp_composition_last_half.csv")

# ---------------- summaries ----------------
def stats(t, y):
    h = t >= t.max()/2
    return dict(mean_full=y.mean(), sd_full=y.std(ddof=1), mean_last_half=y[h].mean(), sd_last_half=y[h].std(ddof=1))
pc = [dict(run=r, metric=m, entity=e, **stats(d.sort_values("time_ns").time_ns.values, d.sort_values("time_ns").value.values))
      for (r, m, e), d in ts.groupby(["run","metric","entity"])]
pd.DataFrame(pc).round(4).to_csv(f"{CMP}/csv/summary_per_chain_and_interface.csv", index=False)

MET = [("RMSD (nm)",["Protein"],"mean","Backbone RMSD, whole trimer (nm)"),
       ("Rg (nm)",["Protein"],"mean","Radius of gyration, whole trimer (nm)"),
       ("SASA (nm2)",["Protein"],"mean","SASA, whole trimer (nm²)"),
       ("H-bonds (trimer)",["Protein"],"mean","H-bonds within trimer"),
       ("Inter H-bonds",PN,"sum","Inter-chain H-bonds (sum of A-B, B-C, A-C)"),
       ("Helix (%)",["Protein"],"mean","Helix (%)"),
       ("Strand (%)",["Protein"],"mean","Strand (%)"),
       ("Coil/turn (%)",["Protein"],"mean","Coil/turn (%)")]
def curve(run, m, ents, agg):
    d = ts[(ts.run==run) & (ts.metric==m) & (ts.entity.isin(ents))]
    if d.empty: return None
    pr = [e for e in ents if (d.entity==e).any()]
    t0 = d[d.entity==pr[0]].sort_values("time_ns").time_ns.values
    Y = np.array([np.interp(t0, *[d[d.entity==e].sort_values("time_ns")[k].values for k in ("time_ns","value")]) for e in pr])
    return t0, (Y.sum(0) if agg=="sum" else Y.mean(0))
agg_rows = []
for run in RUNS:
    for m, ents, agg, yl in MET:
        c = curve(run, m, ents, agg)
        if c: agg_rows.append(dict(run=run, metric=m, **stats(*c)))
S = pd.DataFrame(agg_rows); S.round(4).to_csv(f"{CMP}/csv/summary_all_runs.csv", index=False)
table = S.pivot(index="run", columns="metric", values="mean_last_half").reindex(list(RUNS))
table = table[[m[0] for m in MET if m[0] in table.columns]]
base = table.loc[["300 K r1","300 K r2"]].mean()
th = table.loc["373 K r1"] - base; sh = table.loc["373 K shear"] - table.loc["373 K r1"]
rep = (table.loc["300 K r1"] - table.loc["300 K r2"]).abs()
D = pd.DataFrame({"mean_300K_r1_r2":base, "val_373K_r1":table.loc["373 K r1"], "val_373K_shear":table.loc["373 K shear"],
                  "thermal_delta(373r1-300)":th, "thermal_pct":100*th/base,
                  "shear_delta(shear-373r1)":sh, "shear_pct":100*sh/table.loc["373 K r1"],
                  "replicate_gap(300r1-r2)":rep})
D["thermal_exceeds_replicate_gap"] = th.abs() > rep
D["shear_exceeds_replicate_gap"] = sh.abs() > rep
D.round(4).to_csv(f"{CMP}/csv/condition_deltas.csv")

# ---------------- graphs: time series ----------------
def ts_ax(ax, m, ents, agg, yl):
    for run in RUNS:
        c = curve(run, m, ents, agg)
        if c: ax.plot(c[0], c[1], color=COL[run], ls=LS[run], lw=1.7, label=run)
    ax.set(xlabel="Time (ns)", ylabel=yl, title=yl); ax.grid(alpha=.3); ax.legend(fontsize=8)
fig, axs = plt.subplots(4, 2, figsize=(14, 17))
for ax, (m, ents, agg, yl) in zip(axs.ravel(), MET): ts_ax(ax, m, ents, agg, yl)
fig.suptitle("Vicilin trimer: all runs", y=1.0); fig.tight_layout()
fig.savefig(f"{CMP}/graphs/00_overview_timeseries.png", dpi=150); plt.show()
for i, (m, ents, agg, yl) in enumerate(MET, 1):
    f2, a2 = plt.subplots(figsize=(10, 5), dpi=150); ts_ax(a2, m, ents, agg, yl); f2.tight_layout()
    f2.savefig(f"{CMP}/graphs/ts_{m.split(' (')[0].replace(' ','_').replace('/','_')}.png"); plt.close(f2)

# ---------------- graphs: bars ----------------
fig, axs = plt.subplots(2, 4, figsize=(16, 7))
for ax, (m, ents, agg, yl) in zip(axs.ravel(), MET):
    d = S[S.metric==m].set_index("run").reindex(list(RUNS)).dropna()
    ax.bar(d.index, d.mean_last_half, yerr=d.sd_last_half, color=[COL[r] for r in d.index], capsize=4)
    ax.set_title(m, fontsize=10); ax.tick_params(axis="x", rotation=30); ax.grid(axis="y", alpha=.3)
fig.suptitle("Last-half mean ± SD (whole trimer)"); fig.tight_layout()
fig.savefig(f"{CMP}/graphs/01_bar_summary.png", dpi=150); plt.show()

P = pd.DataFrame(pc)
fig, axs = plt.subplots(2, 3, figsize=(16, 8)); w = 0.2
for ax, m in zip(axs.ravel(), ["RMSD (nm)","Rg (nm)","SASA (nm2)","H-bonds (trimer)","Helix (%)","Strand (%)"]):
    for k, run in enumerate(RUNS):
        d = P[(P.run==run)&(P.metric==m)&(P.entity.isin(CH))].set_index("entity").reindex(CH)
        ax.bar(np.arange(3)+k*w, d.mean_last_half, w, yerr=d.sd_last_half, color=COL[run], label=run, capsize=2)
    ax.set_xticks(np.arange(3)+1.5*w); ax.set_xticklabels(["Chain A","Chain B","Chain C"]); ax.set_title(m); ax.grid(axis="y", alpha=.3)
axs[0,0].legend(fontsize=8); fig.suptitle("Per-chain last-half mean ± SD"); fig.tight_layout()
fig.savefig(f"{CMP}/graphs/02_per_chain_bars.png", dpi=150); plt.show()

# heatmap of % change vs 300 K baseline
pct = 100*(table - base)/base
fig, ax = plt.subplots(figsize=(11, 3.8)); v = np.nanpercentile(np.abs(pct.values), 95)
im = ax.imshow(pct.values, cmap="RdBu_r", vmin=-v, vmax=v, aspect="auto")
ax.set_xticks(range(pct.shape[1])); ax.set_xticklabels(pct.columns, rotation=30, ha="right")
ax.set_yticks(range(pct.shape[0])); ax.set_yticklabels(pct.index)
for i in range(pct.shape[0]):
    for j in range(pct.shape[1]): ax.text(j, i, f"{pct.values[i,j]:.1f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, label="% change vs mean of 300 K r1/r2"); ax.set_title("Change relative to 300 K baseline (%)")
fig.tight_layout(); fig.savefig(f"{CMP}/graphs/03_pct_change_heatmap.png", dpi=150); plt.show()

# DSSP composition
if comp:
    C = pd.DataFrame(comp).T.reindex([r for r in RUNS if r in comp])
    ax = C.plot(kind="bar", stacked=True, figsize=(10, 5), colormap="tab20"); ax.set_ylabel("% of residues (last half)")
    ax.set_title("DSSP composition, whole trimer"); ax.legend(ncol=2, fontsize=8, bbox_to_anchor=(1.01, 1))
    plt.xticks(rotation=20); plt.tight_layout(); plt.savefig(f"{CMP}/graphs/04_dssp_composition.png", dpi=150); plt.show()

# ---------------- graphs: per residue ----------------
rm = res.groupby(["run","metric","residue"]).value.mean()
def rcurve(run, m):
    try: s = rm.loc[(run, m)]; return s.index.values, s.values
    except KeyError: return None
RM = list(RESPAT) + ["SASA per residue (nm2)"]
fig, axs = plt.subplots(4, 2, figsize=(15, 17))
for ax, m in zip(axs.ravel(), RM):
    for run in RUNS:
        c = rcurve(run, m)
        if c: ax.plot(*c, color=COL[run], ls=LS[run], lw=1.3, label=run)
    ax.set(xlabel="Residue", ylabel=m, title=m + " (mean of 3 chains)"); ax.grid(alpha=.3); ax.legend(fontsize=8)
axs.ravel()[-1].axis("off"); fig.tight_layout()
fig.savefig(f"{CMP}/graphs/05_residue_overlays.png", dpi=150); plt.show()

DM = ["RMSF (nm)","SASA per residue (nm2)","Helix fraction","Strand fraction","H-bonds per residue"]
fig, axs = plt.subplots(len(DM), 2, figsize=(15, 3.2*len(DM))); top = []
for i, m in enumerate(DM):
    a300 = [rcurve(r, m) for r in ("300 K r1","300 K r2")]; a300 = [c[1] for c in a300 if c]
    r373, rsh = rcurve("373 K r1", m), rcurve("373 K shear", m)
    if not (a300 and r373 and rsh): continue
    resid = r373[0]; b = np.mean(a300, axis=0)
    for j, (name, dl) in enumerate((("Thermal: 373 K r1 − 300 K mean", r373[1]-b), ("Shear: 373 K shear − 373 K r1", rsh[1]-r373[1]))):
        ax = axs[i, j]; ax.fill_between(resid, dl, 0, where=dl>=0, color="#d62728", alpha=.6)
        ax.fill_between(resid, dl, 0, where=dl<0, color="#1f77b4", alpha=.6)
        ax.axhline(0, color="k", lw=.6); ax.set(xlabel="Residue", ylabel=f"Δ {m}", title=f"{name}"); ax.grid(alpha=.3)
        for k in np.argsort(-np.abs(dl))[:10]: top.append(dict(contrast=name, metric=m, residue=int(resid[k]), delta=float(dl[k])))
fig.tight_layout(); fig.savefig(f"{CMP}/graphs/06_residue_deltas.png", dpi=150); plt.show()
pd.DataFrame(top).round(4).to_csv(f"{CMP}/csv/top10_residue_changes.csv", index=False)

# ---------------- report ----------------
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
print("\nLast-half means, whole trimer:\n", table.round(3))
print("\nDeltas vs replicate gap:\n", D.round(3))
print("\nSaved to:", CMP); os.system(f'ls -R "{CMP}" | head -60')

In [ ]:
import numpy as np
from MDAnalysis.coordinates.XTC import XTCReader
r = XTCReader("/content/shear/md_373_shear_r1.xtc")
d = np.array([ts.dimensions.copy() for ts in r])
print("frames:", len(d))
print("a,b,c start:", d[0,:3].round(2), "end:", d[-1,:3].round(2))
print("alpha,beta,gamma start:", d[0,3:].round(2), "end:", d[-1,3:].round(2))
print("max angle change (deg):", np.abs(d[:,3:] - d[0,3:]).max(axis=0).round(2))

In [ ]:
import os, glob, numpy as np, pandas as pd, MDAnalysis as mda, matplotlib.pyplot as plt
from MDAnalysis.lib.distances import minimize_vectors
from MDAnalysis.analysis.align import rotation_matrix
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
BASE = "/content/drive/MyDrive/Plant Protein Extrusion MD"
os.system(f'ln -sfn "{BASE}" /content/pm')
sr = [d for d in glob.glob("/content/pm/Shear Run*") if os.path.isdir(d)][0]
os.system(f'ln -sfn "{sr}" /content/shear')
RUNS = {"373 K shear":("/content/shear","md_373_shear_r1","vicilin_373_shear_r1"),
        "373 K r1":("/content/pm","md_373_r1","vicilin_373_r1"),
        "300 K r1":("/content/pm","md_300_r1","vicilin_300_r1"),
        "300 K r2":("/content/pm","md_300_r2","vicilin_300_r2")}
sl = [slice(0,6647), slice(6647,13294), slice(13294,19941)]
rows = []
for run, (P, S, name) in RUNS.items():
    F = f"{BASE}/analysism/{name}/frames"
    os.system(f'printf "Protein\\n" | gmx trjconv -s "{F}/protein.tpr" -f "{P}/{S}.xtc" -o /tmp/w.xtc -pbc whole > /tmp/w.log 2>&1')
    u = mda.Universe(f"{F}/first_cl.gro", "/tmp/w.xtc"); ca = u.select_atoms("name CA").indices
    ref = None
    for ts in u.trajectory[::5]:
        pos = u.atoms.positions.copy(); box = ts.dimensions.astype(np.float32)
        a0 = pos[sl[0]].mean(0)
        for s in sl[1:]:
            d = pos[s].mean(0) - a0
            dm = minimize_vectors(d[None,:].astype(np.float32), box)[0]
            pos[s] -= (d - dm)
        x = pos[ca]; x = x - x.mean(0)
        if ref is None: ref = x.copy()
        R, _ = rotation_matrix(x, ref); Rt = R.T          # reference -> this frame
        th = np.arccos(np.clip((np.trace(Rt)-1)/2, -1, 1))
        ax = np.array([Rt[2,1]-Rt[1,2], Rt[0,2]-Rt[2,0], Rt[1,0]-Rt[0,1]])
        n = np.linalg.norm(ax); ax = ax/n if n > 1e-12 else np.zeros(3)
        rv = np.degrees(th)*ax
        rows.append(dict(run=run, time_ns=ts.time/1000, angle_deg=np.degrees(th), rot_x=rv[0], rot_y=rv[1], rot_z=rv[2]))
    os.remove("/tmp/w.xtc")
df = pd.DataFrame(rows)
CMP = f"{BASE}/analysism/comparison_all_runs"
for d in ("csv","graphs"): os.makedirs(f"{CMP}/{d}", exist_ok=True)
df.round(3).to_csv(f"{CMP}/csv/rotation_no_fit.csv", index=False)

COL = {"300 K r1":"#1f77b4","300 K r2":"#7fbfff","373 K r1":"#d62728","373 K shear":"#2ca02c"}
fig, axs = plt.subplots(1, 2, figsize=(14,5), dpi=150)
for run, d in df.groupby("run"):
    axs[0].plot(d.time_ns, d.angle_deg, color=COL[run], label=run)
    axs[1].plot(d.time_ns, d.rot_z, color=COL[run], label=run)
tt = np.linspace(0, 1, 50); axs[1].plot(tt, -7.16*tt, "k--", label="expected, rigid sphere in shear")
axs[0].set(xlabel="Time (ns)", ylabel="Total rotation from frame 0 (deg)", title="Trimer rotation, no rotational fit")
axs[1].set(xlabel="Time (ns)", ylabel="Rotation about z (deg)", title="Rotation about vorticity axis (z)")
for a in axs: a.grid(alpha=.3); a.legend(fontsize=8)
fig.tight_layout(); fig.savefig(f"{CMP}/graphs/07_rotation_no_fit.png"); plt.show()
print(df.groupby("run").tail(1)[["run","angle_deg","rot_x","rot_y","rot_z"]].round(1))

## 10. Replicate 2: 373 K r2 and shear r2
Setup (r2 uses `gen-seed = 222`), NPT and production checks, and building the combined shear r2 trajectory.

In [ ]:
import os, subprocess
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
cmd = r'''cd "/content/drive/MyDrive/Plant Protein Extrusion MD" || exit 1
echo "== molecules in topol.top"; sed -n '/\[ molecules \]/,$p' topol.top
echo; echo "== atoms in em.gro"; sed -n '2p' em.gro
echo; echo "== 373 K files"; ls -1 *373* | head -40
echo; echo "== any r2 files (expect none)"; ls -1 *_r2* 2>&1 | head
echo; echo "== NVT r1 mdp"; cat nvt_373_r1.mdp
echo; echo "== NVT / NPT r1 launch lines"; grep -A1 -m1 "Command line" nvt_373_r1.log; grep -A1 -m1 "Command line" npt_373_r1*.log | head -6
echo; echo "== free space"; df -h /content/drive | tail -1'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

In [ ]:
import os, subprocess
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
cmd = r'''cd "/content/drive/MyDrive/Plant Protein Extrusion MD" || exit 1
if ls nvt_373_r2.* >/dev/null 2>&1; then echo "STOP: nvt_373_r2 files already exist:"; ls nvt_373_r2.*; exit 0; fi
sed 's/^gen-seed = 111/gen-seed = 222/' nvt_373_r1.mdp > nvt_373_r2.mdp
echo "== mdp diff r1 vs r2 (expect gen-seed only)"; diff nvt_373_r1.mdp nvt_373_r2.mdp
echo; echo "== grompp"
gmx grompp -f nvt_373_r2.mdp -c em.gro -r em.gro -p topol.top -o nvt_373_r2.tpr -maxwarn 0 > /tmp/g.txt 2>&1
grep -iE "warning|error|fatal|note" /tmp/g.txt | head -10; tail -3 /tmp/g.txt
ls -lh nvt_373_r1.tpr nvt_373_r2.tpr
echo; echo "== r1 vs r2 tpr"
gmx check -s1 nvt_373_r1.tpr -s2 nvt_373_r2.tpr > /tmp/chk.txt 2>&1
echo "x lines differing: $(grep -cE '^x\[' /tmp/chk.txt)"
echo "v lines differing: $(grep -cE '^v\[' /tmp/chk.txt)"
echo; echo "== other differences (expect only ld_seed)"
grep -vE "^v\[|^x\[|t_resinfo|^ *$|^comparing|Reading|GROMACS|Executable|Data prefix|Working dir|Command line|gmx check|Note:|^ *[0-9.e+-]+ +[0-9.e+-]+ +[0-9.e+-]+\)" /tmp/chk.txt
echo; echo "== r2 NVT files (expect mdp and tpr only)"; ls nvt_373_r2.*'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

In [ ]:
import os, subprocess
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
r = subprocess.run(["bash","-c",'cd "/content/drive/MyDrive/Plant Protein Extrusion MD" && gmx mdrun -deffnm nvt_373_r2 -cpt 5 -ntmpi 1 -ntomp 2 -pin off 2>&1 | tail -5'], capture_output=True, text=True); print(r.stdout)

In [ ]:
import os, subprocess
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
cmd = r'''cd "/content/drive/MyDrive/Plant Protein Extrusion MD" || exit 1
echo "== NVT r2 files"; ls -lh nvt_373_r2.*
echo; echo "== finished normally? (expect 1)"; grep -c "Finished mdrun" nvt_373_r2.log
echo; echo "== NVT r2 temperature"
printf "Temperature\n0\n" | gmx energy -f nvt_373_r2.edr -o /tmp/t_r2.xvg 2>&1 | grep -E "^Temperature|Statistics"
echo; echo "== NVT r1 temperature (comparison)"
printf "Temperature\n0\n" | gmx energy -f nvt_373_r1.edr -o /tmp/t_r1.xvg 2>&1 | grep -E "^Temperature"
echo; echo "== NPT r1 mdp"; cat npt_373_r1.mdp
echo; echo "== any r2 NPT files (expect none)"; ls npt_373_r2* 2>&1'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

In [ ]:
import os, subprocess
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
cmd = r'''cd "/content/drive/MyDrive/Plant Protein Extrusion MD" || exit 1
if ls npt_373_r2.* >/dev/null 2>&1; then echo "STOP: npt_373_r2 files already exist:"; ls npt_373_r2.*; exit 0; fi
cp npt_373_r1.mdp npt_373_r2.mdp
echo "== mdp diff r1 vs r2 (expect none)"; diff npt_373_r1.mdp npt_373_r2.mdp && echo IDENTICAL
echo; echo "== grompp"
gmx grompp -f npt_373_r2.mdp -c nvt_373_r2.gro -r nvt_373_r2.gro -t nvt_373_r2.cpt -p topol.top -o npt_373_r2.tpr -maxwarn 0 > /tmp/g.txt 2>&1
grep -iE "warning|error|fatal|note" /tmp/g.txt | head -10; tail -3 /tmp/g.txt
ls -lh npt_373_r1.tpr npt_373_r2.tpr
echo; echo "== inputrec/topology differences vs r1 (x, v, resinfo hidden)"
gmx check -s1 npt_373_r1.tpr -s2 npt_373_r2.tpr 2>&1 | grep -vE "^v\[|^x\[|t_resinfo|^ *$|^comparing|Reading|GROMACS|Executable|Data prefix|Working dir|Command line|gmx check|Note:|^ *[0-9.e+-]+ +[0-9.e+-]+ +[0-9.e+-]+\)"
echo; echo "== r2 NPT files (expect mdp and tpr only)"; ls npt_373_r2.*'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

In [ ]:
import os, subprocess
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
r = subprocess.run(["bash","-c",'cd "/content/drive/MyDrive/Plant Protein Extrusion MD" && gmx mdrun -deffnm npt_373_r2 -cpt 5 -ntmpi 1 -ntomp 2 -pin off 2>&1 | tail -5'], capture_output=True, text=True); print(r.stdout)

In [ ]:
import os, subprocess
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
cmd = r'''cd "/content/drive/MyDrive/Plant Protein Extrusion MD" || exit 1
echo "== NPT r2 files"; ls -lh npt_373_r2.*
echo; echo "== finished normally? (expect 1)"; grep -c "Finished mdrun" npt_373_r2.log
echo; echo "== NPT r2 averages"
printf "Temperature\nPressure\nDensity\n0\n" | gmx energy -f npt_373_r2.edr -o /tmp/npt_r2.xvg 2>&1 | grep -E "^(Temperature|Pressure|Density)"
echo; echo "== NPT r1 averages (comparison)"
printf "Temperature\nPressure\nDensity\n0\n" | gmx energy -f npt_373_r1.edr -o /tmp/npt_r1.xvg 2>&1 | grep -E "^(Temperature|Pressure|Density)"
echo; echo "== shear r1 mdp (vicilin)"; cat md_373_shear_r1.mdp
echo; echo "== shear r1 launch line"; grep -A1 -m1 "Command line" md_373_shear_r1.log
echo; echo "== did shear r1 start from the same structure as thermal r1? (x lines differing)"
gmx check -s1 md_373_r1.tpr -s2 md_373_shear_r1.tpr 2>&1 | grep -cE "^x\["
echo; echo "== any r2 shear or thermal files (expect none)"; ls md_373_r2* md_373_shear_r2* 2>&1'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

In [ ]:
import os, subprocess
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
cmd = r'''cd "/content/drive/MyDrive/Plant Protein Extrusion MD" || exit 1
if ls md_373_shear_r2.* >/dev/null 2>&1; then echo "STOP: md_373_shear_r2 files already exist:"; ls md_373_shear_r2.*; exit 0; fi
sed 's/^ld-seed = 111/ld-seed = 222/' md_373_shear_r1.mdp > md_373_shear_r2.mdp
echo "== mdp diff r1 vs r2 (expect ld-seed only)"; diff md_373_shear_r1.mdp md_373_shear_r2.mdp
echo; echo "== grompp"
gmx grompp -f md_373_shear_r2.mdp -c npt_373_r2.gro -t npt_373_r2.cpt -p topol.top -o md_373_shear_r2.tpr -maxwarn 0 > /tmp/g.txt 2>&1
grep -iE "warning|error|fatal|note" /tmp/g.txt | head -10; tail -3 /tmp/g.txt
ls -lh md_373_shear_r1.tpr md_373_shear_r2.tpr
echo; echo "== inputrec differences vs r1 (x, v, resinfo hidden)"
gmx check -s1 md_373_shear_r1.tpr -s2 md_373_shear_r2.tpr 2>&1 | grep -vE "^v\[|^x\[|t_resinfo|^ *$|^comparing|Reading|GROMACS|Executable|Data prefix|Working dir|Command line|gmx check|Note:|^ *[0-9.e+-]+ +[0-9.e+-]+ +[0-9.e+-]+\)"
echo; echo "== did r1 shear and r1 thermal share velocities? (0 means same NPT start)"
gmx check -s1 md_373_r1.tpr -s2 md_373_shear_r1.tpr 2>&1 | grep -cE "^v\["
echo; echo "== r2 shear files (expect mdp and tpr only)"; ls md_373_shear_r2.*'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

In [ ]:
import os, subprocess
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
r = subprocess.run(["bash","-c",'cd "/content/drive/MyDrive/Plant Protein Extrusion MD" && gmx mdrun -deffnm md_373_shear_r2 -cpt 5 -ntmpi 1 -ntomp 2 -pin off 2>&1 | tail -5'], capture_output=True, text=True); print(r.stdout)

In [ ]:
import os, subprocess
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
cmd = r'''cd "/content/drive/MyDrive/Plant Protein Extrusion MD" || exit 1
echo "== shear r2 files"; ls -lh --time-style=+%H:%M md_373_shear_r2*
echo; echo "== last progress lines in log"; grep -E "^ +Step +Time" -A1 md_373_shear_r2.log | tail -2
echo; echo "== checkpoint time (step reached)"
gmx check -f md_373_shear_r2.cpt 2>&1 | grep -iE "step|time" | head -5
echo; echo "== finished? (expect 0)"; grep -c "Finished mdrun" md_373_shear_r2.log
echo; echo "== free space"; df -h /content/drive | tail -1'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

In [ ]:
import os, subprocess
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
cmd = r'''cd "/content/drive/MyDrive/Plant Protein Extrusion MD" || exit 1
B=backups/md_373_shear_r2_backup
echo "== backup files"; ls -lh --time-style=+%H:%M $B
echo; echo "== backup xtc: first/last frame"
gmx check -f $B/md_373_shear_r2.xtc 2>&1 | grep -E "Reading frame +0 |Last frame"
echo; echo "== backup edr: time range"
printf "Temperature\n0\n" | gmx energy -f $B/md_373_shear_r2.edr -o /tmp/b.xvg 2>&1 | grep -E "Statistics"
echo; echo "== backup cpt vs main cpt (step reached)"
gmx check -f $B/md_373_shear_r2.cpt 2>&1 | grep -E "Last frame"
gmx check -f md_373_shear_r2.cpt 2>&1 | grep -E "Last frame"
echo; echo "== same cpt in both? (md5)"; md5sum $B/md_373_shear_r2.cpt md_373_shear_r2.cpt'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

In [ ]:
import os, subprocess
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
cmd = r'''cd "/content/drive/MyDrive/Plant Protein Extrusion MD" || exit 1
B=backups/md_373_shear_r2_backup
cp -n $B/md_373_shear_r2.xtc $B/md_373_shear_r2.edr .
echo "== main folder now"; ls -lh --time-style=+%H:%M md_373_shear_r2*
echo; echo "== xtc check, full tail"; gmx check -f md_373_shear_r2.xtc 2>&1 | tail -8'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

In [ ]:
import os, subprocess
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
r = subprocess.run(["bash","-c",'cd "/content/drive/MyDrive/Plant Protein Extrusion MD" && gmx mdrun -s md_373_shear_r2.tpr -deffnm md_373_shear_r2 -cpi md_373_shear_r2.cpt -noappend -cpt 5 -ntmpi 1 -ntomp 2 -pin off 2>&1 | tail -5'], capture_output=True, text=True); print(r.stdout)

In [ ]:
import os, subprocess
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
cmd = r'''cd "/content/drive/MyDrive/Plant Protein Extrusion MD" || exit 1
echo "== part0002 files"; ls -lh --time-style=+%H:%M md_373_shear_r2.part0002.*
echo; echo "== finished normally? (expect 1)"; grep -c "Finished mdrun" md_373_shear_r2.part0002.log
echo; echo "== resumed from which time"; grep -iE "continuing from checkpoint|Started mdrun|restart" md_373_shear_r2.part0002.log | head -3
echo; echo "== part 2 xtc: first / last frame"
gmx check -f md_373_shear_r2.part0002.xtc 2>&1 | grep -E "Reading frame +0 |Last frame"
echo; echo "== part 2 energies"
printf "Temperature\n0\n" | gmx energy -f md_373_shear_r2.part0002.edr -o /tmp/p2.xvg 2>&1 | grep -E "^Temperature|Statistics"
echo; echo "== final gro"; ls -lh md_373_shear_r2.part0002.gro'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

In [ ]:
import os, subprocess
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
cmd = r'''cd "/content/drive/MyDrive/Plant Protein Extrusion MD" || exit 1
if ls md_373_shear_r2_combined.* md_373_shear_r2_p1cut.* >/dev/null 2>&1; then echo "STOP: combined/p1cut files already exist:"; ls md_373_shear_r2_combined.* md_373_shear_r2_p1cut.*; exit 0; fi
echo "== trajectory"
printf "Protein\n" | gmx trjconv -f md_373_shear_r2.xtc -s md_373_shear_r2.tpr -e 259 -o md_373_shear_r2_p1cut.xtc > /tmp/t1.txt 2>&1; tail -2 /tmp/t1.txt
gmx trjcat -f md_373_shear_r2_p1cut.xtc md_373_shear_r2.part0002.xtc -o md_373_shear_r2_combined.xtc > /tmp/t2.txt 2>&1; tail -2 /tmp/t2.txt
echo; echo "== energies"
gmx eneconv -f md_373_shear_r2.edr -e 259 -o md_373_shear_r2_p1cut.edr > /tmp/e1.txt 2>&1; tail -2 /tmp/e1.txt
gmx eneconv -f md_373_shear_r2_p1cut.edr md_373_shear_r2.part0002.edr -o md_373_shear_r2_combined.edr > /tmp/e2.txt 2>&1; tail -2 /tmp/e2.txt
echo; echo "== check combined xtc (expect 1001 frames, last time 1000)"
gmx check -f md_373_shear_r2_combined.xtc 2>&1 | grep -E "Last frame|^Step|^Time"
echo; echo "== check combined edr (expect 0 through 1000 ps)"
printf "Temperature\n0\n" | gmx energy -f md_373_shear_r2_combined.edr -o /tmp/c.xvg 2>&1 | grep -E "Statistics|^Temperature"
echo; echo "== box at the end of the run (6th number = box tilt, expect about 3.387)"; tail -1 md_373_shear_r2.part0002.gro
echo; ls -lh md_373_shear_r2_combined.*'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

## 11. Replicate 2 analysis (one cell per run) and output checks
Single-cell version of the analysis for 373 K shear r2 and 373 K thermal r2, plus checks that all expected graphs and `.xvg` files exist.

In [ ]:
import os, sys, json, itertools, subprocess
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from google.colab import drive
TEMP, RUN = "373", "shear_r2"
TPR_NAME  = "md_373_shear_r2"                 # tpr
TRAJ_NAME = "md_373_shear_r2_combined"        # full trajectory
NCH = 3; DEFORM = 3.387                        # trimer; nm/ns tilt from the mdp
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
os.system('ln -sfn "/content/drive/MyDrive/Plant Protein Extrusion MD" /content/pm')
OUT = f"/content/pm/analysism/vicilin_{TEMP}_{RUN}"; T = "373 K shear (r2)"
os.environ.update(P="/content/pm", S=TPR_NAME, TRAJ=TRAJ_NAME, OUT=OUT)
for d in ["graphs","logs","frames","xvgs"]: os.makedirs(f"{OUT}/{d}", exist_ok=True)
subprocess.run([sys.executable,"-m","pip","install","-q","MDAnalysis","scipy"])
import MDAnalysis as mda
from MDAnalysis.coordinates.XTC import XTCReader
from MDAnalysis.lib.distances import minimize_vectors
from MDAnalysis.analysis.align import rotation_matrix
from scipy.spatial import cKDTree
from matplotlib.colors import ListedColormap
def sh(cmd, tail=0):
    r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True)
    o = (r.stdout + r.stderr).strip().split("\n")
    print("\n".join(o[-tail:] if tail else o))

# ---- check files, mdp, and trajectory continuity
print("== files"); sh('ls -lh $P/$S.tpr $P/$TRAJ.xtc; grep -iE "ref[-_]t|deform" $P/$S.mdp')
rd = XTCReader(f"{OUT.rsplit('/analysism',1)[0]}/{TRAJ_NAME}.xtc")
times = np.array([ts.time for ts in rd]); dims = np.array([ts.dimensions.copy() for ts in rd])
print("frames:", len(times), "| time", times[0], "->", times[-1], "| natoms:", rd.n_atoms)
assert times[0] == 0 and abs(times[-1]-1000) < 1e-3 and len(times) == 1001 and np.all(np.diff(times) == 1), "trajectory not continuous 0-1000 ps"
print("box a,b,c start:", dims[0,:3].round(2), "end:", dims[-1,:3].round(2))
print("angles start:", dims[0,3:].round(2), "end:", dims[-1,3:].round(2))
gd = DEFORM/dims[0,1]; print(f"shear rate {gd:.4f} /ns -> expected rigid rotation {-0.5*gd*180/np.pi:.2f} deg/ns about z")

# ---- protein tpr, whole, first frame
print("== whole")
sh(r'''cd "$OUT/frames"
printf "Protein\n" | gmx convert-tpr -s "$P/$S.tpr" -o protein.tpr 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f "$P/$TRAJ.xtc" -o whole2.xtc -pbc whole 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f whole2.xtc -o first_w.gro -dump 0 2>&1 | tail -1''')

# ---- chain detection from .gro
lines = open(f"{OUT}/frames/first_w.gro").read().split("\n")
n = int(lines[1]); body = lines[2:2+n]
resnum = np.array([int(l[0:5]) for l in body]); nm = [l[10:15].strip() for l in body]
starts = [0] + [i for i in range(1, n) if resnum[i] < resnum[i-1]]; ends = starts[1:] + [n]
nres = [len(set(resnum[s:e])) for s, e in zip(starts, ends)]
print("atoms:", n, "| chains:", len(starts), "| atoms/chain:", [e-s for s,e in zip(starts,ends)], "| res/chain:", nres)
assert len(starts) == NCH, f"expected {NCH} chains - send me the line above"
assert len(set(e-s for s,e in zip(starts,ends))) == 1, "chains differ in size - send me the line above"
NR = nres[0]; CH = ["ch"+"ABCDEF"[i] for i in range(NCH)]; G = {}; ranges = {}
for k, (s, e) in enumerate(zip(starts, ends)):
    r = range(s+1, e+1); c = CH[k]; ranges[c] = [s, e]
    G[c] = list(r); G[c+"_bb"] = [i for i in r if nm[i-1] in ("N","CA","C")]; G[c+"_CA"] = [i for i in r if nm[i-1]=="CA"]
r = range(1, n+1)
G["Protein"] = list(r); G["Protein_bb"] = [i for i in r if nm[i-1] in ("N","CA","C")]; G["Protein_CA"] = [i for i in r if nm[i-1]=="CA"]
with open(f"{OUT}/frames/chains.ndx","w") as f:
    for g, idx in G.items():
        f.write(f"[ {g} ]\n")
        for k in range(0, len(idx), 15): f.write(" ".join(f"{i:5d}" for i in idx[k:k+15]) + "\n")
json.dump(dict(ranges=ranges, nres=NR, natoms=n), open(f"{OUT}/frames/chains.json","w"))

# ---- keep chains together (triclinic-safe)
sl = [slice(*ranges[c]) for c in CH]
u = mda.Universe(f"{OUT}/frames/first_w.gro", f"{OUT}/frames/whole2.xtc")
with mda.Writer(f"{OUT}/frames/together.xtc", u.atoms.n_atoms) as W:
    for ts in u.trajectory[::5]:
        pos = u.atoms.positions.copy(); box = ts.dimensions.astype(np.float32); a0 = pos[sl[0]].mean(0)
        for s in sl[1:]:
            d = pos[s].mean(0) - a0
            dm = minimize_vectors(d[None,:].astype(np.float32), box)[0]; pos[s] -= (d - dm)
        u.atoms.positions = pos; W.write(u.atoms)

# ---- fit
print("== fit")
sh(r'''cd "$OUT/frames"
printf "Backbone\nProtein\n" | gmx trjconv -s protein.tpr -f together.xtc -o fit_cluster_skip5.xtc -fit rot+trans 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f fit_cluster_skip5.xtc -o first_cl.gro -dump 0 2>&1 | tail -1
rm -f whole2.xtc together.xtc
gmx check -f fit_cluster_skip5.xtc 2>&1 | grep -E "Last frame|^Step"''')

# ---- trimer integrity check
u = mda.Universe(f"{OUT}/frames/first_cl.gro", f"{OUT}/frames/fit_cluster_skip5.xtc")
pairs = list(itertools.combinations(range(NCH), 2)); worst = {p: 0.0 for p in pairs}; lonely = 0
for ts in u.trajectory:
    Pp = u.atoms.positions/10; ch = [Pp[s] for s in sl]
    m = {p: cKDTree(ch[p[0]]).query(ch[p[1]])[0].min() for p in pairs}
    for p in pairs: worst[p] = max(worst[p], m[p])
    for i in range(NCH):
        if min(m[p] for p in pairs if i in p) > 0.5: lonely += 1
print({"ABCDEF"[a]+"ABCDEF"[b]: round(float(v),2) for (a,b), v in worst.items()})
print("chains with no neighbour within 0.5 nm:", lonely)
assert lonely == 0, "a chain drifted away"

# ---- GROMACS analysis
print("== gmx analysis (slow)")
PAIRSTR = " ".join(f'"{a} {b}"' for a, b in itertools.combinations(CH, 2))
os.environ.update(CHS=" ".join(["Protein"]+CH), PAIRSTR=PAIRSTR)
sh(r'''cd "$OUT/frames"; X="$OUT/xvgs"; L="$OUT/logs"; T=fit_cluster_skip5.xtc
for c in $CHS; do
  printf "${c}_bb\n${c}_bb\n" | gmx rms  -s first_cl.gro -f $T -n chains.ndx -o $X/rmsd_$c.xvg -tu ns > $L/rmsd_$c.log 2>&1
  printf "${c}_CA\n${c}_CA\n" | gmx rmsf -s first_cl.gro -f $T -n chains.ndx -o $X/rmsf_$c.xvg -res   > $L/rmsf_$c.log 2>&1
  printf "$c\n"               | gmx gyrate -s protein.tpr -f $T -n chains.ndx -o $X/rg_$c.xvg          > $L/rg_$c.log 2>&1
  gmx sasa -s protein.tpr -f $T -n chains.ndx -surface "group Protein" -output "group $c" -o $X/sasa_$c.xvg -or $X/tmp_$c.xvg -tu ns > $L/sasa_$c.log 2>&1
  gmx hbond -s protein.tpr -f $T -n chains.ndx -r "group $c" -t "group $c" -num $X/hbnum_$c.xvg -tu ns > $L/hbond_$c.log 2>&1
done
eval "for p in $PAIRSTR; do set -- \$p
  gmx hbond -s protein.tpr -f $T -n chains.ndx -r \"group \$1\" -t \"group \$2\" -num $X/hbinter_\$1_\$2.xvg -tu ns > $L/hbinter_\$1_\$2.log 2>&1
done"
mv $X/tmp_Protein.xvg $X/sasa_perres_Protein.xvg; rm -f $X/tmp_ch*.xvg
for f in rmsd_Protein rmsd_chA rmsf_chA rg_Protein sasa_chA hbnum_Protein hbinter_chA_chB; do echo $f $(tail -1 $X/$f.xvg); done''')

# ---- plots
X = f"{OUT}/xvgs"
cols = ["#d62728","#1f77b4","#2ca02c","#ff9896","#7fbfff","#98df8a"]
sty = {c: dict(color=cols[i], ls="-", label="Chain "+c[-1]) for i, c in enumerate(CH)}
xvg = lambda f: np.loadtxt(f, comments=('#','@'))
def draw(out, title, xl, yl, items):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for x, y, s in items: ax.plot(x, y, lw=1.6, **s)
    ax.set(xlabel=xl, ylabel=yl, title=title); ax.legend(ncol=2); fig.tight_layout()
    fig.savefig(f"{OUT}/graphs/{out}.png"); plt.show()
chains = lambda pre, col=1, sc=1: [(xvg(f"{X}/{pre}_{c}.xvg")[:,0]/sc, xvg(f"{X}/{pre}_{c}.xvg")[:,col], s) for c, s in sty.items()]
def whole(pre, col=1, sc=1):
    a = xvg(f"{X}/{pre}_Protein.xvg"); return [(a[:,0]/sc, a[:,col], dict(color="grey", label="Whole trimer"))]
for pre, yl, ttl, sc in (("rmsd","RMSD (nm)","Backbone RMSD",1), ("rg","Rg (nm)","Radius of gyration",1000),
                         ("sasa","SASA (nm²)","SASA",1), ("hbnum","H-bonds","Intra-chain H-bonds",1)):
    cc = 2 if pre == "sasa" else 1
    draw(f"{pre}_per_chain", f"{ttl} per chain, {T}", "Time (ns)", yl, chains(pre, cc, sc))
    draw(f"{pre}_protein", f"{ttl}, whole trimer, {T}", "Time (ns)", yl, whole(pre, 1, sc))
draw("rmsf_per_chain", f"C-alpha RMSF per residue, {T}", "Residue", "RMSF (nm)", chains("rmsf"))
d = xvg(f"{X}/sasa_perres_Protein.xvg"); assert len(d) == NCH*NR, len(d)
draw("sasa_per_residue", f"SASA per residue, {T}", "Residue", "SASA (nm²)",
     [(np.arange(1,NR+1), d[i*NR:(i+1)*NR,1], s) for i, s in enumerate(sty.values())])
fig, ax = plt.subplots(figsize=(12,6), dpi=150)
for k, (a_, b_) in enumerate(itertools.combinations(CH, 2)):
    h = xvg(f"{X}/hbinter_{a_}_{b_}.xvg"); ax.plot(h[:,0], h[:,1], lw=1.2, color=plt.cm.tab10(k), label=f"{a_[-1]}–{b_[-1]}")
ax.set(xlabel="Time (ns)", ylabel="H-bonds", title=f"Inter-chain H-bonds, {T}"); ax.legend(ncol=3, fontsize=8)
fig.tight_layout(); fig.savefig(f"{OUT}/graphs/hbonds_inter_chain.png"); plt.show()

# ---- per-residue RMSD and H-bonds
u = mda.Universe(f"{OUT}/frames/first_cl.gro", f"{OUT}/frames/fit_cluster_skip5.xtc")
NA = u.atoms.n_atoms // NCH; cs = {c: slice(i*NA, (i+1)*NA) for i, c in enumerate(CH)}; resR = {}
for name, s in cs.items():
    chn = u.atoms[s]; bb = chn.select_atoms("name N CA C"); ca = chn.select_atoms("name CA")
    u.trajectory[0]; c0 = bb.positions.mean(0); bb0 = bb.positions - c0; ca0 = ca.positions - c0
    d2 = np.zeros(len(ca)); k = 0
    for ts in u.trajectory:
        c = bb.positions.mean(0); R, _ = rotation_matrix(bb.positions - c, bb0)
        d2 += (((ca.positions - c) @ R.T - ca0)**2).sum(1); k += 1
    resR[name] = np.sqrt(d2/k)/10
    np.savetxt(f"{X}/rmsd_perres_{name}.xvg", np.c_[np.arange(1,NR+1), resR[name]], fmt="%d %.6f")
draw("rmsd_per_residue", f"C-alpha RMSD per residue vs first frame, {T}", "Residue", "RMSD (nm)",
     [(np.arange(1,NR+1), resR[c], s) for c, s in sty.items()])
draw("rmsd_per_residue_mean", f"Mean C-alpha RMSD per residue ({NCH} chains), {T}", "Residue", "RMSD (nm)",
     [(np.arange(1,NR+1), np.mean(list(resR.values()), axis=0), dict(color="grey", label="Mean of chains"))])

nmA = np.array([x.strip() for x in u.atoms.names]); rn = u.atoms.resnames
chain = np.arange(len(nmA)) // NA; ridx = u.atoms.resindices - chain*NR
assert ridx.min() == 0 and ridx.max() == NR-1
isH = np.array([x.startswith("H") for x in nmA])
P0 = u.atoms.positions/10; heavy = np.where(~isH)[0]; Hs = np.where(isH)[0]
don = heavy[cKDTree(P0[heavy]).query(P0[Hs])[1]]
kk = np.array([nmA[i][0] in "NO" for i in don]); Hs, don = Hs[kk], don[kk]
acc = np.array([i for i in range(len(nmA)) if nmA[i][0]=="O" or (nmA[i] in ("ND1","NE2") and rn[i].startswith("HI"))])
tot = np.zeros((NCH,NR)); inter = np.zeros((NCH,NR)); nb = []
for ts in u.trajectory:
    Pf = u.atoms.positions/10; k = 0
    for hi, lst in enumerate(cKDTree(Pf[Hs]).query_ball_tree(cKDTree(Pf[acc]), 0.25)):
        D_ = don[hi]
        for aj in lst:
            A = acc[aj]
            if A == D_ or (chain[A]==chain[D_] and ridx[A]==ridx[D_]): continue
            v1, v2 = Pf[Hs[hi]]-Pf[D_], Pf[A]-Pf[D_]; dist = np.linalg.norm(v2)
            if dist > 0.35 or v1@v2/(np.linalg.norm(v1)*dist) < np.cos(np.radians(30)): continue
            k += 1
            for Xa in (D_, A):
                tot[chain[Xa], ridx[Xa]] += 1
                if chain[D_] != chain[A]: inter[chain[Xa], ridx[Xa]] += 1
    nb.append(k)
nf = len(nb); print("mean H-bonds/frame:", round(float(np.mean(nb)),1))
for nmx, arr, ttl in (("hbperres",tot,"All H-bonds per residue"), ("hbperres_inter",inter,"Inter-chain H-bonds per residue")):
    for i, c in enumerate(CH): np.savetxt(f"{X}/{nmx}_{c}.xvg", np.c_[np.arange(1,NR+1), arr[i]/nf], fmt="%d %.5f")
    draw(nmx, f"{ttl}, {T}", "Residue", "H-bonds per frame", [(np.arange(1,NR+1), arr[i]/nf, s) for i, s in enumerate(sty.values())])

# ---- DSSP
print("== dssp")
sh(r'''cd "$OUT/frames"
printf "Protein\n" | gmx dssp -s protein.tpr -f fit_cluster_skip5.xtc -n chains.ndx -o "$OUT/xvgs/dssp_Protein.dat" > "$OUT/logs/dssp_Protein.log" 2>&1
cp "$OUT/xvgs/dssp_Protein.dat" "$OUT/xvgs/dssp_chA.dat"
head -c 80 "$OUT/xvgs/dssp_Protein.dat"; echo''')
a = np.array([list(l.rstrip("\n")) for l in open(f"{X}/dssp_Protein.dat") if l.strip()])
a = a[:, ~(a == "=").all(0)]; print("dssp shape:", a.shape); assert a.shape[1] == NCH*NR, a.shape
Dd = {c: a[:, i*NR:(i+1)*NR] for i, c in enumerate(CH)}; t = np.arange(len(a))*0.005
for name, ss, yl in (("helix","HGI","Helix residues (%)"), ("strand","EB","Strand residues (%)")):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        ax.plot(t, 100*np.isin(Dd[c], list(ss)).mean(1), lw=1.6, **s)
        print(name, c, "mean %:", round(100*np.isin(Dd[c], list(ss)).mean(), 1))
    ax.set(xlabel="Time (ns)", ylabel=yl, title=f"{name.capitalize()} content vs time, {T}")
    ax.legend(ncol=2); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_{name}_time.png"); plt.show()
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        f = np.isin(Dd[c], list(ss)).mean(0); ax.plot(np.arange(1,NR+1), f, lw=1.4, **s)
        np.savetxt(f"{X}/dssp_{name}_perres_{c}.xvg", np.c_[np.arange(1,NR+1), f], fmt="%d %.4f")
    ax.set(xlabel="Residue", ylabel=f"Fraction of frames in {name}", title=f"{name.capitalize()} per residue, {T}")
    ax.legend(ncol=2); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_{name}_perres.png"); plt.show()
fig, ax = plt.subplots(figsize=(12,5), dpi=150)
for c, s in sty.items(): ax.plot(t, 100*(1-np.isin(Dd[c], list("HGIEB")).mean(1)), lw=1.6, **s)
ax.set(xlabel="Time (ns)", ylabel="Coil/turn residues (%)", title=f"Coil/turn content vs time, {T}")
ax.legend(ncol=2); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_coil_time.png"); plt.show()
cmap = ListedColormap(["#eeeeee", "#d62728", "#1f77b4"])
code = lambda m: np.where(np.isin(m, list("HGI")), 1, np.where(np.isin(m, list("EB")), 2, 0))
fig, axs = plt.subplots(NCH, 1, figsize=(12,2.2*NCH+1), dpi=150, sharex=True)
for ax, (c, s) in zip(np.atleast_1d(axs), sty.items()):
    ax.imshow(code(Dd[c]), aspect="auto", cmap=cmap, vmin=0, vmax=2, origin="lower", extent=[1,NR,0,1])
    ax.set_ylabel("Time (ns)"); ax.set_title(s["label"], loc="left", fontsize=10)
np.atleast_1d(axs)[-1].set_xlabel("Residue"); fig.suptitle(f"DSSP map: red = helix, blue = strand, grey = coil/turn, {T}")
fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_map.png"); plt.show()

# ---- summary + all-states plots
def last(f, col=1): b = xvg(f); return b[len(b)//2:, col].mean()
h = len(t)//2; rows = []
for c in CH:
    rows.append(dict(chain=c, RMSD_nm=last(f"{X}/rmsd_{c}.xvg"), Rg_nm=last(f"{X}/rg_{c}.xvg"),
      SASA_nm2=last(f"{X}/sasa_{c}.xvg", 2), intra_Hbonds=last(f"{X}/hbnum_{c}.xvg"),
      helix_pct=100*np.isin(Dd[c][h:], list("HGI")).mean(), strand_pct=100*np.isin(Dd[c][h:], list("EB")).mean()))
rows.append(dict(chain="trimer", RMSD_nm=last(f"{X}/rmsd_Protein.xvg"), Rg_nm=last(f"{X}/rg_Protein.xvg"),
                 SASA_nm2=last(f"{X}/sasa_Protein.xvg"), intra_Hbonds=last(f"{X}/hbnum_Protein.xvg")))
df = pd.DataFrame(rows).round(3); df.to_csv(f"{X}/summary_last0.5ns.csv", index=False); print(df)
STATES = [("Alpha helix","H"),("3-10 helix","G"),("Pi helix","I"),("Beta strand","E"),("Beta bridge","B"),
          ("Turn","T"),("Bend","S"),("Polyproline II helix","P")]
Pc = {k_: 100*(a == c_).mean(1) for k_, c_ in STATES}; Pc["Coil / Unstructured"] = 100 - sum(Pc.values())
tp = np.arange(len(a))*5.0
fig, ax = plt.subplots(figsize=(12,6.2), dpi=150)
for i, (k_, y) in enumerate(Pc.items()): ax.plot(tp, y, color=plt.cm.tab10(i), lw=1.6, label=k_)
ax.set(xlabel="Time (ps)", ylabel="Percentage (%)", title=f"Vicilin All Secondary Structure States — {T}")
ax.grid(alpha=.3); ax.legend(title="Secondary structure", ncol=2, loc="upper right"); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/dssp_all_states_{TEMP}K_{RUN}.png"); plt.show()
Gp = {"Alpha helix": Pc["Alpha helix"], "Beta structure": Pc["Beta strand"]+Pc["Beta bridge"], "Turns": Pc["Turn"]}
Gp["Loops / Other"] = 100 - sum(Gp.values())
fig, ax = plt.subplots(figsize=(10,6), dpi=150)
for i, (k_, y) in enumerate(Gp.items()): ax.plot(tp, y, color=plt.cm.tab10(i), lw=1.8, label=k_)
ax.set(xlabel="Time (ps)", ylabel="Percentage (%)", title=f"Vicilin Secondary Structure — {T}")
ax.grid(alpha=.3); ax.legend(); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/dssp_groups_{TEMP}K_{RUN}.png"); plt.show()
print("DONE ->", OUT)

In [ ]:
import os, subprocess
cmd = r'''cd "/content/drive/MyDrive/Plant Protein Extrusion MD/analysism/vicilin_373_shear_r2" || exit 1
echo "== graphs ($(ls graphs/*.png 2>/dev/null | wc -l))"; ls graphs
echo; echo "== xvg files that are empty or missing (expect nothing)"
for c in Protein chA chB chC; do for p in rmsd rmsf rg sasa hbnum; do [ -s xvgs/${p}_$c.xvg ] || echo "MISSING/EMPTY: ${p}_$c"; done; done
for p in chA_chB chA_chC chB_chC; do [ -s xvgs/hbinter_$p.xvg ] || echo "MISSING/EMPTY: hbinter_$p"; done
for f in sasa_perres_Protein.xvg dssp_Protein.dat dssp_chA.dat; do [ -s xvgs/$f ] || echo "MISSING/EMPTY: $f"; done
echo; echo "== errors in gmx logs (expect nothing)"
grep -liE "fatal error|segmentation|not found|invalid" logs/*.log
echo; echo "== summary (last half of the run)"; cat xvgs/summary_last0.5ns.csv'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

In [ ]:
import os, subprocess
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
cmd = r'''cd "/content/drive/MyDrive/Plant Protein Extrusion MD" || { echo "FOLDER NOT FOUND - wrong Drive account?"; exit 1; }
echo "== gmx"; gmx --version 2>&1 | grep -E "GROMACS version|GPU support"
echo; echo "== r2 thermal inputs"; ls -lh md_373_r2.tpr md_373_r2.mdp
echo; echo "== r2 thermal outputs (expect none before the first run)"; ls md_373_r2.* 2>&1 | grep -vE "\.(tpr|mdp)$"
echo; echo "== GPU"; nvidia-smi -L 2>&1 | head -1'''
r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True); print(r.stdout, r.stderr[-300:])

In [ ]:
import os, subprocess
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
r = subprocess.run(["bash","-c",'cd "/content/drive/MyDrive/Plant Protein Extrusion MD" && gmx mdrun -deffnm md_373_r2 -cpt 5 -ntmpi 1 -ntomp 2 -pin off 2>&1 | tail -5'], capture_output=True, text=True); print(r.stdout)

In [ ]:
import os, sys, json, itertools, subprocess
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from google.colab import drive
TEMP, RUN = "373", "r2"
TPR_NAME  = "md_373_r2"                       # thermal r2 tpr
TRAJ_NAME = "md_373_r2"                       # ran in one part
NCH = 3; DEFORM = 0.0                          # trimer; thermal run, no shear
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
if os.system("gmx --version > /dev/null 2>&1") != 0: raise SystemExit("GROMACS not found - run your GROMACS setup cell first, then run this cell again")
os.system('ln -sfn "/content/drive/MyDrive/Plant Protein Extrusion MD" /content/pm')
PM = "/content/drive/MyDrive/Plant Protein Extrusion MD"
if os.path.exists(f"{PM}/md_373_r2.part0002.xtc"): raise SystemExit("md_373_r2 was restarted (part0002 files exist) - combine the parts first; send me the file list")
if not os.path.exists(f"{PM}/md_373_r2.log") or "Finished mdrun" not in open(f"{PM}/md_373_r2.log").read(): raise SystemExit("md_373_r2.log is missing or has no Finished mdrun line - the run did not finish")
OUT = f"/content/pm/analysism/vicilin_{TEMP}_{RUN}"; T = "373 K thermal (r2)"
os.environ.update(P="/content/pm", S=TPR_NAME, TRAJ=TRAJ_NAME, OUT=OUT)
for d in ["graphs","logs","frames","xvgs"]: os.makedirs(f"{OUT}/{d}", exist_ok=True)
subprocess.run([sys.executable,"-m","pip","install","-q","MDAnalysis","scipy"])
import MDAnalysis as mda
from MDAnalysis.coordinates.XTC import XTCReader
from MDAnalysis.lib.distances import minimize_vectors
from MDAnalysis.analysis.align import rotation_matrix
from scipy.spatial import cKDTree
from matplotlib.colors import ListedColormap
def sh(cmd, tail=0):
    r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True)
    o = (r.stdout + r.stderr).strip().split("\n")
    print("\n".join(o[-tail:] if tail else o))

# ---- check files, mdp, and trajectory continuity
print("== files"); sh('ls -lh $P/$S.tpr $P/$TRAJ.xtc; grep -iE "ref[-_]t|deform" $P/$S.mdp')
rd = XTCReader(f"{OUT.rsplit('/analysism',1)[0]}/{TRAJ_NAME}.xtc")
times = np.array([ts.time for ts in rd]); dims = np.array([ts.dimensions.copy() for ts in rd])
print("frames:", len(times), "| time", times[0], "->", times[-1], "| natoms:", rd.n_atoms)
assert rd.n_atoms == 19941, "xtc is not protein-only (expected 19941 atoms) - send me this number"
assert times[0] == 0 and abs(times[-1]-1000) < 1e-3 and len(times) == 1001 and np.all(np.diff(times) == 1), "trajectory not continuous 0-1000 ps"
print("box a,b,c start:", dims[0,:3].round(2), "end:", dims[-1,:3].round(2))
print("angles start:", dims[0,3:].round(2), "end:", dims[-1,3:].round(2))
if DEFORM:
    gd = DEFORM/(dims[0,1]/10); print(f"shear rate {gd:.4f} /ns -> expected rigid rotation {-0.5*gd*180/np.pi:.2f} deg/ns about z")
else: print("thermal run: box angles should stay 60/60/90")

# ---- protein tpr, whole, first frame
print("== whole")
sh(r'''cd "$OUT/frames"
printf "Protein\n" | gmx convert-tpr -s "$P/$S.tpr" -o protein.tpr 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f "$P/$TRAJ.xtc" -o whole2.xtc -pbc whole 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f whole2.xtc -o first_w.gro -dump 0 2>&1 | tail -1''')

# ---- chain detection from .gro
lines = open(f"{OUT}/frames/first_w.gro").read().split("\n")
n = int(lines[1]); body = lines[2:2+n]
resnum = np.array([int(l[0:5]) for l in body]); nm = [l[10:15].strip() for l in body]
starts = [0] + [i for i in range(1, n) if resnum[i] < resnum[i-1]]; ends = starts[1:] + [n]
nres = [len(set(resnum[s:e])) for s, e in zip(starts, ends)]
print("atoms:", n, "| chains:", len(starts), "| atoms/chain:", [e-s for s,e in zip(starts,ends)], "| res/chain:", nres)
assert len(starts) == NCH, f"expected {NCH} chains - send me the line above"
assert len(set(e-s for s,e in zip(starts,ends))) == 1, "chains differ in size - send me the line above"
NR = nres[0]; CH = ["ch"+"ABCDEF"[i] for i in range(NCH)]; G = {}; ranges = {}
for k, (s, e) in enumerate(zip(starts, ends)):
    r = range(s+1, e+1); c = CH[k]; ranges[c] = [s, e]
    G[c] = list(r); G[c+"_bb"] = [i for i in r if nm[i-1] in ("N","CA","C")]; G[c+"_CA"] = [i for i in r if nm[i-1]=="CA"]
r = range(1, n+1)
G["Protein"] = list(r); G["Protein_bb"] = [i for i in r if nm[i-1] in ("N","CA","C")]; G["Protein_CA"] = [i for i in r if nm[i-1]=="CA"]
with open(f"{OUT}/frames/chains.ndx","w") as f:
    for g, idx in G.items():
        f.write(f"[ {g} ]\n")
        for k in range(0, len(idx), 15): f.write(" ".join(f"{i:5d}" for i in idx[k:k+15]) + "\n")
json.dump(dict(ranges=ranges, nres=NR, natoms=n), open(f"{OUT}/frames/chains.json","w"))

# ---- keep chains together (triclinic-safe)
sl = [slice(*ranges[c]) for c in CH]
u = mda.Universe(f"{OUT}/frames/first_w.gro", f"{OUT}/frames/whole2.xtc")
with mda.Writer(f"{OUT}/frames/together.xtc", u.atoms.n_atoms) as W:
    for ts in u.trajectory[::5]:
        pos = u.atoms.positions.copy(); box = ts.dimensions.astype(np.float32); a0 = pos[sl[0]].mean(0)
        for s in sl[1:]:
            d = pos[s].mean(0) - a0
            dm = minimize_vectors(d[None,:].astype(np.float32), box)[0]; pos[s] -= (d - dm)
        u.atoms.positions = pos; W.write(u.atoms)

# ---- fit
print("== fit")
sh(r'''cd "$OUT/frames"
printf "Backbone\nProtein\n" | gmx trjconv -s protein.tpr -f together.xtc -o fit_cluster_skip5.xtc -fit rot+trans 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f fit_cluster_skip5.xtc -o first_cl.gro -dump 0 2>&1 | tail -1
rm -f whole2.xtc together.xtc
gmx check -f fit_cluster_skip5.xtc 2>&1 | grep -E "Last frame|^Step"''')

# ---- trimer integrity check
u = mda.Universe(f"{OUT}/frames/first_cl.gro", f"{OUT}/frames/fit_cluster_skip5.xtc")
pairs = list(itertools.combinations(range(NCH), 2)); worst = {p: 0.0 for p in pairs}; lonely = 0
for ts in u.trajectory:
    Pp = u.atoms.positions/10; ch = [Pp[s] for s in sl]
    m = {p: cKDTree(ch[p[0]]).query(ch[p[1]])[0].min() for p in pairs}
    for p in pairs: worst[p] = max(worst[p], m[p])
    for i in range(NCH):
        if min(m[p] for p in pairs if i in p) > 0.5: lonely += 1
print({"ABCDEF"[a]+"ABCDEF"[b]: round(float(v),2) for (a,b), v in worst.items()})
print("chains with no neighbour within 0.5 nm:", lonely)
assert lonely == 0, "a chain drifted away"

# ---- GROMACS analysis
print("== gmx analysis (slow)")
PAIRSTR = " ".join(f'"{a} {b}"' for a, b in itertools.combinations(CH, 2))
os.environ.update(CHS=" ".join(["Protein"]+CH), PAIRSTR=PAIRSTR)
sh(r'''cd "$OUT/frames"; X="$OUT/xvgs"; L="$OUT/logs"; T=fit_cluster_skip5.xtc
for c in $CHS; do
  printf "${c}_bb\n${c}_bb\n" | gmx rms  -s first_cl.gro -f $T -n chains.ndx -o $X/rmsd_$c.xvg -tu ns > $L/rmsd_$c.log 2>&1
  printf "${c}_CA\n${c}_CA\n" | gmx rmsf -s first_cl.gro -f $T -n chains.ndx -o $X/rmsf_$c.xvg -res   > $L/rmsf_$c.log 2>&1
  printf "$c\n"               | gmx gyrate -s protein.tpr -f $T -n chains.ndx -o $X/rg_$c.xvg          > $L/rg_$c.log 2>&1
  gmx sasa -s protein.tpr -f $T -n chains.ndx -surface "group Protein" -output "group $c" -o $X/sasa_$c.xvg -or $X/tmp_$c.xvg -tu ns > $L/sasa_$c.log 2>&1
  gmx hbond -s protein.tpr -f $T -n chains.ndx -r "group $c" -t "group $c" -num $X/hbnum_$c.xvg -tu ns > $L/hbond_$c.log 2>&1
done
eval "for p in $PAIRSTR; do set -- \$p
  gmx hbond -s protein.tpr -f $T -n chains.ndx -r \"group \$1\" -t \"group \$2\" -num $X/hbinter_\$1_\$2.xvg -tu ns > $L/hbinter_\$1_\$2.log 2>&1
done"
mv $X/tmp_Protein.xvg $X/sasa_perres_Protein.xvg; rm -f $X/tmp_ch*.xvg
for f in rmsd_Protein rmsd_chA rmsf_chA rg_Protein sasa_chA hbnum_Protein hbinter_chA_chB; do echo $f $(tail -1 $X/$f.xvg); done''')

# ---- plots
X = f"{OUT}/xvgs"
cols = ["#d62728","#1f77b4","#2ca02c","#ff9896","#7fbfff","#98df8a"]
sty = {c: dict(color=cols[i], ls="-", label="Chain "+c[-1]) for i, c in enumerate(CH)}
xvg = lambda f: np.loadtxt(f, comments=('#','@'))
def draw(out, title, xl, yl, items):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for x, y, s in items: ax.plot(x, y, lw=1.6, **s)
    ax.set(xlabel=xl, ylabel=yl, title=title); ax.legend(ncol=2); fig.tight_layout()
    fig.savefig(f"{OUT}/graphs/{out}.png"); plt.show()
chains = lambda pre, col=1, sc=1: [(xvg(f"{X}/{pre}_{c}.xvg")[:,0]/sc, xvg(f"{X}/{pre}_{c}.xvg")[:,col], s) for c, s in sty.items()]
def whole(pre, col=1, sc=1):
    a = xvg(f"{X}/{pre}_Protein.xvg"); return [(a[:,0]/sc, a[:,col], dict(color="grey", label="Whole trimer"))]
for pre, yl, ttl, sc in (("rmsd","RMSD (nm)","Backbone RMSD",1), ("rg","Rg (nm)","Radius of gyration",1000),
                         ("sasa","SASA (nm²)","SASA",1), ("hbnum","H-bonds","Intra-chain H-bonds",1)):
    cc = 2 if pre == "sasa" else 1
    draw(f"{pre}_per_chain", f"{ttl} per chain, {T}", "Time (ns)", yl, chains(pre, cc, sc))
    draw(f"{pre}_protein", f"{ttl}, whole trimer, {T}", "Time (ns)", yl, whole(pre, 1, sc))
draw("rmsf_per_chain", f"C-alpha RMSF per residue, {T}", "Residue", "RMSF (nm)", chains("rmsf"))
d = xvg(f"{X}/sasa_perres_Protein.xvg"); assert len(d) == NCH*NR, len(d)
draw("sasa_per_residue", f"SASA per residue, {T}", "Residue", "SASA (nm²)",
     [(np.arange(1,NR+1), d[i*NR:(i+1)*NR,1], s) for i, s in enumerate(sty.values())])
fig, ax = plt.subplots(figsize=(12,6), dpi=150)
for k, (a_, b_) in enumerate(itertools.combinations(CH, 2)):
    h = xvg(f"{X}/hbinter_{a_}_{b_}.xvg"); ax.plot(h[:,0], h[:,1], lw=1.2, color=plt.cm.tab10(k), label=f"{a_[-1]}–{b_[-1]}")
ax.set(xlabel="Time (ns)", ylabel="H-bonds", title=f"Inter-chain H-bonds, {T}"); ax.legend(ncol=3, fontsize=8)
fig.tight_layout(); fig.savefig(f"{OUT}/graphs/hbonds_inter_chain.png"); plt.show()

# ---- per-residue RMSD and H-bonds
u = mda.Universe(f"{OUT}/frames/first_cl.gro", f"{OUT}/frames/fit_cluster_skip5.xtc")
NA = u.atoms.n_atoms // NCH; cs = {c: slice(i*NA, (i+1)*NA) for i, c in enumerate(CH)}; resR = {}
for name, s in cs.items():
    chn = u.atoms[s]; bb = chn.select_atoms("name N CA C"); ca = chn.select_atoms("name CA")
    u.trajectory[0]; c0 = bb.positions.mean(0); bb0 = bb.positions - c0; ca0 = ca.positions - c0
    d2 = np.zeros(len(ca)); k = 0
    for ts in u.trajectory:
        c = bb.positions.mean(0); R, _ = rotation_matrix(bb.positions - c, bb0)
        d2 += (((ca.positions - c) @ R.T - ca0)**2).sum(1); k += 1
    resR[name] = np.sqrt(d2/k)/10
    np.savetxt(f"{X}/rmsd_perres_{name}.xvg", np.c_[np.arange(1,NR+1), resR[name]], fmt="%d %.6f")
draw("rmsd_per_residue", f"C-alpha RMSD per residue vs first frame, {T}", "Residue", "RMSD (nm)",
     [(np.arange(1,NR+1), resR[c], s) for c, s in sty.items()])
draw("rmsd_per_residue_mean", f"Mean C-alpha RMSD per residue ({NCH} chains), {T}", "Residue", "RMSD (nm)",
     [(np.arange(1,NR+1), np.mean(list(resR.values()), axis=0), dict(color="grey", label="Mean of chains"))])

nmA = np.array([x.strip() for x in u.atoms.names]); rn = u.atoms.resnames
chain = np.arange(len(nmA)) // NA; ridx = u.atoms.resindices - chain*NR
assert ridx.min() == 0 and ridx.max() == NR-1
isH = np.array([x.startswith("H") for x in nmA])
P0 = u.atoms.positions/10; heavy = np.where(~isH)[0]; Hs = np.where(isH)[0]
don = heavy[cKDTree(P0[heavy]).query(P0[Hs])[1]]
kk = np.array([nmA[i][0] in "NO" for i in don]); Hs, don = Hs[kk], don[kk]
acc = np.array([i for i in range(len(nmA)) if nmA[i][0]=="O" or (nmA[i] in ("ND1","NE2") and rn[i].startswith("HI"))])
tot = np.zeros((NCH,NR)); inter = np.zeros((NCH,NR)); nb = []
for ts in u.trajectory:
    Pf = u.atoms.positions/10; k = 0
    for hi, lst in enumerate(cKDTree(Pf[Hs]).query_ball_tree(cKDTree(Pf[acc]), 0.25)):
        D_ = don[hi]
        for aj in lst:
            A = acc[aj]
            if A == D_ or (chain[A]==chain[D_] and ridx[A]==ridx[D_]): continue
            v1, v2 = Pf[Hs[hi]]-Pf[D_], Pf[A]-Pf[D_]; dist = np.linalg.norm(v2)
            if dist > 0.35 or v1@v2/(np.linalg.norm(v1)*dist) < np.cos(np.radians(30)): continue
            k += 1
            for Xa in (D_, A):
                tot[chain[Xa], ridx[Xa]] += 1
                if chain[D_] != chain[A]: inter[chain[Xa], ridx[Xa]] += 1
    nb.append(k)
nf = len(nb); print("mean H-bonds/frame:", round(float(np.mean(nb)),1))
for nmx, arr, ttl in (("hbperres",tot,"All H-bonds per residue"), ("hbperres_inter",inter,"Inter-chain H-bonds per residue")):
    for i, c in enumerate(CH): np.savetxt(f"{X}/{nmx}_{c}.xvg", np.c_[np.arange(1,NR+1), arr[i]/nf], fmt="%d %.5f")
    draw(nmx, f"{ttl}, {T}", "Residue", "H-bonds per frame", [(np.arange(1,NR+1), arr[i]/nf, s) for i, s in enumerate(sty.values())])

# ---- DSSP
print("== dssp")
sh(r'''cd "$OUT/frames"
printf "Protein\n" | gmx dssp -s protein.tpr -f fit_cluster_skip5.xtc -n chains.ndx -o "$OUT/xvgs/dssp_Protein.dat" > "$OUT/logs/dssp_Protein.log" 2>&1
cp "$OUT/xvgs/dssp_Protein.dat" "$OUT/xvgs/dssp_chA.dat"
head -c 80 "$OUT/xvgs/dssp_Protein.dat"; echo''')
a = np.array([list(l.rstrip("\n")) for l in open(f"{X}/dssp_Protein.dat") if l.strip()])
a = a[:, ~(a == "=").all(0)]; print("dssp shape:", a.shape); assert a.shape[1] == NCH*NR, a.shape
Dd = {c: a[:, i*NR:(i+1)*NR] for i, c in enumerate(CH)}; t = np.arange(len(a))*0.005
for name, ss, yl in (("helix","HGI","Helix residues (%)"), ("strand","EB","Strand residues (%)")):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        ax.plot(t, 100*np.isin(Dd[c], list(ss)).mean(1), lw=1.6, **s)
        print(name, c, "mean %:", round(100*np.isin(Dd[c], list(ss)).mean(), 1))
    ax.set(xlabel="Time (ns)", ylabel=yl, title=f"{name.capitalize()} content vs time, {T}")
    ax.legend(ncol=2); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_{name}_time.png"); plt.show()
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        f = np.isin(Dd[c], list(ss)).mean(0); ax.plot(np.arange(1,NR+1), f, lw=1.4, **s)
        np.savetxt(f"{X}/dssp_{name}_perres_{c}.xvg", np.c_[np.arange(1,NR+1), f], fmt="%d %.4f")
    ax.set(xlabel="Residue", ylabel=f"Fraction of frames in {name}", title=f"{name.capitalize()} per residue, {T}")
    ax.legend(ncol=2); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_{name}_perres.png"); plt.show()
fig, ax = plt.subplots(figsize=(12,5), dpi=150)
for c, s in sty.items(): ax.plot(t, 100*(1-np.isin(Dd[c], list("HGIEB")).mean(1)), lw=1.6, **s)
ax.set(xlabel="Time (ns)", ylabel="Coil/turn residues (%)", title=f"Coil/turn content vs time, {T}")
ax.legend(ncol=2); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_coil_time.png"); plt.show()
cmap = ListedColormap(["#eeeeee", "#d62728", "#1f77b4"])
code = lambda m: np.where(np.isin(m, list("HGI")), 1, np.where(np.isin(m, list("EB")), 2, 0))
fig, axs = plt.subplots(NCH, 1, figsize=(12,2.2*NCH+1), dpi=150, sharex=True)
for ax, (c, s) in zip(np.atleast_1d(axs), sty.items()):
    ax.imshow(code(Dd[c]), aspect="auto", cmap=cmap, vmin=0, vmax=2, origin="lower", extent=[1,NR,0,1])
    ax.set_ylabel("Time (ns)"); ax.set_title(s["label"], loc="left", fontsize=10)
np.atleast_1d(axs)[-1].set_xlabel("Residue"); fig.suptitle(f"DSSP map: red = helix, blue = strand, grey = coil/turn, {T}")
fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_map.png"); plt.show()

# ---- summary + all-states plots
def last(f, col=1): b = xvg(f); return b[len(b)//2:, col].mean()
h = len(t)//2; rows = []
for c in CH:
    rows.append(dict(chain=c, RMSD_nm=last(f"{X}/rmsd_{c}.xvg"), Rg_nm=last(f"{X}/rg_{c}.xvg"),
      SASA_nm2=last(f"{X}/sasa_{c}.xvg", 2), intra_Hbonds=last(f"{X}/hbnum_{c}.xvg"),
      helix_pct=100*np.isin(Dd[c][h:], list("HGI")).mean(), strand_pct=100*np.isin(Dd[c][h:], list("EB")).mean()))
rows.append(dict(chain="trimer", RMSD_nm=last(f"{X}/rmsd_Protein.xvg"), Rg_nm=last(f"{X}/rg_Protein.xvg"),
                 SASA_nm2=last(f"{X}/sasa_Protein.xvg"), intra_Hbonds=last(f"{X}/hbnum_Protein.xvg")))
df = pd.DataFrame(rows).round(3); df.to_csv(f"{X}/summary_last0.5ns.csv", index=False); print(df)
STATES = [("Alpha helix","H"),("3-10 helix","G"),("Pi helix","I"),("Beta strand","E"),("Beta bridge","B"),
          ("Turn","T"),("Bend","S"),("Polyproline II helix","P")]
Pc = {k_: 100*(a == c_).mean(1) for k_, c_ in STATES}; Pc["Coil / Unstructured"] = 100 - sum(Pc.values())
tp = np.arange(len(a))*5.0
fig, ax = plt.subplots(figsize=(12,6.2), dpi=150)
for i, (k_, y) in enumerate(Pc.items()): ax.plot(tp, y, color=plt.cm.tab10(i), lw=1.6, label=k_)
ax.set(xlabel="Time (ps)", ylabel="Percentage (%)", title=f"Vicilin All Secondary Structure States — {T}")
ax.grid(alpha=.3); ax.legend(title="Secondary structure", ncol=2, loc="upper right"); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/dssp_all_states_{TEMP}K_{RUN}.png"); plt.show()
Gp = {"Alpha helix": Pc["Alpha helix"], "Beta structure": Pc["Beta strand"]+Pc["Beta bridge"], "Turns": Pc["Turn"]}
Gp["Loops / Other"] = 100 - sum(Gp.values())
fig, ax = plt.subplots(figsize=(10,6), dpi=150)
for i, (k_, y) in enumerate(Gp.items()): ax.plot(tp, y, color=plt.cm.tab10(i), lw=1.8, label=k_)
ax.set(xlabel="Time (ps)", ylabel="Percentage (%)", title=f"Vicilin Secondary Structure — {T}")
ax.grid(alpha=.3); ax.legend(); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/dssp_groups_{TEMP}K_{RUN}.png"); plt.show()
print("DONE ->", OUT)